# 05-05. 상세 분석 근거 및 14 LOT 전수 자료

**담당:** 정수진 · **Issue:** #65

본문(01~04)에서 확대·비교하지 않은 것까지 **14 LOT 전체를 그대로 보관**한다. LOT마다 전체 22개 신호의 확인 지도 → 공정 전체 흐름 → 구간 원본 확대 순서이며, 강종별 전체 비교·주기 위상과 정비 기록 확대(07/08 BM)를 함께 둔다. 판단은 본문을 따른다.

## 분석 보고서 구성 및 읽는 흐름

| 파일 | 읽는 흐름 |
|---|---|
| **[01 분석 개요 및 판정 기준](05_all_data_timeseries_jinny_01.ipynb)** | 핵심 결과 요약 → 분석 대상 및 자료 범위 → 연간 흐름·월별 관측 구간 → 운전·정비 이력 → 품질 및 데이터 신뢰성 검토 → 판정 기준(운전 상태·평상시 대비 변동 수준·반복성 및 우연 변동 가능성) |
| **[02 탐지 구간 분류 및 분석 대상 선별](05_all_data_timeseries_jinny_02.ipynb)** | 탐지 구간 전체 현황 → 운전 상태별 분류(정지·승온) → 공정 위치·신호 특성 검토(Z3·세정 입구·소규모 CP·Z9 및 냉각) → 분석 제외·판단 보류·정밀 분석 대상 구분 및 근거 → 중점 분석 대상 우선순위 |
| **[03 주요 이상 의심 구간 상세 분석 및 비교](05_all_data_timeseries_jinny_03.ipynb)** | 핵심 결과 요약 → 제공 14 LOT의 CP 특성 및 검토 기준 → 주요 LOT(240069·240037·240117) 구간 확대 → 가스 분석계 셀 관련 신호 검토 → 동일 강종 비교 및 공정 조건 검토(Z7·Z8, 출력 Z4) → 정기수리 전후 변화 비교 → 주요 분석 결과 정리 |
| **[04 종합 판정 및 후속 검증 계획](05_all_data_timeseries_jinny_04.ipynb)** | 14 LOT × 공정별 종합 판정 → 주요 판단 근거 표 → 이상 원인 가설 및 불확실성 → 추가 확인 항목·검증 방법 → 후속 조치 및 판정 변경 조건 → 부록: 탐지 구간 전수 및 판정 근거 |
| **[05 상세 분석 근거 및 14 LOT 전수 자료](05_all_data_timeseries_jinny_05.ipynb)** | LOT별 전체 22개 신호 현황 → 공정 흐름 및 주요 구간 확대 → 강종별 전체 비교 → Z9 주기 특성 → 정비 이력 상세 → LOT별 선별 구간·비교 근거와 본문 판정 연결 |

대상은 CAL01의 제공된 14 LOT·22개 센서와 CAL01 정비/같은 WO의 부품교체, CR1 강종변경·정기수리다. 관측 범위는 실제 가동시간이나 소재 입·출측 시간이 아니다. 같은 강종·같은 경과 분은 비교 기준이며 같은 설정값·속도·두께를 보장하지 않는다. CP-Z4·CP-Z4M은 단위 미확인이며, 물리적 정의와 산식은 별도로 확인해야 한다. 시간 겹침·선후관계만으로 고장 원인이나 정비 효과를 확정하지 않는다.

**분석 대상 선별 근거**는 [02](05_all_data_timeseries_jinny_02.ipynb), **확대·비교**는 [03](05_all_data_timeseries_jinny_03.ipynb), **판정**은 [04](05_all_data_timeseries_jinny_04.ipynb)에 있다.

## 기본 설정

관측 시작·종료는 제공된 기록의 범위이며 소재의 실제 입·출측 시각이 아닙니다. 원본에서 확인된 TC 단위는 ℃, OP 단위는 %입니다. 목표값·운전 모드가 없어 공학적 정상 한계는 정하지 않습니다. CP-Z4와 CP-Z4M은 단위 미확인 기록값으로 원본 값을 그대로 표시합니다.

In [ ]:
import os
import pandas as pd

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
FIG_DIR = os.path.join(ROOT, "figures")
ENC = "utf-8-sig"

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")
BASE_CSV = os.path.join(PROCESSED_DIR, "cal01_base_v1.csv")
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

In [ ]:
"""Shared presentation rules for the five CAL01 time-series notebooks.

Only artists and display labels are changed here. Sensor values, thresholds,
aggregation windows and axis limits belong to the notebooks.
"""


import html
import re
from pathlib import Path

from matplotlib import rcParams
from matplotlib.lines import Line2D

EXPORT_DPI = 180
EXPORT_PADDING = 0.20
INK = '#1F2937'
MUTED = '#64748B'

# A color always identifies the same member of the same domain. Different
# domains use separate legends. LOT colors contrast most within the same grade.
PROCESS_COLORS = {'세정': '#2563EB', '건조': '#D97706', '가열·균열': '#DC4C3E', '과시효': '#059669', '냉각': '#7C3AED'}
GRADE_COLORS = {'SGCC': '#B9D6D9', 'SPFH590': '#B4634E', 'SPCE': '#3F909A', 'SPCEN': '#8676AC', 'SPHC': '#A48258', 'DP590': '#2B527A', 'SPCD': '#3E75A8', 'SPCC': '#9AA7B5'}
LOT_COLORS = {240001: '#334155', 240024: '#2563EB', 240108: '#F97316', 240037: '#1D4ED8', 240121: '#D946EF', 240133: '#10B981', 240038: '#0891B2', 240114: '#DC2626', 240044: '#0F766E', 240061: '#D97706', 240069: '#7C3AED', 240091: '#DB2777', 240058: '#92400E', 240117: '#475569'}
MAINTENANCE_COLORS = {'BM': '#C81E3A', 'CBM': '#6D28D9', 'TBM': '#0E7490'}
MAINTENANCE_LABELS = {"BM": "BM · 고장 후 정비", "CBM": "CBM · 상태 기준 정비", "TBM": "TBM · 주기 기준 정비"}
MAINTENANCE_HATCHES = {"BM": "///", "CBM": "...", "TBM": "xxx"}
ZONE_COLORS = {1: '#475569', 2: '#B45309', 3: '#7C3AED', 4: '#2563EB', 5: '#F97316', 6: '#059669', 7: '#DB2777', 8: '#0891B2', 9: '#A16207'}
SENSOR_COLORS = {'CLN-IN': '#0F766E', 'CP-Z4': '#E11D48', 'CP-Z4M': '#64748B', 'TC-OUT1': '#4F46E5', 'TC-OUT2': '#CA8A04'}

# LOT is encoded by color. Measured OP uses a dashed line; TC/CP use solid lines.
LOT_LINESTYLES = {240001: '-', 240024: '-', 240037: '-', 240038: '-', 240044: '-', 240058: '-', 240061: '-', 240069: '-', 240091: '-', 240108: '-', 240114: '-', 240117: '-', 240121: '-', 240133: '-'}

# Full original explanations remain available immediately below each figure.
# Short figure text describes the target, comparison rule and reading purpose.
_HEADINGS = {
    "연간 공정 흐름": ("2024년 CAL01 관측·정비 흐름", "2024년 관측창과 정비 시각을 대조해 비교 범위 확인"),
    "LOT별 자료 상태 · 분석에 쓸 수 있는 구간과 빠진 곳": ("CAL01 14 LOT 관측 품질 비교", "LOT별 기록량과 결측을 대조해 분석 가능 구간 확인"),
    "측정 기록 공백·센서 NaN 비교": ("CAL01 측정 공백·센서 결측 비교", "같은 LOT의 TC·OP를 대조해 결측 종류 구분"),
    "운전 상태별 급변 — 정지·승온을 따로 떼어야 하는 이유": ("CAL01 운전 상태별 급변 크기", "신호별 안정 운전 기준으로 정지·승온의 급변 비교"),
    "정비 일정 전후 정지·재가동 — 설명됨의 근거": ("CAL01 정비 전후 정지·재가동", "14 LOT의 온도·출력을 일정과 대조해 전환 구간 확인"),
    "CP-Z4 · 14 LOT 원본에서 기준(±0.05)을 넘은 곳": ("CAL01 14 LOT의 CP 기준 이탈", "CP 편차(단위 미확인) ±0.05 이탈의 위치·횟수 비교"),
    "탐지된 구간이 어디로 갔나 — 근거가 있어 빼는 것과 03으로 넘기는 것": ("CAL01 탐지 구간의 검토 대상 선별", "탐지 구간을 설명 근거별로 나눠 상세 검토 대상 확인"),
    "안정 참고 구간 · 가열·균열 Z3 수준": ("CAL01 강종별 Z3 온도·출력 수준", "같은 강종의 안정 참고 중앙값으로 LOT 수준 비교"),
    "안정 참고 구간 · Z3 제어출력과 온도를 같이 보기": ("CAL01 LOT별 Z3 온도·출력 반응", "LOT별 안정 참고 구간에서 온도와 출력의 동행 확인"),
    "출력이 0%·100%에 붙을 때 Z3 온도는 어떻게 움직이나": ("CAL01 Z3 출력 한계·온도 반응", "출력 0%·100% 지속 구간의 온도 반응을 함께 확인"),
    "세정 입구 온도 (CLN-IN)": ("CAL01 LOT별 세정 입구 온도", "LOT 중앙값과 같은 강종의 흐름으로 세정 온도 비교"),
    "정비 전후 흐름 · LOT 사이 정비와 LOT별 상태 지표": ("CAL01 정비 시각·LOT 지표 비교", "관측 날짜 순 지표와 LOT 사이 정비의 시간 관계 확인"),
    "볼 것 고르기 — 평소 흔들림 대비 편차": ("CAL01 평소 변동 대비 검토 우선순위", "신호별 평소 변동으로 나눈 편차로 검토 대상 비교"),
    "CP의 평소 관계 — 14 LOT에서 먼저 확인": ("CAL01 14 LOT의 CP 동시 관계", "안정 참고 1초 변화 상관으로 CP와 센서의 동행 확인"),
    "어느 LOT이 CP의 평소 관계에서 벗어났나 — 14 LOT 비교": ("CAL01 LOT별 CP 변동·동조 비교", "안정 참고 구간의 이탈·변동·동조로 검토 LOT 비교"),
    "CP 관점에서 확인할 LOT · 같은 강종부터 비교": ("CAL01 강종별 CP 검토 LOT 비교", "같은 강종의 시간당 이탈 빈도와 크기로 후보 비교"),
    "가스 분석계 셀 교체와 CP 신호 상태": ("CAL01 분석계 셀 교체·CP 상태", "CP 계측 부품의 교체 시각과 시간별 신호 상태 대조"),
    "같은 강종끼리 대조한 LOT별 요약 · 과시효 Z7·Z8": ("CAL01 동일 강종의 Z7·Z8 비교", "같은 강종 다른 LOT 중앙값 대비 온도·출력 차이 확인"),
    "14 LOT × 공정 · 확인 이슈와 근거 한눈에 보기": ("CAL01 14 LOT 공정별 검토 근거", "공정별 비교 가능성과 근거로 추가 확인할 LOT 선별"),
    "과시효 Z7·Z8 · 같은 강종 LOT끼리 비교": ("CAL01 동일 강종의 과시효 온도 비교", "안정 참고 1분 중앙값 분포로 LOT 수준 차이 확인"),
}


def configure():
    rcParams.update({
        "font.size": 11.5, "axes.titlesize": 13, "axes.labelsize": 11.5,
        "xtick.labelsize": 10.5, "ytick.labelsize": 10.5,
        "savefig.dpi": EXPORT_DPI, "savefig.pad_inches": EXPORT_PADDING,
        "svg.fonttype": "none", "pdf.fonttype": 42,
    })


def sensor_color(sensor):
    if sensor.startswith(("TC-Z", "OP-Z")):
        return ZONE_COLORS[int(sensor.rsplit("Z", 1)[1])]
    return SENSOR_COLORS[sensor]


def lot_linestyle(lot):
    return '-'


def process_title_color(title, default=INK):
    return INK


def palette_preview(path):
    """Read the guide in order: signal type, zone color, then compared LOTs."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    from IPython.display import display
    fig = plt.figure(figsize=(16, 10.2), facecolor="#F5F7FB")
    fig.text(.045, .955, "CAL01 시계열 · 색상과 선 표기", fontsize=25, weight="bold", color=INK, va="top")
    fig.text(.045, .909, "TC는 실선 · OP는 점선  |  같은 존은 같은 색  |  LOT 비교는 같은 강종 안에서 색으로 구분", fontsize=12, color=MUTED, va="top")

    def card(x, y, w, h, title, subtitle=""):
        fig.add_artist(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.008,rounding_size=0.012",
                       transform=fig.transFigure,fc="white",ec="#E2E8F0",lw=.8,zorder=0))
        ax=fig.add_axes([x+.014,y+.014,w-.028,h-.028],zorder=1)
        ax.set_xlim(0,1);ax.set_ylim(0,1);ax.axis("off")
        ax.text(0,.96,title,va="top",fontsize=14,weight="bold",color=INK)
        if subtitle:ax.text(0,.79,subtitle,va="top",fontsize=10.5,color=MUTED)
        return ax

    ax=card(.045,.661,.438,.207,"1  선 모양 = 측정 신호","TC·OP는 색을 바꾸지 않고 선 모양으로 구분")
    for y,s,label in [(.55,"TC-Z4","TC 온도 (℃) · 실선"),(.29,"OP-Z4","OP 제어출력 (%) · 점선")]:
        ax.plot([.015,.27],[y,y],color=ZONE_COLORS[4],lw=2.8,ls=signal_linestyle(s))
        ax.text(.32,y,label,va="center",fontsize=12.5,color=INK)
    ax.text(.015,.06,"CP는 실선 · 단위 미확인",fontsize=10.5,color=MUTED)

    ax=card(.504,.661,.451,.207,"2  선 색 = 존(ZONE)","같은 ZONE의 TC와 OP는 아래 두 선처럼 같은 색")
    for i,(z,col) in enumerate(ZONE_COLORS.items()):
        row,column=divmod(i,3);x=column*.337;y=.56-row*.20
        ax.text(x,y,f"Z{z}",va="center",fontsize=11.5,color=INK)
        ax.plot([x+.085,x+.175],[y+.027,y+.027],color=col,lw=2.1)
        ax.plot([x+.085,x+.175],[y-.027,y-.027],color=col,lw=2.1,ls="--")

    for x,w,title,colors in [(.045,.291,"공정 · 축 옆 구획색",PROCESS_COLORS),
                              (.357,.291,"강종 · 그룹 표식",{k:GRADE_COLORS[k] for k in ["SPCC","SPCD","SPCE","SPCEN","SPHC","SPFH590","DP590","SGCC"]}),
                              (.669,.286,"정비 · 실제 작업 구간",MAINTENANCE_COLORS)]:
        ax=card(x,.476,w,.151,title)
        columns=3 if len(colors)>5 else 2 if len(colors)>3 else 3
        for i,(label,col) in enumerate(colors.items()):
            row,column=divmod(i,columns);xx=column/columns;yy=.61-row*.235
            ax.plot([xx+.01,xx+.048],[yy,yy],color=col,lw=6,solid_capstyle="round")
            ax.text(xx+.073,yy,label,va="center",fontsize=9.8,color=INK)
        if title.startswith("정비"):
            ax.text(.01,.18,"BM 고장후  ·  CBM 상태기준  ·  TBM 주기기준",fontsize=9,color=MUTED)

    fig.text(.045,.437,"3  LOT 비교 · 아래 같은 강종 안에서 선 색을 대조",fontsize=14,weight="bold",color=INK)
    groups=[("SGCC",[240001]),("SPFH590",[240024,240108]),("SPCE",[240037,240121,240133]),
            ("SPCEN",[240038,240114]),("SPHC",[240044,240061,240069,240091]),
            ("DP590",[240058]),("SPCD",[240117])]
    for i,(grade,ids) in enumerate(groups):
        row,column=divmod(i,4);x=.045+column*.232;y=.237-row*.196
        ax=card(x,y,.213,.171,grade)
        for j,k in enumerate(ids):
            yy=.69-j*.175
            ax.plot([.015,.29],[yy,yy],color=LOT_COLORS[k],lw=2.6,ls="-")
            ax.text(.35,yy,str(k),va="center",fontsize=11.5,color=INK)
    ax=card(.741,.041,.213,.171,"비교 그림 읽는 법")
    ax.text(.01,.67,"대상 LOT  굵게\n대조 LOT  얇게\nTC 실선 / OP 점선",va="top",fontsize=11.5,color=MUTED,linespacing=1.65)
    Path(path).parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(path,dpi=EXPORT_DPI,facecolor=fig.get_facecolor(),bbox_inches="tight",pad_inches=.18)
    fig.set_dpi(72);display(fig);plt.close(fig)



def signal_linestyle(signal):
    return "--" if str(signal).startswith("OP-") else "-"


def _signal_kind(label, axis_text=""):
    label=str(label)
    if re.search(r"\bOP-Z\d",label) or label.startswith("OP "):return "OP"
    if re.search(r"\bTC-(?:Z\d|OUT)",label) or label=="CLN-IN" or label.startswith("TC "):return "TC"
    if label.startswith("CP-"):return "CP"
    if "출력" in label and "온도" not in label:return "OP"
    # A LOT label carries color identity; its panel supplies the measurement.
    if "제어출력" in axis_text or "출력[%]" in axis_text or "[%]" in axis_text:return "OP"
    if "온도" in axis_text or "[℃]" in axis_text:return "TC"
    if "단위 미확인" in axis_text and "CP" in axis_text:return "CP"
    return None


def style_signal_lines(fig):
    """Color identifies a zone/LOT; a dashed measured curve identifies OP."""
    for ax in fig.axes:
        titles=" ".join([ax.title.get_text(),ax._left_title.get_text(),ax._right_title.get_text()])
        context=ax.get_ylabel()+" "+titles
        for line in ax.lines:
            if len(line.get_xdata())<3 or line.get_linestyle() in ["None","none","",None]:continue
            if line.get_transform()!=ax.transData:continue
            label=line.get_label();kind=_signal_kind(label,context)
            if kind:line.set_linestyle("--" if kind=="OP" else "-")
            if re.fullmatch(r"(?:TC-Z[1-9]|OP-Z[1-8]|TC-OUT[12]|CP-Z4M?|CLN-IN)",str(label)):
                line.set_color(sensor_color(label))
        # A restrained axis rail carries process context; titles stay dark.
        found=[p for p in PROCESS_COLORS if p in titles or (p=="가열·균열" and ("가열" in titles or "균열" in titles))]
        if len(found)==1 and ax.axison:
            ax.spines["left"].set_color(PROCESS_COLORS[found[0]])
            ax.spines["left"].set_alpha(.65)
            ax.spines["left"].set_linewidth(1.0)
        for title in [ax.title,ax._left_title,ax._right_title]:
            if title.get_text() and str(title.get_color()).lower() not in ["#c83745","#d62728","#e11d48"]:
                title.set_color(INK)
        legend=ax.get_legend()
        if legend is not None:_style_legend(legend,context)
    for legend in fig.legends:_style_legend(legend)


def _style_legend(legend, context=""):
    handles=getattr(legend,"legend_handles",getattr(legend,"legendHandles",[]))
    for handle,text in zip(handles,legend.get_texts()):
        if not isinstance(handle,Line2D) or handle.get_linestyle() in ["None","none","",None]:continue
        label=text.get_text()
        if any(word in label for word in ["기준","중앙값","경계","감지","정비","수리","범위"]):continue
        kind=_signal_kind(label,context)
        if kind:handle.set_linestyle("--" if kind=="OP" else "-")
        if re.fullmatch(r"(?:TC-Z[1-9]|OP-Z[1-8]|TC-OUT[12]|CP-Z4M?|CLN-IN)",label):
            handle.set_color(sensor_color(label))


def _heading(title, subtitle):
    if title in _HEADINGS:
        return _HEADINGS[title]
    patterns = [
        (r"CAL01 정비 이력 (\d+)건", lambda m: (f"CAL01 정비 {m[1]}건의 작업 흐름", "작업별 감지·착수·완료와 관측창을 대조해 해석 범위 확인")),
        (r"남은 짧은 변화 · (\d+)", lambda m: (f"LOT {m[1]} 탐지 구간과 반복 파형 대조", "같은 LOT의 1초 원본으로 탐지 구간의 동시 변화 확인")),
        (r"같은 강종 LOT 전체 비교 · (\w+) · (.+)", lambda m: (f"{m[1]} LOT별 공정 신호 비교", "같은 강종·같은 신호의 경과시간으로 LOT 간 차이 확인")),
        (r"같은 강종 LOT 비교 · (\w+) · 대상 (\d+) · Z4", lambda m: (f"LOT {m[2]} 동일 강종 Z4 비교", "안정 참고 시작을 맞춰 같은 강종의 온도·출력 비교")),
        (r"(\d+) 정비 전·중·후", lambda m: (f"LOT {m[1]} 정비 전후 신호 비교", "같은 LOT의 실제 작업 경계와 온도·출력 변화를 대조")),
        (r"(\d+) CP 변화·정비", lambda m: (f"LOT {m[1]} CP·정비 시각 대조", "같은 LOT의 CP 극값과 정비·교체 기록의 시간 관계 확인")),
        (r"(\d+) · 언제 CP", lambda m: (f"LOT {m[1]} CP 동조 변화 구간", "안정 참고의 시간별 변동과 1초 동조로 변화 시점 확인")),
        (r"CP 펄스 뒤 무엇이 달라졌나 · (.+)", lambda m: (f"CAL01 CP 펄스 전후 반응 {m[1]}", "같은 LOT의 펄스 없는 안정 참고 시각과 전후 반응 비교")),
    ]
    for pattern, make in patterns:
        match = re.search(pattern, title)
        if match:
            return make(match)
    return title, subtitle


def header(fig, title, subtitle="", handles=(), ncol=4):
    original_title, original_subtitle = title, subtitle
    title, subtitle = _heading(title, subtitle)
    fig.suptitle(title, x=.035, y=.975, ha="left", va="top",
                 fontsize=23, fontweight="bold", color=INK)
    sub = fig.text(.035, .924, subtitle, ha="left", va="top", fontsize=12, color=MUTED)
    fig._report_subtitle = sub
    fig._report_original = {"title": original_title, "subtitle": original_subtitle}
    fig._report_details = []
    if handles:
        # Legend order is the caller's comparison order, never an alphabetic sort.
        labels = [h.get_label() for h in handles]
        fig.legend(handles=handles, labels=labels, loc="upper left", bbox_to_anchor=(.031, .879),
                   ncol=ncol, frameon=False, fontsize=11, handlelength=2.4,
                   columnspacing=1.5, labelspacing=.65)


def _panel_titles(fig, renderer):
    """Keep the identifying clause on the chart and retain the full caption."""
    for ax in fig.axes:
        title = ax._left_title if ax._left_title.get_text() else ax.title
        original = title.get_text()
        if title.get_color().lower() in [INK, "#17344d"]:
            title.set_color(process_title_color(original, title.get_color()))
        if not original or "\n" in original:
            continue
        if len(original) <= 66 and title.get_window_extent(renderer).width <= ax.bbox.width:
            continue
        # A long explanatory clause belongs below the figure. Do not cut a
        # label in the middle of a sensor name, quantity or comparison value.
        short = re.split(r" — | · (?=회색|위 |아래 |점 =|빨간|파란|평소|숫자|칸 =|세로)| \| ", original, maxsplit=1)[0]
        if short == original or len(short) > 66:
            continue
        fig._report_details.append(original)
        title.set_text(short)


def _legend_outside(fig, renderer):
    """Move an inside legend to a measured free title-row space when it fits."""
    moved = 0
    for ax in fig.axes:
        leg = ax.get_legend()
        if leg is None or not leg.get_window_extent(renderer).overlaps(ax.bbox):
            continue
        handles, labels = ax.get_legend_handles_labels()
        if not labels:
            continue
        candidate = ax.legend(handles, labels, loc="lower right", bbox_to_anchor=(1, 1.025),
                              borderaxespad=0, ncol=min(4, len(labels)), frameon=False,
                              fontsize=10.5, handlelength=2, columnspacing=1.2)
        fig.canvas.draw()
        box = candidate.get_window_extent(renderer)
        obstacles = [a.bbox for a in fig.axes if a is not ax and a.bbox != ax.bbox]
        texts = list(fig.texts)
        for other in fig.axes:
            texts += [other._left_title, other._right_title, other.title, other.xaxis.label]
            texts += list(other.texts)
        obstacles += [t.get_window_extent(renderer) for t in texts if t.get_visible() and t.get_text()]
        if any(box.overlaps(b) for b in obstacles) or box.y1 > fig.bbox.y1:
            candidate.remove()
            ax.add_artist(leg)
            ax.legend_ = leg
        else:
            candidate._detail_legend = True
            moved += 1
    return moved


def prepare(fig):
    """Arrange the figure header in physical units, preserving data limits."""
    if getattr(fig, "_report_prepared", False):
        return
    fig._report_prepared = True
    style_signal_lines(fig)
    fig._report_details = getattr(fig, "_report_details", [])
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    _panel_titles(fig, renderer)
    if hasattr(fig, "_report_subtitle"):
        height = fig.bbox.height
        gap = 9 * fig.dpi / 72
        title = fig._suptitle
        title.set_y(1 - .20 / fig.get_figheight())
        title.set_fontsize(min(25, title.get_fontsize()))
        fig.canvas.draw()
        sub = fig._report_subtitle
        sub.set_position((.035, (title.get_window_extent(renderer).y0 - gap) / height))
        sub.set_fontsize(12)
        sub.set_va("top")
        fig.canvas.draw()
        cursor = sub.get_window_extent(renderer).y0 - gap
        for legend in fig.legends:
            legend.set_bbox_to_anchor((.031, cursor / height), transform=fig.transFigure)
            fig.canvas.draw()
            cursor = legend.get_window_extent(renderer).y0 - gap
        # Existing charts already allocate a header. Grow it only if the
        # measured text requires additional room, including secondary axes.
        if fig.axes:
            positions = {ax: ax.get_position().frozen() for ax in fig.axes}
            low = min(p.y0 for p in positions.values())
            high = max(p.y1 for p in positions.values())
            visible_top = max((ax.get_tightbbox(renderer).y1 for ax in fig.axes if ax.get_visible()), default=high*height)
            extra = max(0, visible_top - high*height)
            target = (cursor - extra) / height
            target = min(target, high + .55 / fig.get_figheight())
            if abs(target-high) > .003:
                scale = (target-low) / (high-low)
                if scale < .55:
                    raise RuntimeError("그림 머리글에 비해 본문 높이가 부족합니다.")
                for ax, p in positions.items():
                    ax.set_position([p.x0, low+(p.y0-low)*scale, p.width, p.height*scale])
                for text in fig.texts:
                    if text in [title, sub]:
                        continue
                    x, y = text.get_position()
                    if low <= y <= high:
                        text.set_position((x, low+(y-low)*scale))
    fig.canvas.draw()
    fig._report_legends_moved = _legend_outside(fig, fig.canvas.get_renderer())


def details_html(fig):
    original = getattr(fig, "_report_original", {})
    details = [original.get("subtitle", ""), *getattr(fig, "_report_details", [])]
    details = list(dict.fromkeys(text for text in details if text))
    if not details:
        return ""
    body = "".join(f"<p>{html.escape(text)}</p>" for text in details)
    return '<details data-report-visuals="caption"><summary>그림 상세 기준·해석 펼치기</summary>' + body + "</details>"


# 각 노트북 안에서 완결되는 공통 그림 설정
from types import SimpleNamespace
report_viz = SimpleNamespace(
    EXPORT_DPI=EXPORT_DPI,
    EXPORT_PADDING=EXPORT_PADDING,
    PROCESS_COLORS=PROCESS_COLORS,
    GRADE_COLORS=GRADE_COLORS,
    LOT_COLORS=LOT_COLORS,
    MAINTENANCE_COLORS=MAINTENANCE_COLORS,
    MAINTENANCE_LABELS=MAINTENANCE_LABELS,
    MAINTENANCE_HATCHES=MAINTENANCE_HATCHES,
    ZONE_COLORS=ZONE_COLORS,
    SENSOR_COLORS=SENSOR_COLORS,
    LOT_LINESTYLES=LOT_LINESTYLES,
    header=header,
    prepare=prepare,
    details_html=details_html,
    sensor_color=sensor_color,
    lot_linestyle=lot_linestyle,
    process_title_color=process_title_color,
    palette_preview=palette_preview,
    signal_linestyle=signal_linestyle,
    style_signal_lines=style_signal_lines,
)
configure()


## 자료 읽기와 연결

기존 01과 같은 규칙으로 읽습니다. LOT 시작과 CR1 강종변경 시각의 정확 일치를 확인하고, CAL01 설비의 WO와 부품교체를 실제 키로 연결합니다. 수리의 마지막 날짜는 하루 전체를 포함하는 계획으로 읽습니다.

In [ ]:
NOTEBOOK_NO = "05"
from pathlib import Path
from IPython.display import display, Markdown, Image
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator
from scipy.signal import find_peaks

temp = pd.read_csv(TEMP_CSV, encoding=ENC)
event = pd.read_csv(EVENT_CSV, encoding=ENC)
repair = pd.read_csv(REPAIR_CSV, encoding=ENC)
eqp = pd.read_csv(EQP_CSV, encoding=ENC)
part = pd.read_csv(PART_CSV, encoding=ENC)
maint = pd.read_csv(MAINT_CSV, encoding=ENC)
partchg = pd.read_csv(PARTCHG_CSV, encoding=ENC)

for frame, columns in [(temp, ["MEAS_DT"]), (event, ["EVT_DT"]),
                       (repair, ["STRT_DT", "END_DT"]),
                       (maint, ["DETC_DT", "STRT_DT", "END_DT"])]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column])
CAL_TAG = "P1-CR1-CAL01"
YEAR_START, YEAR_END = pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01")
grade_evt = event.loc[event.PLANT_CD.eq("CR1") & event.EVT_TYPE.eq("강종변경")].sort_values("EVT_DT")
assert not grade_evt.EVT_DT.duplicated().any()
lots = temp.groupby("LOT_NO", as_index=False).agg(
    START=("MEAS_DT", "min"), END=("MEAS_DT", "max"), N_ROWS=("MEAS_DT", "size"))
lots = lots.merge(grade_evt[["EVT_DT", "AFT_VAL"]], left_on="START", right_on="EVT_DT",
                  how="left", validate="one_to_one").rename(columns={"AFT_VAL": "GRADE"})
lots = lots.sort_values("START").reset_index(drop=True)
assert lots.GRADE.notna().all()
lots["MINUTES"] = (lots.END - lots.START).dt.total_seconds() / 60
lot_meta = lots.set_index("LOT_NO")
cal_work = maint.loc[maint.EQP_TAG.eq(CAL_TAG)].sort_values("STRT_DT").copy()
assert cal_work.WO_NO.is_unique and cal_work.END_DT.ge(cal_work.STRT_DT).all()
cal_parts = (partchg.merge(cal_work[["WO_NO", "EQP_TAG", "MNT_TYPE", "DETC_DT", "STRT_DT", "END_DT", "REMARK"]],
                           on="WO_NO", validate="many_to_one")
             .merge(part, on="PART_CD", validate="many_to_one").sort_values("END_DT"))
plans = repair.loc[repair.PLANT_CD.eq("CR1")].sort_values("STRT_DT").copy()
plans["END_EXCL"] = plans.END_DT.dt.normalize() + pd.Timedelta(days=1)
TC = ["CLN-IN"] + [f"TC-Z{i}" for i in range(1, 10)] + ["TC-OUT1", "TC-OUT2"]
OP = [f"OP-Z{i}" for i in range(1, 9)]
SIGNALS = TC + OP + ["CP-Z4", "CP-Z4M"]
HEAT = [f"TC-Z{i}" for i in range(3, 7)]
PROCESS = {"CLN-IN": "세정", **{f"TC-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 10)},
           "TC-OUT1": "냉각", "TC-OUT2": "냉각",
           **{f"OP-Z{i}": "건조" if i < 3 else "가열·균열" if i < 7 else "과시효" for i in range(1, 9)},
           "CP-Z4": "가열·균열", "CP-Z4M": "가열·균열"}
LOT_COLOR = report_viz.LOT_COLORS.copy()
COL = {"LOT": "#087f9c", "GRADE": "#477eb5", "BM": report_viz.MAINTENANCE_COLORS["BM"],
       "CBM": report_viz.MAINTENANCE_COLORS["CBM"], "TBM": report_viz.MAINTENANCE_COLORS["TBM"], "PART": "#7540a8", "PLAN": "#b8e2f2",
       "WORK": "#9a6324", "DETECT": "#34495e", "FOCUS": "#c83745", "QUALITY": "#bfc8cf"}
DUMMY_PRIOR = {"SPHC": "사용자 후보 가정", "SPCC": "사용자 후보 가정", "SPCE": "사용자 후보 가정",
               "SPCEN": "사용자 후보 가정", "SPFH590": "사용자 제외 가정", "DP590": "사용자 제외 가정",
               "SGCC": "사용자 제외 가정", "SPCD": "추가 검토"}
lots["DUMMY_PRIOR"] = lots.GRADE.map(DUMMY_PRIOR)
context_rows = []
for r in lots.itertuples():
    tasks = cal_work.loc[(cal_work.STRT_DT < r.END) & (cal_work.END_DT > r.START)]
    detected = cal_work.loc[cal_work.DETC_DT.between(r.START, r.END)]
    relationships = []
    for p in plans.itertuples():
        before = lots.loc[lots.END < p.STRT_DT].tail(1)
        after = lots.loc[lots.START >= p.END_EXCL].head(1)
        if len(before) and before.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 전 마지막 관측")
        if len(after) and after.LOT_NO.iloc[0] == r.LOT_NO:
            relationships.append("수리 후 첫 관측")
    context_rows.append({"LOT": int(r.LOT_NO), "GRADE": r.GRADE, "START": r.START, "END": r.END,
                         "MINUTES": r.MINUTES, "CAL_WORK": len(tasks), "CAL_DETECT": len(detected),
                         "PLAN_RELATION": " · ".join(relationships),
                         "DUMMY_PRIOR": DUMMY_PRIOR[r.GRADE]})
context = pd.DataFrame(context_rows).sort_values("START")
OUT_DIR = Path(FIG_DIR) / f"05_all_data_timeseries_jinny_{NOTEBOOK_NO}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
figure_checks = []
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)

def note(text):
    display(Markdown(text))

def table(frame, columns=None):
    shown = frame if columns is None else frame[columns]
    # 작은 CP-Z4M도 0으로 반올림하지 않고 과학 표기법으로 보여준다.
    with pd.option_context("display.float_format", lambda value: f"{value:.4g}"):
        display(shown)

# 정비 종류는 색으로도 구분한다.
COL.update({"BM":report_viz.MAINTENANCE_COLORS["BM"],"CBM":report_viz.MAINTENANCE_COLORS["CBM"],"TBM":report_viz.MAINTENANCE_COLORS["TBM"]})

## 그림 표시 규칙

01~05는 공통 색상·제목·범례 규칙을 사용합니다. 제목은 대상과 비교를 짧게 표시하고, 긴 정의와 해석은 그림 아래 **그림 상세 기준·해석 펼치기**에 둡니다.

- **범주 구분:** 공정은 패널 제목, 강종은 그룹 표식, LOT는 비교선, 정비유형은 작업 구간의 색으로 구분합니다. 같은 항목은 모든 노트북에서 같은 색을 씁니다.
- **색 외의 구분:** 같은 강종 LOT는 대비되는 고정 색으로 비교합니다. TC·CP는 실선, OP는 점선으로 표시하며, 같은 존의 TC·OP는 같은 색을 사용합니다. 검토 지점은 기존 (a)·(b)·(c) 또는 구간 번호와 확대 그림을 연결합니다.

- **축과 단위:** 실제 시각과 경과 분을 구분합니다. TC의 ℃·OP의 %는 원본에서 확인된 단위입니다. CP는 단위 미확인 기록값입니다.
- **검토:** 저장 크기에서 제목·축·눈금·범례·주석의 겹침과 잘림을 검사합니다.


In [ ]:
rcParams.update({"font.size": 11.5, "axes.titlesize": 12.5, "axes.labelsize": 11.5,
                 "xtick.labelsize": 10, "ytick.labelsize": 10.5, "savefig.dpi": 160})

def unit(signal):
    if signal in TC:
        return "[℃]"
    if signal in OP:
        return "[%]"
    if signal == "CP-Z4M":
        return "[단위 미확인]"
    if signal == "CP-Z4":
        return "[단위 미확인]"
    return signal

def style(ax, xlabel="관측 시작 후 경과시간[분]"):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#a9bac7")
    ax.grid(color="#e4ebf0", linewidth=.6)
    ax.set_axisbelow(True)
    ax.set_xlabel(xlabel, labelpad=8)
    ax.tick_params(pad=5)

def header(fig, title, subtitle="", handles=(), ncol=4):
    return report_viz.header(fig, title, subtitle, handles, ncol)

def lot_handles(ids):
    return [Line2D([0], [0], color=LOT_COLOR[int(k)], lw=2,
                   label=f"{int(k)} · {lot_meta.loc[int(k), 'GRADE']} · {lot_meta.loc[int(k), 'START']:%m/%d}", ls=report_viz.lot_linestyle(int(k))) for k in ids]

def text_box(artist, renderer):
    patch = artist.get_bbox_patch() if hasattr(artist, "get_bbox_patch") else None
    return patch.get_window_extent(renderer) if patch is not None else artist.get_window_extent(renderer)


# 설명은 글자 높이와 짧은 간격에 맞춰 모으고, 확보한 공간을 그래프에 돌려준다.
layout_reviews = []

def compact_explanations(fig):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    axes = list(fig.axes)
    visible = [ax for ax in axes if ax.axison]
    area_before = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report = {"CAPTIONS": 0, "AREA_BEFORE": area_before, "AREA_AFTER": area_before,
              "GAIN_PCT": 0.0, "LIMITS_PRESERVED": True}
    if not axes or not visible:
        return report
    positions = {ax: ax.get_position().frozen() for ax in axes}
    bottom = min(p.y0 for p in positions.values())
    top = max(p.y1 for p in positions.values())
    captions = [text for text in fig.texts
                if text is not fig._suptitle and text.get_visible() and text.get_text()
                and text.get_position()[1] < bottom - .003]
    if not captions:
        return report
    # 표·설명 상자처럼 전체 배치가 하나인 그림은 그대로 둔다.
    for artist in fig.artists:
        if hasattr(artist, "get_xy") and hasattr(artist, "get_height"):
            if artist.get_y() < bottom and artist.get_height() > .02:
                return report
    figure_height = fig.get_figheight()
    gap = 7 / 72 / figure_height
    cursor = 10 / 72 / figure_height
    caption_ids = {id(text) for text in captions}
    heights = {id(text): text_box(text, renderer).height / fig.bbox.height for text in captions}
    for text in sorted(captions, key=lambda t: t.get_position()[1]):
        text.set_position((text.get_position()[0], cursor))
        text.set_verticalalignment("bottom")
        cursor += heights[id(text)] + gap
    # 축 아래의 눈금·단위가 설명과 겹치지 않도록 실제 글자 크기를 반영한다.
    depth = 0.0
    for ax in visible:
        if positions[ax].y0 > bottom + .025:
            continue
        labels = [ax.xaxis.label] + ax.get_xticklabels()
        boxes = [text_box(t, renderer) for t in labels if t.get_visible() and t.get_text()]
        if boxes:
            depth = max(depth, bottom - min(b.y0 for b in boxes) / fig.bbox.height)
    target_bottom = max(.065, cursor + depth + gap)
    limits = [(ax.get_xlim(), ax.get_ylim()) for ax in axes]
    if target_bottom < bottom:
        scale = (top - target_bottom) / (top - bottom)
        for ax, old in positions.items():
            ax.set_position([old.x0, target_bottom + (old.y0-bottom)*scale,
                             old.width, old.height*scale])
        for text in fig.texts:
            if id(text) in caption_ids:
                continue
            x, y = text.get_position()
            if bottom <= y <= top:
                text.set_position((x, target_bottom + (y-bottom)*scale))
    # 알파벳은 축 높이와 무관하게 파형 위 8pt 위치에 놓는다.
    for ax in axes:
        height = fig.get_figheight() * ax.get_position().height
        for text in ax.texts:
            if text.get_transform() == ax.transAxes and text.get_text() in ["(a)","(b)","(c)"]:
                text.set_y(1 + 8 / 72 / height)
                text.set_verticalalignment("bottom")
        for line in ax.lines:
            if line.get_transform() != ax.transAxes:
                continue
            y = np.asarray(line.get_ydata(), dtype=float)
            if len(y) == 2 and np.all(y > 1) and np.max(y) < 1.2:
                line.set_ydata([1 + 1 / 72 / height, 1 + 7 / 72 / height])
    assert limits == [(ax.get_xlim(), ax.get_ylim()) for ax in axes], "배치 변경으로 축 범위가 달라짐"
    fig.canvas.draw()
    area_after = sum(ax.get_position().width * ax.get_position().height for ax in visible)
    report.update({"CAPTIONS": len(captions), "AREA_AFTER": area_after,
                   "GAIN_PCT": 100*(area_after/area_before-1)})
    return report

# 저장 파일과 노트북 화면 모두 그림 네 변의 여백을 같게(0.2인치) 둔다.
PAD = report_viz.EXPORT_PADDING
rcParams["savefig.pad_inches"] = PAD

def finish(fig, name, svg=False):
    """저장할 크기에서 모든 글자와 범례의 겹침·잘림을 확인한다."""
    report_viz.style_signal_lines(fig)
    add_non_lot_observation(fig, name)
    fig.set_dpi(160)
    layout = separate_observation_captions(fig)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for ax in fig.axes:
        for tag in ax.texts:
            if getattr(tag, "_review_label", False):
                assert not text_box(tag, renderer).overlaps(ax.get_window_extent(renderer)), "확인 표시가 파형을 가림"
    artists = list(fig.texts) + list(fig.legends)
    for ax in fig.axes:
        if not ax.axison:
            continue
        artists += [ax.title, ax._left_title, ax.xaxis.label, ax.yaxis.label,
                    ax.xaxis.offsetText, ax.yaxis.offsetText] + list(ax.texts)
        if ax.get_legend() is not None and getattr(ax.get_legend(), "_detail_legend", False):
            artists.append(ax.get_legend())
        for axis in [ax.xaxis, ax.yaxis]:
            low, high = sorted(axis.get_view_interval())
            for tick in axis.get_major_ticks():
                if low <= tick.get_loc() <= high:
                    artists += [tick.label1, tick.label2]
    unique = {}
    for artist in artists:
        if not artist.get_visible():
            continue
        if hasattr(artist, "get_text") and not artist.get_text():
            continue
        unique[id(artist)] = artist
    boxes = [(artist, text_box(artist, renderer)) for artist in unique.values()]
    for artist, box in boxes:
        if box.x0 < -.5 or box.y0 < -.5 or box.x1 > fig.bbox.x1 + .5 or box.y1 > fig.bbox.y1 + .5:
            raise RuntimeError(f"글자 잘림: {name}: {getattr(artist, 'get_text', lambda: '범례')()}")
    for i, (artist, box) in enumerate(boxes):
        for other, other_box in boxes[i + 1:]:
            if box.overlaps(other_box):
                raise RuntimeError(f"글자 겹침: {name}: {getattr(artist, 'get_text', lambda: '범례')()} / {getattr(other, 'get_text', lambda: '범례')()}")
    for legend in list(fig.legends) + [ax.get_legend() for ax in fig.axes if ax.get_legend() is not None and getattr(ax.get_legend(), "_detail_legend", False)]:
        box = legend.get_window_extent(renderer)
        if any(box.overlaps(ax.get_window_extent(renderer)) for ax in fig.axes if ax.axison):
            raise RuntimeError(f"범례가 그래프를 가림: {name}")
    # 여백을 잘라 저장할 때 그림 위쪽 범례만 따로 밀리지 않도록, 범례 위치를 그림 좌표에 다시 묶는다.
    for legend in fig.legends:
        anchor = legend.get_bbox_to_anchor().transformed(fig.transFigure.inverted())
        legend.set_bbox_to_anchor(anchor, transform=fig.transFigure)
    path = OUT_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=report_viz.EXPORT_DPI, facecolor="white", bbox_inches="tight", pad_inches=PAD)
    if svg:
        vector = path.with_suffix(".svg")
        fig.savefig(vector, facecolor="white", bbox_inches="tight", pad_inches=PAD)
        vector.write_text("\n".join(line.rstrip() for line in vector.read_text(encoding="utf-8").splitlines()) + "\n", encoding="utf-8")
    record_observation_captions(fig, name)
    figure_checks.append({"FILE": name, "TITLE": fig._suptitle.get_text(), "N_AXES": len(fig.axes),
                          "TEXT_OVERLAP": "통과", "TEXT_BOUNDS": "통과", "LEGEND_BOUNDS": "통과", "CAPTION_GAP_PT": fig._observation_gap_pt})
    layout_reviews.append({"FILE": name, **layout})
    # 저장한 파일을 다시 불러오지 않고, 방금 그린 그림을 노트북에 바로 그린다.
    # 파일은 위에서 원본 해상도(180dpi)로 저장했고, 노트북 화면용으로만 해상도를 낮춘다.
    fig.set_dpi(72)
    display(fig)
    caption = report_viz.details_html(fig)
    if caption:
        from IPython.display import HTML
        display(HTML(caption))
    plt.close(fig)

In [ ]:
import matplotlib

# 시간축 태그: 그림 오른쪽 위에 이 그림을 어떤 기준으로 읽는지 적는다.
TAG_REAL = "TIME AXIS · 실제 측정 날짜·시각"
TAG_ELAPSED = "TIME AXIS · 관측 시작 후 경과시간[분]"
TAG_REL = "TIME AXIS · LOT 시작 기준 상대시간[시간]"

def axis_tag(fig, text):
    fig.text(.975, .978, text, ha="right", va="top", fontsize=13, color="#17344d",
             bbox=dict(boxstyle="round,pad=.45", fc="#eef3f7", ec="#9fb3c3", lw=.9))

def compare_tag(target):
    return f"COMPARISON · {target}"

def elapsed_top(ax, start, label="경과[분]"):
    """실제 시각 축 위에 같은 LOT의 관측 시작 후 경과 분을 함께 적는다."""
    base = mdates.date2num(start)
    top = ax.secondary_xaxis("top", functions=(lambda x: (x - base) * 1440, lambda m: base + m / 1440))
    top.set_xlabel(label, fontsize=11, color="#526d80", labelpad=6)
    top.tick_params(labelsize=10, colors="#526d80")
    top.xaxis.set_major_locator(MaxNLocator(6))
    return top

def compact_header(fig, sub_y, legend_y, title_size=25, sub_size=14, legend_size=13):
    """키 큰 그림에서 제목·부제·범례 사이 빈 공간을 줄이고 글자를 키운다."""
    fig._suptitle.set_fontsize(title_size)
    for text in fig.texts:
        if np.isclose(text.get_position()[1], .924):
            text.set_position((.035, sub_y))
            text.set_fontsize(sub_size)
    if fig.legends:
        fig.legends[0].set_bbox_to_anchor((.031, legend_y))
        for text in fig.legends[0].get_texts():
            text.set_fontsize(legend_size)

def shades(color, n=4):
    """LOT 색 하나를 진한 색 → 밝은 색으로 나눈다. 구역 순서를 같은 LOT 안에서 구분한다."""
    rgb = np.array(matplotlib.colors.to_rgb(color))
    dark, light = rgb * .55, rgb * .62 + .38
    return [tuple(dark * (1 - t) + light * t) for t in np.linspace(0, 1, n)]

## 상세 시계열 준비

1초 기대 간격의 품질, 1분 집계, 사후 형태 상태(초기 승온·종료 전환·안정 참고), 같은 강종·안정 참고·같은 경과 분 비교, 실제 작업 교집합을 계산한다. 기존 01~08과 같은 코드다.

In [ ]:
# 1초 기대 간격은 원본 시각 차이의 최빈값으로 확인한다. 결측은 보간하지 않는다.
raw, profile, quality, expected, states = {}, {}, {}, {}, {}
quality_rows, state_rows = [], []
STATE_COLOR = {"안정 참고": "#009e73", "초기 승온": "#e69f00", "종료 전환": "#d55e00"}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = temp.loc[temp.LOT_NO.eq(k)].sort_values("MEAS_DT").copy()
    assert not g.MEAS_DT.duplicated().any()
    gaps_s = g.MEAS_DT.diff().dt.total_seconds().dropna()
    assert gaps_s.mode().iloc[0] == 1 and gaps_s.ge(1).all()
    g["ELAPSED"] = (g.MEAS_DT - r.START).dt.total_seconds() / 60
    g["MINUTE"] = np.floor(g.ELAPSED).astype(int)
    grid = pd.RangeIndex(int(np.floor(r.MINUTES)) + 1, name="MINUTE")
    p = g.groupby("MINUTE")[SIGNALS].agg(["median", "min", "max", "count"]).reindex(grid)
    exp = pd.Series(np.minimum(60, int((r.END-r.START).total_seconds()) + 1 - grid.to_numpy()*60), index=grid)
    counts = g.groupby("MINUTE").size().reindex(grid, fill_value=0)
    q = p.xs("count", axis=1, level=1).fillna(0).div(exp, axis=0)
    raw[k], profile[k], quality[k], expected[k] = g, p, q, exp
    for s in SIGNALS:
        quality_rows.append({"LOT": k, "GRADE": r.GRADE, "SENSOR": s, "ROWS": len(g),
                             "EXPECTED_SECONDS": int(exp.sum()), "MISSING_TIMES_S": int(exp.sum()-len(g)),
                             "SENSOR_NAN": int(g[s].isna().sum()), "VALID_RATIO": g[s].count()/exp.sum(),
                             "LOW_90_WINDOWS": int(q[s].lt(.9).sum())})
    # 사후 형태 분류다. 실제 운전 모드나 온라인 모델 입력으로 사용하지 않는다.
    st = pd.Series("안정 참고", index=grid, dtype="object")
    early = g.loc[g.ELAPSED.lt(5), HEAT].median()
    later = g.loc[g.ELAPSED.between(80,100), HEAT].median()
    rise = later - early
    warm_end, tail_start = np.nan, np.nan
    if rise.notna().all() and rise.min() >= 20 and g.ELAPSED.between(80,100).shape[0] >= 600:
        near = pd.concat([(p[(s,"median")]-later[s]).abs().le(5 if s=="TC-Z3" else 2) for s in HEAT], axis=1).all(axis=1)
        slow = pd.concat([p[(s,"median")].diff(5).abs().le(2) for s in HEAT[1:]], axis=1).all(axis=1)
        run = (near & slow & grid.to_series().ge(5)).rolling(5, min_periods=5).sum().eq(5)
        warm_end = int(run.index[run][0])+1 if run.any() else len(grid)
        st.loc[grid < warm_end] = "초기 승온"
    middle = g.loc[g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6), HEAT].median()
    tail = g.loc[g.ELAPSED.ge(max(0,r.MINUTES-5)), HEAT].median()
    if (tail-middle).notna().all() and (tail-middle).max() <= -10:
        midmask = g.ELAPSED.between(r.MINUTES*.45,r.MINUTES*.6)
        drop = pd.concat([(p[(s,"median")]-middle[s]).le(-10) for s in HEAT], axis=1).sum(axis=1).ge(3)
        cp_drop = (p[("CP-Z4","median")]-g.loc[midmask,"CP-Z4"].median()).le(-.05).rolling(3).sum().eq(3)
        op = ["OP-Z4","OP-Z5","OP-Z6"]
        op_drop = (p.loc[:,[(s,"median") for s in op]].median(axis=1)-g.loc[midmask,op].median().median()).le(-10).rolling(3).sum().eq(3)
        hit = (drop | cp_drop | op_drop) & grid.to_series().ge(r.MINUTES*.6)
        if hit.any():
            tail_start = int(hit.index[hit][0])
            st.loc[grid >= tail_start] = "종료 전환"
    states[k] = st
    state_rows.append({"LOT": k, "GRADE": r.GRADE, "RISE_MIN_C": rise.min(), "RISE_MAX_C": rise.max(),
                       "WARM_END_MIN": warm_end, "TAIL_START_MIN": tail_start,
                       "STABLE_WINDOWS": int(st.eq("안정 참고").sum())})
quality_summary = pd.DataFrame(quality_rows)
state_summary = pd.DataFrame(state_rows)
def runs(mask, minimum=1):
    """연속 True의 [시작, 끝 제외) 경과 분을 돌려준다."""
    a = np.flatnonzero(mask.to_numpy(bool))
    if not len(a):
        return []
    groups = np.split(a, np.flatnonzero(np.diff(a)>1)+1)
    return [(int(mask.index[z[0]]), int(mask.index[z[-1]])+1) for z in groups if len(z)>=minimum]

In [ ]:
def line(ax, lot, signal, start=None, end=None, clock=False, raw_values=False, envelope=False):
    k = int(lot)
    if raw_values:
        r = lot_meta.loc[k]
        a = max(0, start or 0)
        b = min(r.MINUTES, r.MINUTES if end is None else end)
        # 표시용 1초 격자에만 재배치하여 결측 시각에서 선을 끊는다.
        times = pd.date_range(r.START+pd.Timedelta(seconds=int(a*60)), r.START+pd.Timedelta(seconds=int(b*60)), freq="s")
        p = raw[k].set_index("MEAS_DT").reindex(times)
        x = times if clock else (times-r.START).total_seconds()/60
        y = p[signal].to_numpy()
        low = quality[k][signal].loc[quality[k].index.to_series().between(np.ceil(a),np.floor(b))].lt(.9)
        if low.any():
            marks = low.index[low].to_numpy()
            qx = r.START + pd.to_timedelta(marks,unit="min") if clock else marks
            ax.scatter(qx,np.full(len(marks),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    else:
        p = profile[k]
        p = p.loc[(p.index >= (0 if start is None else start)) & (p.index <= (np.inf if end is None else end))]
        x = lot_meta.loc[k,"START"] + pd.to_timedelta(p.index,unit="min") if clock else p.index.to_numpy()
        y = p[(signal,"median")].to_numpy()
        if envelope:
            ax.fill_between(x,p[(signal,"min")].to_numpy(),p[(signal,"max")].to_numpy(),color=LOT_COLOR[k],alpha=.14,lw=0)
        low = quality[k].loc[p.index,signal].lt(.9)
        if low.any():
            ax.scatter(np.asarray(x)[low],np.full(low.sum(),.02),transform=ax.get_xaxis_transform(),marker="|",color="#7c8995",s=25,lw=.6,zorder=5)
    ax.plot(x,y,color=LOT_COLOR[k],lw=1.8, ls=report_viz.lot_linestyle(k))
    ax.set_title(f"{PROCESS[signal]} [{signal}]",loc="left",pad=12)
    ax.set_ylabel(unit(signal),labelpad=10)
    style(ax,"관측 시각[월/일 시:분]" if clock else "관측 시작 후 경과시간[분]")
    ax.xaxis.set_major_locator(MaxNLocator(5)) if not clock else None
    ax.yaxis.set_major_locator(MaxNLocator(4))
    if signal == "CP-Z4M":
        ax.ticklabel_format(axis="y",style="sci",scilimits=(0,0))
    if clock:
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))

QUALITY_HANDLE = Line2D([0],[0],color="#7c8995",marker="|",linestyle="None",markersize=10,label="분당 유효 기록 90% 미만")
FOCUS_HANDLE = Patch(facecolor=COL["FOCUS"],alpha=.12,label="검토·확대 시간창")
ENVELOPE_HANDLE = Patch(facecolor="#97b5cc",alpha=.35,label="각 LOT 색 음영: 분당 최소–최대")

In [ ]:
# 같은 강종·안정 참고 상태·같은 경과 분에서 대상 LOT을 뺀 대조만 쓴다.
# 비교할 수 없는 후반 구간을 잘라내거나 대조 중앙값으로 채우지 않는다.
comparison, sensor_rows, episode_rows = {}, [], []
NONCYCLIC = [s for s in TC if s not in ["TC-Z9","TC-OUT1","TC-OUT2"]]
for grade, sub in lots.groupby("GRADE",sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    grid = pd.RangeIndex(max(len(profile[k]) for k in ids))
    for s in SIGNALS:
        values = pd.DataFrame({k:profile[k][(s,"median")].where(quality[k][s].ge(.9)&states[k].eq("안정 참고")).reindex(grid) for k in ids})
        for k in ids:
            peers = values.drop(columns=k)
            n = peers.count(axis=1)
            center = peers.median(axis=1)
            delta = (values[k]-center).where(n.ge(1))
            d = pd.DataFrame({"VALUE":values[k],"CENTER":center,"N_PEERS":n,"DELTA":delta})
            comparison[k,s] = d
            v = delta.dropna()
            sensor_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"COMPARE_WINDOWS":len(v),
                                "MEDIAN_DELTA":v.median(),"Q05_DELTA":v.quantile(.05),"Q95_DELTA":v.quantile(.95),
                                "MAX_ABS_DELTA":v.abs().max(),"MIN_PEERS":int(n.loc[v.index].min()) if len(v) else 0})
            if s in NONCYCLIC:
                for a,b in runs(delta.abs().ge(3)&n.ge(2),3):
                    block = d.loc[a:b-1]
                    episode_rows.append({"LOT":k,"GRADE":grade,"SENSOR":s,"START_MIN":a,"END_MIN":b,
                                         "DURATION_MIN":b-a,"MAX_ABS_DELTA_C":block.DELTA.abs().max(),
                                         "MIN_PEERS":int(block.N_PEERS.min())})
sensor_summary = pd.DataFrame(sensor_rows)
episodes = pd.DataFrame(episode_rows,columns=["LOT","GRADE","SENSOR","START_MIN","END_MIN","DURATION_MIN","MAX_ABS_DELTA_C","MIN_PEERS"])
# CP는 단위 미확인이다. 자체 참고 중앙값과 차이를 탐색하며 정상 한계로 해석하지 않는다.
cp_rows, cp_flags = [], {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    g = raw[k]
    ref = profile[k][("CP-Z4","median")].where(states[k].eq("안정 참고")&quality[k]["CP-Z4"].ge(.9)).median()
    hit = (g["CP-Z4"]-ref).abs().ge(.05)
    cp_flags[k] = pd.Series(profile[k].index.isin(g.loc[hit,"MINUTE"]),index=profile[k].index)
    if hit.any():
        j = (g["CP-Z4"]-ref).abs().idxmax()
        peak = g.loc[j]
        # 5초 초과 시각 공백에서 분리한다. 짧은 공백도 보간하지 않고 실제 기록 행수·최대 간격을 따로 남긴다.
        cluster = ((hit != hit.shift())|g.MEAS_DT.diff().dt.total_seconds().gt(5)).cumsum()
        pulse = g.loc[cluster.eq(cluster.loc[j])&hit]
        cp_rows.append({"LOT":k,"GRADE":r.GRADE,"BASE":ref,"PEAK_MIN":peak.ELAPSED,"PEAK_TIME":peak.MEAS_DT,
                        "PEAK_VALUE":peak["CP-Z4"],"DELTA":peak["CP-Z4"]-ref,
                        "START_MIN":pulse.ELAPSED.min(),"END_MIN":pulse.ELAPSED.max()+1/60,
                        "PULSE_WIDTH_MIN":(pulse.MEAS_DT.max()-pulse.MEAS_DT.min()).total_seconds()/60+1/60,
                        "MAX_RECORD_GAP_S":pulse.MEAS_DT.diff().dt.total_seconds().max(),
                        "RECORDED_SECONDS":len(pulse),"TOTAL_HIT_SECONDS":int(hit.sum()),
                        "TOTAL_HIT_MIN_WINDOWS":int(cp_flags[k].sum())})
cp_candidates = pd.DataFrame(cp_rows,columns=["LOT","GRADE","BASE","PEAK_MIN","PEAK_TIME","PEAK_VALUE","DELTA","START_MIN","END_MIN","PULSE_WIDTH_MIN","MAX_RECORD_GAP_S","RECORDED_SECONDS","TOTAL_HIT_SECONDS","TOTAL_HIT_MIN_WINDOWS"])

In [ ]:
# 관측 범위와 실제 CAL01 작업의 교집합을 경과 분으로 계산한다. 같은 시각의 작업은 한 번만 센다.
def merge_ranges(intervals):
    merged = []
    for start, end in sorted(intervals):
        if end <= start:
            continue
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((start, end))
    return merged

work_ranges = {}
for r in lots.itertuples():
    k = int(r.LOT_NO)
    tasks = cal_work.loc[cal_work.STRT_DT.lt(r.END) & cal_work.END_DT.gt(r.START)]
    work_ranges[k] = merge_ranges([((max(r.START, t.STRT_DT) - r.START).total_seconds() / 60,
                                    (min(r.END, t.END_DT) - r.START).total_seconds() / 60) for t in tasks.itertuples()])
assert np.isclose(sum(b - a for a, b in work_ranges[240117]), 304)

In [ ]:
hypotheses = []
def verdict(hypothesis, result, evidence, reason, use, limit="운전 설정·인과·실제 고장 정보 부족", source="기존 03"):
    hypotheses.append({"가설": hypothesis, "판정": result, "검증 근거": evidence, "판정 이유": reason,
                       "08 활용": use, "남은 한계": limit, "출처": source})

def raw_window(lot, a, b):
    """1초 격자에 원본을 놓아 기록 공백에서 선을 끊는다. 값을 채우지 않는다."""
    r = lot_meta.loc[lot]
    times = pd.date_range(r.START + pd.Timedelta(seconds=int(a * 60)), min(r.END, r.START + pd.Timedelta(seconds=int(b * 60))), freq="s")
    return raw[lot].set_index("MEAS_DT").reindex(times)

def clock_axis(ax, day):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3, maxticks=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    style(ax, f"실제 측정 시각[{day:%m/%d} 시:분]")

def clock_top(ax, start, label="같은 LOT 실제 시각[시:분]"):
    """경과 분 축 위에 대상 LOT의 실제 시각을 함께 적는다."""
    base = mdates.date2num(start)
    top = ax.secondary_xaxis("top", functions=(lambda m: base + m / 1440, lambda d: (d - base) * 1440))
    top.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    top.set_xlabel(label, fontsize=11, color="#526d80", labelpad=6)
    top.tick_params(labelsize=10, colors="#526d80")
    return top

def mean30(k, cols):
    """팀 자료와 같은 집계: 센서별 유효값 30개 이상인 1분 평균."""
    g = raw[k].set_index("MEAS_DT")[cols]
    return g.resample("1min").mean().where(g.resample("1min").count().ge(30))

def sustained_from(series, limit):
    """기준보다 낮은 상태가 관측 끝까지 이어지는 구간의 시작 시각."""
    s = series.dropna(); low = s.lt(limit)
    if not low.iloc[-1]:
        return None
    i = len(low) - 1
    while i > 0 and low.iloc[i - 1]:
        i -= 1
    return low.index[i]

In [ ]:
# ------------------------------------------------------------------------------------------------
# 예정된 변화와 탐지 구간 분류용 기준 ①~④ (01~04가 같은 코드로 계산한다 · 최종 확인 대상은 04에서 정한다)
# ------------------------------------------------------------------------------------------------
# 예정된 변화: 계획수리 직전의 종료 전환(끄면 출력·온도가 함께 내려감), 직후의 초기 승온(켜면 함께 올라감)
relation = context.set_index("LOT").PLAN_RELATION
expected_rows = []
for k in lots.LOT_NO.astype(int):
    for state, kind in [("종료 전환", "예정된 정지"), ("초기 승온", "재가동 승온")]:
        for a, b in runs(states[k].eq(state)):
            near_plan = bool(relation.get(k))
            expected_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "종류": kind if near_plan else state + " · 일정 없음",
                                  "START_MIN": float(a), "END_MIN": float(min(b, lot_meta.loc[k, "MINUTES"])),
                                  "시작": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(a)),
                                  "끝": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=float(min(b, lot_meta.loc[k, "MINUTES"]))),
                                  "수리 관계": relation.get(k) or "-"})
expected = pd.DataFrame(expected_rows)

def in_expected(k, minute):
    d = expected.loc[expected.LOT.eq(k)]
    return bool(((d.START_MIN <= minute) & (d.END_MIN > minute)).any())

# ① 출력과 온도가 반대로 움직인 구간: 출력 0%(≤2%)가 5분 이상인데 온도가 내려가지 않음 / 출력 100%(≥98%)가 5분 이상인데 온도가 오르지 않음
opposite_rows = []
for k in lots.LOT_NO.astype(int):
    p = profile[k]
    for z in range(1, 9):
        op = p[(f"OP-Z{z}", "median")].where(quality[k][f"OP-Z{z}"].ge(.9))
        tc = p[(f"TC-Z{z}", "median")].where(quality[k][f"TC-Z{z}"].ge(.9))
        for kind, on, bad in [("출력 0%인데 온도 안 내려감", op.le(2), lambda d: d >= 0),
                              ("출력 100%인데 온도 안 오름", op.ge(98), lambda d: d <= 0)]:
            for a, b in runs(on.fillna(False), 5):
                change = tc.iloc[min(b, len(tc) - 1)] - tc.iloc[a]
                if bad(change):
                    opposite_rows.append({"LOT": k, "GRADE": lot_meta.loc[k, "GRADE"], "ZONE": f"Z{z}", "종류": kind,
                                          "START_MIN": a, "END_MIN": b, "길이[분]": b - a, "온도 변화[℃]": change,
                                          "시작": lot_meta.loc[k, "START"] + pd.Timedelta(minutes=a),
                                          "예정된 변화 중": in_expected(k, a)})
opposite = pd.DataFrame(opposite_rows)

# ②·③ 같은 강종 대비 LOT 전체 수준 차이: 안정 참고 1분 중앙값의 LOT 중앙값 - 같은 강종 다른 LOT 중앙값들의 중앙값
level_rows = []
for grade, sub in lots.groupby("GRADE", sort=False):
    ids = sub.LOT_NO.astype(int).tolist()
    for s in SIGNALS:
        med = {k: profile[k][(s, "median")].where(states[k].eq("안정 참고") & quality[k][s].ge(.9)).median() for k in ids}
        for k in ids:
            peers = [med[o] for o in ids if o != k and pd.notna(med[o])]
            level_rows.append({"LOT": k, "GRADE": grade, "SENSOR": s, "LOT_MEDIAN": med[k],
                               "PEER_MEDIAN": np.median(peers) if peers else np.nan, "N_PEERS": len(peers)})
level_gap = pd.DataFrame(level_rows)
level_gap["GAP"] = level_gap.LOT_MEDIAN - level_gap.PEER_MEDIAN
overage = level_gap.loc[level_gap.SENSOR.isin(["TC-Z7", "TC-Z8"]) & level_gap.GAP.ge(2.5)]       # ② 과시효 Z7·Z8이 같은 강종보다 2.5℃ 이상 높은 LOT
OVERAGE_LOTS = sorted(overage.LOT.unique())

# ③ 세정 입구 온도(CLN-IN) LOT 중앙값을 정렬해 가장 큰 간격에서 두 묶음으로 나눈다.
cln = level_gap.loc[level_gap.SENSOR.eq("CLN-IN")].set_index("LOT").LOT_MEDIAN.sort_values()
cut = cln.diff().idxmax()                                   # 큰 묶음의 첫 LOT
CLN_SPLIT = (cln.loc[:cut].iloc[-2] + cln.loc[cut]) / 2
cln_group = pd.Series(np.where(cln.ge(CLN_SPLIT), f"{CLN_SPLIT:.1f}℃ 이상", f"{CLN_SPLIT:.1f}℃ 미만"), index=cln.index)

# ④ CP 변화: 자체 안정 참고 기준과 가장 크게 벗어난 펄스 (예정된 정지 중의 CP 하강은 제외)
cp_main = cp_candidates.loc[[not in_expected(int(r.LOT), r.PEAK_MIN) for r in cp_candidates.itertuples()]]
CP_TOP = cp_main.loc[cp_main.DELTA.abs().idxmax()]

# ① 대표 사례: 출력 0%가 5분 이상인데 온도가 가장 크게 오른 곳
z0 = opposite.loc[opposite["종류"].str.startswith("출력 0%") & ~opposite["예정된 변화 중"]]
Z3_TOP = z0.loc[z0["온도 변화[℃]"].idxmax()]

# 03에서 자세히 본 LOT과 이유 (그림에서 빨간색으로 표시한다)
SELECTED = {}
for k in OVERAGE_LOTS:
    SELECTED.setdefault(int(k), []).append("② Z7·Z8")
SELECTED.setdefault(int(CP_TOP.LOT), []).append("④ CP")
SELECTED.setdefault(int(Z3_TOP.LOT), []).append("① Z3")
SELECT_COLOR = "#d62728"

MARK = {"①": "#c0392b", "②": "#8e44ad", "③": "#16a085", "④": "#d35400"}
MARK_NAME = {"①": "출력·온도 반대 움직임", "②": "과시효 Z7·Z8 수준 차이", "③": "세정 입구 온도 두 수준", "④": "가장 큰 CP 변화"}
EXPECTED_COLOR = "#8d99a6"

In [ ]:
from matplotlib.patches import Rectangle
GRADE_COLOR=report_viz.GRADE_COLORS.copy()
ZONE_COLOR=report_viz.ZONE_COLORS.copy()
SENSOR_ZONE_COLOR=dict(ZONE_COLOR)
ZONES=[3,4,5,6]
SELECT_COLOR="#c83745"
MARK_NAME["①"]="출력·온도 응답 확인"
PROCESS["TC-Z9"]="Z9"
def sensor_color(s):
    return report_viz.sensor_color(s)
def sensor_scale(s):return 1e10 if s=="CP-Z4M" else 1
def shown_unit(s):return "[단위 미확인]\n×10⁻¹⁰" if s=="CP-Z4M" else unit(s)
def when(k,a,b):
    start=lot_meta.loc[k,"START"]
    return f"{start+pd.Timedelta(minutes=float(a)):%m/%d %H:%M:%S}–{start+pd.Timedelta(minutes=float(b)):%m/%d %H:%M:%S}"
def clock_axis(ax,day=None):
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=6))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
    style(ax,"실제 측정 날짜·시각[월/일 시:분]")
def raw_window(lot,a,b):
    r=lot_meta.loc[lot]
    times=pd.date_range(r.START+pd.Timedelta(seconds=int(a*60)),min(r.END,r.START+pd.Timedelta(seconds=int(b*60))),freq="s")
    return raw[lot].set_index("MEAS_DT").reindex(times)
def remark_text(t):return str(t.REMARK).strip() if pd.notna(t.REMARK) else "작업 내용 미기재"
def parts_for(wo):
    return [f"{p.PART_NM} [{p.PART_CD}] ×{p.QTY:g} ({p.CHG_RSN})" for p in cal_parts.loc[cal_parts.WO_NO.eq(wo)].itertuples()]
def linked_work(k,pad_hours=0):
    r=lot_meta.loc[k]
    return cal_work.loc[cal_work.STRT_DT.lt(r.END+pd.Timedelta(hours=pad_hours))&cal_work.END_DT.gt(r.START-pd.Timedelta(hours=pad_hours))].sort_values("STRT_DT")


maintenance_stage_rows=[]


**그림 저장 순서:** A. 같은 LOT 공정 반응: `01_전체22신호_확인지도.png` → `02_공정전체흐름.png` → `03_같은LOT공정반응_01.png`. B. 같은 강종 LOT 비교: `04_같은강종LOT비교_01.png`부터 별도로 봅니다.

A는 같은 LOT·같은 실제 시각의 온도와 출력이며 **선 하나 = 센서**입니다. B는 서로 다른 날짜의 같은 강종에서 같은 측정 신호를 비교하며 **선 하나 = LOT**입니다. 두 종류를 한 그림에 섞지 않습니다.

![240037과 같은 SPCE LOT 비교, LOT별 고정색](<../../figures/05_all_data_timeseries_jinny_05/02_LOT/SPCE/240037/04_같은강종LOT비교_01.png>)

## 비교 대상과 읽는 법

- [SGCC](#grade-sgcc) · [SPFH590](#grade-spfh590) · [SPCE](#grade-spce) · [SPCEN](#grade-spcen) · [SPHC](#grade-sphc) · [DP590](#grade-dp590) · [SPCD](#grade-spcd)
- **① 전체 신호 지도:** 22개 신호에서 색 띠·구간 경계·번호·대표점을 보고 어디를 확인할지 찾습니다.
- **② 확인 공정 흐름:** 같은 번호의 구간을 관련 공정의 전체 시간축에서 확인합니다.
- **③ 같은 LOT 공정 반응:** 필요한 시간대만 확대하고, 같은 LOT의 온도·출력을 같은 실제 시각으로 맞춥니다. 선 색은 센서입니다.
- **④ 같은 강종 LOT 비교:** 원본 확대가 끝난 뒤 별도 그림으로 봅니다. LOT별 고정색을 쓰고 대상 LOT는 굵게, 대조 LOT는 얇게 표시합니다. 범례에 LOT와 날짜를 적습니다. 수치는 양쪽의 같은 경과 분·안정 참고·유효 기록 조건을 모두 만족한 분에서 계산합니다.
- **Z3 분리:** 가열 초입 Z3 온도·출력은 Z4~Z6와 별도 패널로 보고, Z3 자체의 반복·하강·복귀를 비교합니다. 분리 자체가 정상 판정을 뜻하지 않습니다.
- 빨강은 먼저 볼 변화, 주황은 함께 볼 변화, 회남색은 초기·종료 흐름입니다. 점은 대표 위치, 세로 경계는 확대 범위입니다.
- CP 극값은 1초 원본이며, 첫째·둘째 그림의 다른 대표점은 1분 집계값 위치입니다. 전체 LOT 수준 비교에는 단일 대표점을 강제로 정하지 않습니다.
- 온도축 범위는 패널별입니다. 두 시간대를 나란히 비교할 때 같은 신호는 세로축 범위를 맞춥니다.

전체 흐름은 CP-Z4 1초 원본, 나머지는 1분 중앙값입니다. 유효 기록률 90% 미만인 분은 표시에서 제외되므로 빈 구간 전체가 실제 누락 시간은 아닙니다.
원본 확대에서는 없는 시각을 보간하지 않습니다. 같은 강종 대조는 다른 날짜 LOT의 같은 경과 분이며 같은 부하·설정·소재 위치를 보장하지 않습니다.

**그림 공통 표시 기준:** 같은 열의 음영·점선은 같은 시각입니다. 강종 대조는 다른 날짜 LOT의 같은 경과 분 중 안정 참고 기록만 씁니다.
Z3는 자체 반복과 하강·복귀를 보고, Z4~Z6와 별도 온도축을 씁니다. 온도·출력의 서로 다른 축 높이를 직접 비교하지 않습니다.
LOT 비교에는 공정 반응의 음영·대표점·범위선을 반복하지 않습니다. 대조 LOT마다 실제 공통 유효 분과 그 구간의 중앙값 차이를 적습니다.
신호선의 색은 센서 범례를 따릅니다. 강종 대조에서는 LOT별 고정색과 선 굵기로 대상을 구분합니다. TC는 실선, OP는 점선입니다.
원본 확대는 1초 값, 강종 대조는 1분 값입니다. 그림 설명의 ‘온도폭’은 해당 구간 최댓값−최솟값이며 원본·1분 값을 구분합니다.
이력 비교의 관측 ±72시간은 검색 범위입니다. 실제 작업 교집합·감지 전후 관측과 날짜상 가까운 작업을 구분하며, 부품은 같은 WO로 연결합니다.
수리 전 종료 변화는 마지막 5분−LOT 경과 45~60% 구간, 수리 후 초기 변화는 경과 80~100분−처음 5분의 온도 중앙값 차이입니다.
반복 파형의 위상 비교는 검출 피크 3개 이상·피크 간격 IQR/중앙값 20% 이하인 LOT·센서만 사용합니다.

각 그림 아래에는 이 공통 안내를 반복하지 않고 **관찰 포인트**와 **확인할 차이**를 적습니다.


In [ ]:
# 공통 준비 셀의 raw/profile/quality/비교표를 직접 사용한다.
RV_INK,RV_SUB,RV_GRID='#1d3d55','#61798b','#e7edf2'
RV_ZONE_COLOR=report_viz.ZONE_COLORS.copy()
RV_FIRST = '#c83c52'
RV_TOGETHER = '#ce8f27'
RV_TRANSITION = '#748697'
RV_PEER = '#7862a2'
RV_LEVEL = '#258b88'
RV_COLORS = {'cp': RV_FIRST, 'level': RV_LEVEL, 'thermal': RV_TOGETHER,
             'dry': RV_TOGETHER, 'clean': RV_TOGETHER, 'age': RV_TOGETHER,
             'peer': RV_PEER, 'transition': RV_TRANSITION}



RV_NAMES={'cp':'CP 변화','level':'별도 LOT 수준 비교','thermal':'가열·균열 반응','dry':'건조 변화','clean':'세정 변화','age':'과시효 변화','peer':'별도 LOT 구간 비교','transition':'초기·종료 흐름'}
RV_ORDER=list(RV_NAMES)
def rv_style(ax, xlabel=True):
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['left', 'bottom']].set_color('#bdcad4')
    ax.tick_params(labelsize=11, colors=RV_SUB, pad=4)
    ax.grid(axis='y', color=RV_GRID, lw=0.7)
    ax.set_axisbelow(True)
    ax.yaxis.set_major_locator(MaxNLocator(4))
    if xlabel:
        ax.set_xlabel('LOT 관측 시작 후 경과시간[분]', fontsize=11.5, color=RV_INK, labelpad=7)

def rv_color(s):
    return report_viz.sensor_color(s)

def rv_process(s):
    if s == 'CLN-IN':
        return '세정'
    if s in ['CP-Z4', 'CP-Z4M']:
        return 'CP'
    if s.startswith('TC-OUT') or s == 'TC-Z9':
        return 'Z9·냉각'
    z = int(s.rsplit('Z', 1)[1])
    return '건조' if z < 3 else '가열·균열' if z < 7 else '과시효'

def rv_stamp(k, a, b=None):
    r = lot_meta.loc[k]
    t = r.START + pd.Timedelta(minutes=float(a))
    if b is None:
        return f'{t:%m/%d %H:%M:%S}'
    e = min(r.END, r.START + pd.Timedelta(minutes=float(b)))
    return f'{t:%m/%d %H:%M:%S} ~ {e:%m/%d %H:%M:%S}'

def rv_raw_window(k, a, b):
    r = lot_meta.loc[k]
    times = pd.date_range(r.START + pd.Timedelta(seconds=int(a * 60)), min(r.END, r.START + pd.Timedelta(seconds=int(b * 60))), freq='s')
    frame = raw[k].set_index('MEAS_DT').reindex(times)
    return ((times - r.START).total_seconds() / 60, frame)

def rv_profile(k, s):
    return profile[k][s, 'median'].where(quality[k][s].ge(0.9))

def rv_cp_events(k):
    g = raw[k].set_index('MEAS_DT')
    s = g['CP-Z4'].dropna()
    dev = s - s.median()
    off = dev.abs().gt(0.02)
    groups = off.ne(off.shift()).cumsum()
    out = []
    for _, part in dev[off].groupby(groups[off]):
        t = part.abs().idxmax()
        if abs(part.loc[t]) < 0.05:
            continue
        a = (part.index[0] - lot_meta.loc[k, 'START']).total_seconds() / 60
        b = (part.index[-1] - lot_meta.loc[k, 'START']).total_seconds() / 60 + 1 / 60
        m = (t - lot_meta.loc[k, 'START']).total_seconds() / 60
        exp = in_expected(k, m)
        out.append(dict(k=k, family='transition' if exp else 'cp', a=a, b=b, sensors=['CP-Z4'], text=f'CP {s.loc[t]:.3f} · LOT 중앙값 대비 {part.loc[t]:+.3f}', peak=m, dev=float(part.loc[t])))
    return out

def rv_catalog(k):
    events = rv_cp_events(k)
    frame = opposite.loc[opposite.LOT.eq(k) & ~opposite['예정된 변화 중']]
    for row in frame.to_dict('records'):
        z = int(row['ZONE'][1:])
        events.append(dict(k=k, family='thermal', a=float(row['START_MIN']), b=float(row['END_MIN']), sensors=[f'TC-Z{z}', f'OP-Z{z}'], text=f"{row['종류']} · 온도 변화 {row['온도 변화[℃]']:+.1f}℃"))
    for s in NONCYCLIC + OP:
        y = rv_profile(k, s)
        past = y.shift(1)
        center = past.rolling(30, min_periods=20).median()
        spread = past.rolling(30, min_periods=20).std().clip(lower=2.0 if s.startswith('OP-') else 0.5)
        z = (y - center) / spread
        mask = z.abs().ge(4) & states[k].eq('안정 참고')
        for a, b in runs(mask.fillna(False)):
            family = 'clean' if s == 'CLN-IN' else 'dry' if rv_process(s) == '건조' else 'thermal' if rv_process(s) == '가열·균열' else 'age'
            events.append(dict(k=k, family=family, a=float(a), b=float(b), sensors=[s], text=f'{s} 이전 30분 대비 변화 · 최대 |Z| {z.loc[a:b - 1].abs().max():.1f}'))
    for row in episodes.loc[episodes.LOT.eq(k)].itertuples():
        events.append(dict(k=k, family='peer', a=float(row.START_MIN), b=float(row.END_MIN), sensors=[row.SENSOR], text=f'{row.SENSOR} 대조 중앙값과 최대 차이 {row.MAX_ABS_DELTA_C:.1f}℃ · 대조 2 LOT 이상'))
    levels = level_gap.loc[level_gap.LOT.eq(k) & level_gap.SENSOR.isin(['CLN-IN', 'TC-Z7', 'TC-Z8'])]
    for row in levels.loc[levels.N_PEERS.ge(1) & levels.GAP.abs().ge(2.5)].itertuples():
        events.append(dict(k=k, family='level', a=0.0, b=float(lot_meta.loc[k, 'MINUTES']), sensors=[row.SENSOR], text=f'{row.SENSOR} LOT 중앙값 {row.LOT_MEDIAN:.1f}℃ · 같은 강종 중앙값 대비 {row.GAP:+.1f}℃ · 대조 {row.N_PEERS} LOT'))
    for row in expected.loc[expected.LOT.eq(k)].itertuples():
        events.append(dict(k=k, family='transition', a=float(row.START_MIN), b=float(row.END_MIN), sensors=HEAT + ['CP-Z4'], text=str(row.종류) + ' · 사후 형태와 실제 이력을 구분해 확인'))
    return events

def rv_base_windows(k, family, events):
    if family == 'level':
        sensors = sorted({s for e in events for s in e['sensors'] if s in TC})
        return [dict(a=0.0, b=float(lot_meta.loc[k, 'MINUTES']), sensors=[s], events=[e for e in events if s in e['sensors']]) for s in sensors]
    buckets = defaultdict(list)
    for e in events:
        key = e['sensors'][0] if family == 'peer' else 'all'
        buckets[key].append(e)
    result = []
    for bucket in buckets.values():
        merged = []
        for e in sorted(bucket, key=lambda x: x['a']):
            duration = float(lot_meta.loc[k, 'MINUTES'])
            a = max(0, e['a'] - 5)
            b = e['b'] + 5
            if family == 'thermal' and len(e['sensors']) == 2:
                b = max(b, e['a'] + 45)
            if family == 'dry':
                b = e['b'] + 15
            b = min(duration, b)
            if merged and a <= merged[-1]['b'] + 3:
                merged[-1]['b'] = max(merged[-1]['b'], b)
                merged[-1]['events'].append(e)
                merged[-1]['sensors'] = sorted(set(merged[-1]['sensors']) | set(e['sensors']))
            else:
                merged.append(dict(a=a, b=b, sensors=list(e['sensors']), events=[e]))
        result += merged
    return result

def rv_cycle_note(k):
    parts = []
    for s in ['TC-Z9', 'TC-OUT1', 'TC-OUT2']:
        y = rv_profile(k, s).where(states[k].eq('안정 참고')).rolling(5, center=True, min_periods=3).median()
        peaks, _ = find_peaks(y.to_numpy(), prominence=5, distance=80 if s == 'TC-Z9' else 30)
        if len(peaks) >= 3:
            gap = np.diff(peaks)
            median = np.median(gap)
            consistent = (np.quantile(gap, 0.75) - np.quantile(gap, 0.25)) / median <= 0.2
            note = f'주기 약 {median:g}분' if consistent else '피크 간격 불일치'
        else:
            note = f'검출 피크 {len(peaks)}개 · 주기 비교 제한'
        parts.append(f'{s}: {note}')
    return ' / '.join(parts)

def rv_title_summary(k, events):
    cp = [e for e in events if e['family'] == 'cp']
    if cp:
        return f"CP 기준 초과 {len(cp)}회 · 최대 차이 {max(cp, key=lambda x: abs(x['dev']))['dev']:+.3f}"
    level = [e for e in events if e['family'] == 'level']
    if level:
        return '같은 강종의 ' + '·'.join(dict.fromkeys((rv_process(e['sensors'][0]) for e in level))) + ' 수준 대조'
    return '온도·출력 반응과 관측 조건 확인'

def rv_record_status(k):
    r = lot_meta.loc[k]
    expected = int((r.END - r.START).total_seconds()) + 1
    actual = raw[k]['MEAS_DT'].nunique()
    return f'관측 {r.START:%m/%d %H:%M:%S} ~ {r.END:%m/%d %H:%M:%S} · 1초 기록률 {actual / expected:.1%} · 기록이 없는 시각 {expected - actual:,}초'



def process_axes(ax, k, sensors, a=None, b=None, clock=True, raw_values=False, envelope=True):
    r = lot_meta.loc[k]
    for s in sensors:
        scale = sensor_scale(s)
        if raw_values:
            w = raw_window(k, a, b)
            x = w.index if clock else (w.index - r.START).total_seconds() / 60
            y = w[s] * scale
            ax.plot(x, y, color=sensor_color(s), lw=1.3,label=s)
        else:
            p = profile[k]
            x = r.START + pd.to_timedelta(p.index, unit='min') if clock else p.index
            ax.plot(x, p[s, 'median'] * scale, color=sensor_color(s), lw=1.4,label=s)
            if envelope:
                ax.fill_between(x, p[s, 'min'] * scale, p[s, 'max'] * scale, color=sensor_color(s), alpha=0.1, lw=0)
    ax.set_ylabel(shown_unit(sensors[0]), labelpad=8)
    if clock:
        clock_axis(ax)
    else:
        style(ax)
    ax.yaxis.set_major_locator(MaxNLocator(5))
    ax.tick_params(labelsize=12)


# LOT 전체 → 번호가 붙은 확인 구간 → 다음 원본 확대. 이 노트북 안에서 계산하고 그린다.
from collections import defaultdict
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.text import Text

RV_W = 24
rv_coverage, rv_window_audit, rv_image_manifest, rv_observation_audit = {}, [], [], []

def rv_unit(s):
    return '[%]' if s.startswith('OP-') else '[단위 미확인]\n×10⁻¹⁰' if s=='CP-Z4M' else '[단위 미확인]' if s=='CP-Z4' else '[℃]'

def rv_tags(signals):
    if len(signals)==1:return signals[0]
    prefix=signals[0].split('-')[0]
    if all(s.startswith(prefix+'-Z') for s in signals):
        zones=sorted(int(s.rsplit('Z',1)[1]) for s in signals)
        if zones==list(range(zones[0],zones[-1]+1)):return f'{prefix}-Z{zones[0]}~Z{zones[-1]}'
    return ' · '.join(signals)

def rv_axis(fig,left,top,width,height):
    return fig.add_axes([left/RV_W,1-(top+height)/fig.get_figheight(),width/RV_W,height/fig.get_figheight()])

def rv_text(fig,left,top,text,**kwargs):
    return fig.text(left/RV_W,1-top/fig.get_figheight(),text,**kwargs)



def rv_save(fig,name,k,mode,window=None,show=True):
    fig.set_dpi(report_viz.EXPORT_DPI)
    rv_role_key(fig)
    report_viz.style_signal_lines(fig)
    separate_observation_captions(fig)
    fig.canvas.draw(); renderer=fig.canvas.get_renderer(); labels=list(fig.texts)
    for ax in fig.axes:
        labels+=list(ax.texts)+[ax.xaxis.label,ax.yaxis.label,ax.title,ax._left_title,ax._right_title]
        labels += [ax.xaxis.offsetText, ax.yaxis.offsetText]
        for axis in [ax.xaxis, ax.yaxis]:
            low, high = sorted(axis.get_view_interval())
            for tick in axis.get_major_ticks():
                if low <= tick.get_loc() <= high:
                    labels += [tick.label1, tick.label2]
        legend=ax.get_legend()
        if legend is not None:
            assert legend.get_window_extent(renderer).y0>ax.bbox.y1, '범례가 파형을 가림'
            labels+=legend.get_texts()
    boxes=[]
    for label in labels:
        if not label.get_visible() or not label.get_text():continue
        box=Text.get_window_extent(label,renderer)
        assert box.x0>=0 and box.y0>=0 and box.x1<=fig.bbox.x1 and box.y1<=fig.bbox.y1, label.get_text()
        for other,old in boxes:
            dx=min(box.x1,old.x1)-max(box.x0,old.x0)
            dy=min(box.y1,old.y1)-max(box.y0,old.y0)
            assert dx<2 or dy<2, f'글자 겹침: {label.get_text()} / {other.get_text()}'
        boxes.append((label,box))
    path=OUT_DIR/name
    assert path.resolve().is_relative_to(OUT_DIR.resolve())
    path.parent.mkdir(parents=True,exist_ok=True)
    fig.savefig(path,dpi=report_viz.EXPORT_DPI,facecolor='white')
    rv_image_manifest.append({'LOT':k,'종류':mode,'구간':window['label'] if window else '',
                              '파일':name,'확인': '글자·범례 통과'})
    record_observation_captions(fig, name)
    figure_checks.append({'FILE':name,'TITLE':fig.texts[0].get_text(),'N_AXES':len(fig.axes),
                          'TEXT_OVERLAP':'통과','TEXT_BOUNDS':'통과','LEGEND_BOUNDS':'통과',
                          'CAPTION_GAP_PT':fig._observation_gap_pt})
    fig.set_dpi(72)
    if show:
        display(fig)
    plt.close(fig)

def rv_letter(i):
    text=''
    while True:
        text=chr(97+i%26)+text
        i=i//26-1
        if i<0:return text

def rv_windows(k,family,events):
    if family!='level':return rv_base_windows(k,family,events)
    grouped=defaultdict(list)
    for e in events:
        for s in e['sensors']:
            grouped[rv_process(s)].append(s)
    return [{'a':0.,'b':float(lot_meta.loc[k,'MINUTES']),
             'sensors':sorted(set(sensors)),
             'events':[e for e in events if any(s in e['sensors'] for s in sensors)]}
            for stage,sensors in grouped.items()]

def rv_anchor(w):
    cp=[e for e in w['events'] if 'peak' in e]
    if cp:return max(cp,key=lambda e:abs(e['dev']))['peak']
    responses=[e for e in w['events'] if len(e['sensors'])==2]
    return min(e['a'] for e in responses or w['events'])



def rv_directory(k):
    return f"02_LOT/{lot_meta.loc[k,'GRADE']}/{k}"





def rv_scope(k,events,windows):
    groups=[('세정',['CLN-IN']),('건조',['TC-Z1','TC-Z2','OP-Z1','OP-Z2']),
            ('가열·균열',HEAT+[f'OP-Z{z}' for z in range(3,7)]),('CP',['CP-Z4','CP-Z4M']),
            ('과시효',['TC-Z7','TC-Z8','OP-Z7','OP-Z8']),('Z9·냉각',['TC-Z9','TC-OUT1','TC-OUT2'])]
    rows=[]
    for stage,signals in groups:
        selected=[w['label'] for w in windows if set(w['sensors'])&set(signals)]
        own=level_gap.loc[level_gap.LOT.eq(k)&level_gap.SENSOR.isin(signals)&level_gap.SENSOR.isin(TC)]
        values=' · '.join(f'{r.SENSOR} {r.LOT_MEDIAN:.2f}℃' for r in own.itertuples())
        result='확대 '+', '.join(selected) if selected else '현재 급변·강종 차이 조건의 추가 구간 없음'
        if stage=='Z9·냉각':result=rv_cycle_note(k)
        rows.append({'공정':stage,'확인 신호':' · '.join(signals),'결과':result,'안정 참고 중심':values})
    assert {s for _,ss in groups for s in ss}==set(SIGNALS)
    return pd.DataFrame(rows)

# 이 소스는 05-02 노트북의 공통 그림 셀에 반영한다.
def rv_legend(ax):
    handles, labels = ax.get_legend_handles_labels()
    if labels:
        order = sorted(range(len(labels)), key=lambda i: 0 if labels[i].startswith('대상 ') or labels[i].endswith(' 대상') else 1)
        handles = [handles[i] for i in order]
        labels = [labels[i] for i in order]
        ax.legend(handles, labels, loc='lower right', bbox_to_anchor=(1, 1.025),
                  borderaxespad=0, ncol=min(4, len(labels)), frameon=False,
                  fontsize=10.5, handlelength=1.6, columnspacing=1.15)




def rv_unique_events(events):
    # 한 관찰이 여러 후보 시간창에 들어가도 관찰 내용은 한 번만 보존한다.
    seen, result = set(), []
    for event in events:
        if id(event) not in seen:
            result.append(event)
            seen.add(id(event))
    return result


def rv_families(w):
    return [f for f in RV_ORDER if any(e['family'] == f for e in w['events'])]


def rv_prepare_lot(k):
    events = rv_catalog(k)
    for e in events:
        if e['family'] == 'thermal':
            stage = rv_process(e['sensors'][0])
            if stage == '건조':
                e['family'] = 'dry'
            if stage == '과시효':
                e['family'] = 'age'
    seeds = []
    for family in RV_ORDER:
        for w in rv_windows(k, family, [e for e in events if e['family'] == family]):
            w.update(family=family, seed_count=1)
            seeds.append(w)

    # LOT 전체 수준 대조는 전체 범위로 묶고, 짧은 확대를 흡수하지 않는다.
    levels = [w for w in seeds if w['family'] == 'level']
    windows = []
    for w in sorted((w for w in seeds if w['family'] != 'level'), key=lambda w: (w['a'], w['b'])):
        if windows:
            previous = windows[-1]
            union = max(previous['b'], w['b']) - previous['a']
            limit = max(120., previous['b'] - previous['a'], w['b'] - w['a'])
            if w['a'] <= previous['b'] and union <= limit + 1e-8:
                previous['b'] = max(previous['b'], w['b'])
                previous['events'] = rv_unique_events(previous['events'] + w['events'])
                previous['sensors'] = sorted(set(previous['sensors']) | set(w['sensors']))
                previous['seed_count'] += 1
                continue
        windows.append(dict(w, events=list(w['events']), sensors=list(w['sensors'])))
    if levels:
        windows.append(dict(a=0., b=float(lot_meta.loc[k, 'MINUTES']), family='level',
                            events=rv_unique_events([e for w in levels for e in w['events']]),
                            sensors=sorted({s for w in levels for s in w['sensors']}),
                            seed_count=len(levels)))
    short_names = {'cp': 'CP', 'thermal': '가열·균열', 'dry': '건조', 'clean': '세정',
                   'age': '과시효', 'peer': '강종 대조', 'transition': '초기·종료', 'level': 'LOT 수준 대조'}
    for i, w in enumerate(windows):
        families = rv_families(w)
        w['family'] = families[0]
        w['anchor'] = rv_anchor(w)
        w['label'] = '3-' + rv_letter(i)
        w['title'] = ' · '.join(short_names[f] for f in families)
        w['title'] = rv_window_title(w)
        w['at'] = rv_stamp(k, w['anchor'])
    primary = rv_primary(windows)
    for w in windows:
        w['_first'] = w is primary
    covered = {id(e) for w in windows for e in w['events']}
    assert covered == {id(e) for e in events}, f'{k}: 묶음에서 빠진 관찰'
    rv_observation_audit.extend({'LOT': k, '유형': RV_NAMES[e['family']], '센서': ' · '.join(e['sensors']),
        '시작': rv_stamp(k, e['a']), '끝': rv_stamp(k, e['b']), '결과': e['text']} for e in events)
    return events, windows




def rv_raw_lines(ax, k, w, signals, focus=None, scale=False):
    x, g = rv_raw_window(k, w['a'], w['b'])
    for s in signals:
        strong = focus is None or s in focus
        ax.plot(x, g[s] * sensor_scale(s), color=rv_color(s), lw=1.45 if strong else .85,
                alpha=1 if strong else .65, label=s, zorder=3 if strong else 2)
    rv_marks(ax, w)
    ax.set_xlim(w['a'], w['b'])
    rv_style(ax, False)


def rv_peer_lines(ax, k, w, s):
    grade = lot_meta.loc[k, 'GRADE']
    ks = lots.loc[lots.GRADE.eq(grade), 'LOT_NO'].astype(int)
    styles = ['--', ':', '-.']
    for i, other in enumerate([int(o) for o in ks if o != k] + [k]):
        y = rv_profile(other, s).where(states[other].eq('안정 참고'))
        y = y.loc[(y.index+.5 >= w['a']) & (y.index+.5 <= w['b'])]
        target = other == k
        ax.plot(y.index+.5, y*sensor_scale(s), color=LOT_COLOR[other],
                lw=1.85 if target else 1.15, ls=report_viz.lot_linestyle(other),
                alpha=1 if target else .8, label=f'{other}'+(' 대상' if target else ' 대조'))
    if any(e['family'] == 'level' and s in e['sensors'] for e in w['events']):
        row = level_gap.loc[level_gap.LOT.eq(k) & level_gap.SENSOR.eq(s)].iloc[0]
        reference = row.LOT_MEDIAN-row.GAP
        ax.axhline(reference, color=RV_LEVEL, ls=':', lw=1.6,
                   label=f'대조 중앙값 {reference:.1f}℃', zorder=3)
    ax.set_xlim(w['a'], w['b'])
    rv_style(ax, False)



def rv_compare_direction(k, w):
    stages = list(dict.fromkeys(rv_process(s) for s in w['sensors']))
    directions = ['같은 열의 모든 패널은 같은 시간 범위입니다. ' + '·'.join(stages) + '의 동반 변화와 변화 순서를 비교합니다.']
    if any(f in rv_families(w) for f in ['peer', 'level', 'clean']):
        directions.append('강종 대조는 다른 날짜 LOT의 같은 경과 분이며 안정 참고 기록만 표시합니다.')
    return ' '.join(directions)




def rv_pages(windows):
    pages = []
    for w in windows:
        if pages and len(pages[-1]) == 1:
            previous = pages[-1][0]
            specs = {spec for item in [previous, w] for pair in rv_panel_pairs(item) for spec in pair}
            # 두 시간대를 나란히 놓을 때도 파형 높이를 충분히 확보한다.
            if 'level' not in rv_families(previous) + rv_families(w) and len(specs) <= 8:
                pages[-1].append(w)
                continue
        pages.append([w])
    return pages

def rv_primary(windows):
    cp = [w for w in windows if any(e['family'] == 'cp' for e in w['events'])]
    if cp:
        return max(cp, key=lambda w: max(abs(e.get('dev', 0)) for e in w['events'] if e['family'] == 'cp'))
    changes = [w for w in windows if any(e['family'] in ['thermal', 'dry', 'age', 'clean'] for e in w['events'])]
    return max(changes or windows, key=lambda w: len(w['events'])) if windows else None


def rv_card(fig, x, width, title, headline, body, detail, accent):
    h = fig.get_figheight()
    fill, edge = {RV_FIRST: ('#fff5f7', '#ebcbd2'),
                  RV_TOGETHER: ('#fff8eb', '#ead6ae')}.get(accent, ('#f5f8fb', '#dce6ed'))
    fig.add_artist(FancyBboxPatch((x/RV_W, 1-2.85/h), width/RV_W, 1.68/h,
        transform=fig.transFigure, boxstyle='round,pad=.005', fc=fill, ec=edge, lw=.8))
    for top, value, size, color, bold in [
        (1.36, title, 12, accent if title != '2024년 관측 정보' else RV_SUB, True),
        (1.76, headline, 16.5, accent, True),
        (2.15, body, 11.8, RV_INK, False), (2.52, detail, 11.5, RV_SUB, False)]:
        rv_text(fig, x+.2, top, value, fontsize=size, color=color,
                weight='bold' if bold else 'normal', va='top')



def rv_focus_limits(ax):
    values = []
    for line in ax.lines:
        if line.get_label().startswith('_'):
            continue
        x, y = np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)
        values.extend(y[(x >= ax.get_xlim()[0]) & (x <= ax.get_xlim()[1]) & np.isfinite(y)])
    return values

# 첫 그림(전수 위치 지도) → 둘째 그림(관련 공정 흐름) → 번호별 원본 확대.
RV_WINDOW_ALPHA, RV_EVENT_ALPHA = .10, .22


def rv_scan_specs():
    return [['CLN-IN'], ['TC-Z1', 'TC-Z2'], ['OP-Z1', 'OP-Z2'], ['TC-Z3'], ['OP-Z3'],
            [f'TC-Z{z}' for z in range(4, 7)], [f'OP-Z{z}' for z in range(4, 7)],
            ['CP-Z4'], ['CP-Z4M'], ['TC-Z7', 'TC-Z8'], ['OP-Z7', 'OP-Z8'],
            ['TC-Z9'], ['TC-OUT1', 'TC-OUT2']]


def rv_window_title(w):
    zones = {int(s.rsplit('Z', 1)[1]) for e in w['events'] if e['family'] == 'thermal'
             for s in e['sensors'] if s.startswith(('TC-Z', 'OP-Z'))}
    if not zones:
        return w['title']
    label = 'Z3 온도·출력' if zones == {3} else 'Z3·Z4~Z6 반응' if 3 in zones else 'Z4~Z6 반응'
    return w['title'].replace('가열·균열', label)


def rv_panel(ax, signals, title=None):
    kind = '출력' if signals[0].startswith('OP-') else '온도'
    if signals[0].startswith('CP-'):
        title = title or rv_tags(signals)
    else:
        first = signals[0]
        process = ('가열 초입 Z3' if first in ['TC-Z3', 'OP-Z3'] else
                   '가열·균열 Z4~Z6' if first in ['TC-Z4', 'OP-Z4'] else
                   '냉각' if first.startswith('TC-OUT') else 'Z9' if first == 'TC-Z9' else rv_process(first))
        title = title or f'{process} {kind} · {rv_tags(signals)}'
    ax.set_ylabel(rv_unit(signals[0]), fontsize=12, color=RV_INK, labelpad=9)
    ax.set_title(title, loc='left', fontsize=12.5, color=report_viz.process_title_color(title, RV_INK), pad=9)
    rv_legend(ax)
    ax.tick_params(labelsize=11)
    ax.margins(y=.16)


def rv_relevant_events(w, signals):
    paired = set(signals)
    paired |= {s.replace('TC-', 'OP-') for s in signals} | {s.replace('OP-', 'TC-') for s in signals}
    if paired & {'CP-Z4', 'CP-Z4M', 'TC-Z4', 'OP-Z4'}:
        paired |= {'CP-Z4', 'CP-Z4M'}
    return [e for e in w['events'] if e['family'] == 'transition' or paired.intersection(e['sensors'])]


def rv_keypoint(k, w):
    if '_point' in w:
        return w['_point']
    cp = [e for e in w['events'] if e['family'] == 'cp']
    if cp:
        e = max(cp, key=lambda e: abs(e['dev']))
        p = dict(minute=float(e['peak']), signal='CP-Z4', kind='CP 극값')
    elif all(e['family'] == 'level' for e in w['events']):
        p = None
    else:
        events = [e for e in w['events'] if e['family'] != 'transition'] or w['events']
        tagged = [(e, s) for e in events for s in e['sensors'] if s in TC]
        tagged = tagged or [(e, s) for e in events for s in e['sensors'] if s in OP]
        candidates = []
        for e, s in tagged:
            y = rv_profile(k, s)
            x = y.index + .5
            selected = y.loc[(x >= e['a']) & (x <= min(e['b'], w['b']))].dropna()
            if selected.empty:
                continue
            reference = y.loc[(x >= max(0, e['a'] - 10)) & (x < e['a'])].median()
            if pd.isna(reference):
                reference = selected.median()
            change = (selected - reference).abs()
            i = int(change.idxmax())
            # 같은 단위 안에서 대표 변화점을 고른다. 관찰 검출 기준은 바꾸지 않는다.
            candidates.append((float(change.loc[i]), i + .5, s))
        if candidates:
            _, minute, signal = max(candidates, key=lambda item: item[0])
            p = dict(minute=float(minute), signal=signal, kind='1분 대표점')
        else:
            p = None
    w['_point'] = p
    return p


def rv_plot_point(ax, signals, point):
    if point is None:
        return None
    s = point['signal']
    counterpart = s.replace('TC-', 'OP-') if s.startswith('TC-') else s.replace('OP-', 'TC-')
    target = s if s in signals else counterpart if counterpart in signals else None
    if target is None and s == 'CP-Z4':
        target = next((v for v in signals if v in ['CP-Z4M', 'TC-Z4', 'OP-Z4']), None)
    if target is None:
        return None
    line = next((line for line in ax.lines if line.get_label() == target), None)
    if line is None:
        return None
    x, y = np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)
    valid = np.isfinite(y) & (x >= ax.get_xlim()[0]) & (x <= ax.get_xlim()[1])
    if not valid.any():
        return None
    x, y = x[valid], y[valid]
    i = int(np.abs(x - point['minute']).argmin())
    if abs(x[i] - point['minute']) > .55:
        return None
    return float(x[i]), float(y[i])


def rv_marks(ax, w):
    # 비교 범위는 얇은 범위선으로 표시하고, 파형 배경은 실제 변화에만 쓴다.
    for family in ['peer']:
        events = [e for e in w['events'] if e['family'] == family]
        if events:
            rv_comparison_rail(ax, family, events)
    for transition in [False, True]:
        events = [e for e in w['events'] if e['family'] not in ['peer', 'level']
                  and (e['family'] == 'transition') == transition]
        color = RV_TRANSITION if transition else rv_role_color(w)
        for a, b in merge_ranges([(e['a'], e['b']) for e in events]):
            patch = ax.axvspan(a, b, color=color, alpha=.13 if transition else .22, lw=0, zorder=0)
            patch._review_family = 'transition' if transition else 'change'
        for peak in sorted({e['peak'] for e in events if 'peak' in e}):
            ax.axvline(peak, color=color, ls='--', lw=1.3, alpha=.95)



def rv_attention(ax, k, windows, signals, labels=True):
    own = [(w, rv_relevant_events(w, signals)) for w in windows]
    own = [(w, events) for w, events in own if events]
    positions = []
    lo, hi = ax.get_xlim()
    for w, events in sorted(own, key=lambda item: item[0]['a']):
        color = rv_role_color(w)
        active = [e for e in events if e['family'] not in ['peer', 'level']]
        if active:
            # 확대 범위는 선만 남겨, 긴 대조 범위가 배경을 덮지 않게 한다.
            for boundary in [w['a'], w['b']]:
                ax.axvline(boundary, color=color, lw=.8, alpha=.55, zorder=1)
        rv_marks(ax, dict(w, events=events))
        point = rv_keypoint(k, w)
        located = rv_plot_point(ax, signals, point)
        center = point['minute'] if point else (w['a'] + w['b'])/2
        if point:
            ax.axvline(point['minute'], color=color, lw=1.25, ls='--', alpha=.95)
        if located:
            ax.scatter([located[0]], [located[1]], s=42, color=color,
                       edgecolor='white', linewidth=.9, zorder=6)
        if labels:
            positions.append([w, center, color])
    if positions:
        positions.sort(key=lambda item: item[1])
        xs, gap = [], (hi-lo)*.038
        for _, center, _ in positions:
            xs.append(max(center, lo+(hi-lo)*.025 if not xs else xs[-1]+gap))
        if xs[-1] > hi-(hi-lo)*.025:
            xs[-1] = hi-(hi-lo)*.025
            for i in range(len(xs)-2, -1, -1):
                xs[i] = min(xs[i], xs[i+1]-gap)
        low, high = ax.get_ylim()
        ax.set_ylim(low, high+(high-low)*.3)
        for (w, center, color), x in zip(positions, xs):
            ax.text(x, .90, w['label'], transform=ax.get_xaxis_transform(), ha='center', va='center',
                    color=color, fontsize=11.5, weight='bold',
                    bbox=dict(boxstyle='round,pad=.18', facecolor='white', edgecolor=color, linewidth=1.05), zorder=7)
            if abs(x-center) > (hi-lo)*.005:
                ax.plot([x, center], [.82, .72], transform=ax.get_xaxis_transform(), color=color, lw=.7)
        if signals[0].startswith('OP-'):
            ax.set_ylim(-4, 130)
            ax.set_yticks([0, 25, 50, 75, 100])



def rv_full_signals(k, events, windows):
    r = lot_meta.loc[k]
    specs = rv_scan_specs()
    primary = rv_primary(windows)
    height, first, pitch = 4.25 + 1.62 * len(specs), 2.55, 1.62
    fig = plt.figure(figsize=(RV_W, height), facecolor='white')
    rv_text(fig, .65, .35, f'{k} · {r.GRADE} 전체 신호 지도', fontsize=23, color=RV_INK, weight='bold', va='top')
    rv_text(fig, .65, .97, f'{rv_stamp(k, 0, float(r.MINUTES))} · CP-Z4 1초 / 나머지 1분 중앙값', fontsize=12, color=RV_SUB)
    if primary:
        point = rv_keypoint(k, primary)
        target = f"{primary['label']} · {primary['title']}"
        if point:
            target += f" · {point['signal']} 대표점 {rv_stamp(k, point['minute'])}"
        rv_text(fig, .65, 1.49, '먼저 볼 위치  ' + target, fontsize=14, color=rv_role_color(primary), weight='bold')
    rv_text(fig, .65, 1.95, '선택 범위 ' + rv_stamp(k, primary['a'], primary['b']) + ' · 관련 신호 ' + rv_tags(primary['sensors']), fontsize=12, color=RV_SUB)
    seen, axes = set(), []
    for row, signals in enumerate(specs):
        ax = rv_axis(fig, 1.65, first + row*pitch, 21.4, 1.04)
        if axes:
            ax.sharex(axes[0])
        axes.append(ax)
        for s in signals:
            seen.add(s)
            if s == 'CP-Z4':
                x, g = rv_raw_window(k, 0, float(r.MINUTES))
                ax.plot(x, g[s], color=rv_color(s), lw=.7, label=s)
            else:
                y = rv_profile(k, s)
                ax.plot(y.index+.5, y*sensor_scale(s), color=rv_color(s), lw=1.15, label=s)
        rv_style(ax, False)
        rv_panel(ax, signals)
        ax.set_xlim(0, float(r.MINUTES))
        rv_attention(ax, k, windows, signals)
        ax.tick_params(labelbottom=row == len(specs)-1)
        if row == len(specs)-1:
            ax.set_xlabel('LOT 관측 시작 후 경과시간[분]', color=RV_INK, labelpad=7)
    assert seen == set(SIGNALS)
    rv_coverage[k] = seen
    bottom = first + pitch*len(specs)
    add_observation_caption(fig, .65, bottom+.12, 22.0, *window_observation(k, primary), section=primary['label'])
    rv_save(fig, rv_directory(k)+'/01_전체22신호_확인지도.png', k, '전체 22개 신호·확인 위치')


def rv_panel_pairs(w):
    stages = {rv_process(s) for s in w['sensors']}
    zones = {int(s.rsplit('Z', 1)[1]) for s in w['sensors'] if s.startswith(('TC-Z', 'OP-Z'))}
    cp = any(f in rv_families(w) for f in ['cp', 'transition'])
    peers = sorted({s for e in w['events'] if e['family'] in ['peer', 'level', 'clean']
                    for s in e['sensors'] if s in TC}, key=SIGNALS.index)
    groups = []
    if cp or 'CP' in stages:
        groups.append([['CP-Z4'], ['CP-Z4M']])
    if 3 in zones:
        groups.append([['TC-Z3'], ['OP-Z3']])
    if cp or any(z in zones for z in [3, 4, 5, 6]):
        groups.append([[f'TC-Z{z}' for z in range(4, 7)], [f'OP-Z{z}' for z in range(4, 7)]])
    if '건조' in stages:
        groups.append([['TC-Z1', 'TC-Z2'], ['OP-Z1', 'OP-Z2']])
    if '과시효' in stages:
        groups.append([['TC-Z7', 'TC-Z8'], ['OP-Z7', 'OP-Z8']])
    if '세정' in stages:
        groups.append([['CLN-IN']])
    result = []
    for group in groups:
        specs = [('raw', tuple(tags)) for tags in group]
        # LOT comparisons are rendered only in the separate fourth view.
        result.append(specs)
    return result


def render_lot_review(k):
    from html import escape
    from IPython.display import HTML
    events, windows = rv_prepare_lot(k)
    r = lot_meta.loc[k]
    own_windows = rv_raw_windows(windows)
    own_events = [e for e in events if e['family'] not in ['peer', 'level']]
    peer_windows = rv_peer_windows(k, windows)
    pages = rv_pages(own_windows)
    assert own_windows, f'{k}: 공정 확인 구간 없음'
    note(f'### A. 같은 LOT의 공정 반응 · {k} · {r.GRADE}\n'
         '① 전체 신호 위치 → ② 공정 전체 흐름 → ③ 선택 구간 원본. **선 하나는 센서이며, 같은 실제 시각의 온도·출력을 봅니다.**')
    rv_full_signals(k, own_events, own_windows)
    table(pd.DataFrame([{'공정 확인 구간': w['label'], '같은 LOT에서 볼 신호': w['title'],
                        '실제 날짜·시각': rv_stamp(k, w['a'], w['b']), '공정 확대 그림': f'03-{i:02d}'}
                       for i, batch in enumerate(pages, 1) for w in batch]))
    rv_overview(k, own_events, own_windows)
    for i, batch in enumerate(pages, 1):
        note(f'#### ③ 같은 LOT 공정 반응 {i:02d} · ' + ' / '.join(w['label'] for w in batch))
        rv_detail(k, batch, i)
        for w in batch:
            values = list(dict.fromkeys(e['text'] for e in sorted(w['events'], key=lambda e: (e['family'] != 'cp', e['a']))))
            note(f'**{w["label"]} · {w["title"]}** — ' + '; '.join(values[:2]) + '.')
            if len(values) > 2:
                display(HTML('<details><summary>이 구간의 전체 관찰 보기</summary><ul>'
                             + ''.join('<li>' + escape(value) + '</li>' for value in values) + '</ul></details>'))
    note(f'### B. 같은 강종의 다른 LOT 비교 · {r.GRADE}\n'
         '**선 하나는 LOT이며, 같은 측정 신호끼리만 비교합니다. 서로 다른 실제 날짜를 같은 경과 분으로 맞춥니다.**')
    if peer_windows:
        for i, w in enumerate(peer_windows, 1):
            rv_peer_review(k, w, i)
        data = pd.DataFrame(rv_peer_comparison_audit)
        data = data.loc[data['대상 LOT'].eq(k)]
        if len(data):
            display(HTML('<details><summary>공통 유효 분·실제 날짜와 수치 보기</summary>'
                         + data.to_html(index=False, escape=True) + '</details>'))
    else:
        peers = lots.loc[lots.GRADE.eq(r.GRADE) & lots.LOT_NO.ne(k), 'LOT_NO'].astype(int).tolist()
        note('같은 강종의 다른 LOT이 없어 LOT 간 비교는 하지 않습니다.' if not peers else
             '현재 선정 기준으로 별도 확인할 LOT 차이 구간은 없습니다. 강종 전체 비교는 아래의 별도 절에서 확인합니다.')
    display(HTML('<details><summary>공정별 확인 결과 보기</summary>'
                 + rv_scope(k, events, windows).to_html(index=False, escape=True) + '</details>'))


def rv_overview(k, events, windows):
    r = lot_meta.loc[k]
    duration = float(r.MINUTES)
    primary = rv_primary(windows)
    accent = rv_role_color(primary) if primary else RV_INK
    cp = [e for e in events if e['family'] == 'cp']
    stages = {rv_process(s) for e in events for s in e['sensors']}
    if 'CP' in stages:
        stages.add('가열·균열')
    specs = [(['CP-Z4'], 'CP'), (['TC-Z3'], '가열·균열'), (['OP-Z3'], '가열·균열'),
             (HEAT[1:], '가열·균열'), ([f'OP-Z{z}' for z in range(4, 7)], '가열·균열'),
             (['TC-Z1', 'TC-Z2'], '건조'), (['OP-Z1', 'OP-Z2'], '건조'),
             (['TC-Z7', 'TC-Z8'], '과시효'), (['OP-Z7', 'OP-Z8'], '과시효'), (['CLN-IN'], '세정')]
    tracks = [signals for signals, stage in specs if stage in stages]
    if not tracks:
        tracks = [['TC-Z3'], ['OP-Z3'], HEAT[1:], [f'OP-Z{z}' for z in range(4, 7)]]
    families = [f for f in RV_ORDER if any(e['family'] == f for e in events)]
    first_top, pitch, plot_height = 3.75, 1.65, 1.10
    map_title = first_top + pitch * len(tracks) + .55
    map_top, map_height = map_title + .55, max(1.35, .48 * len(families))
    height = map_top + map_height + 2.0
    fig = plt.figure(figsize=(RV_W, height), facecolor='white')
    rv_text(fig, .65, .35, f'{k} · {r.GRADE}', fontsize=25, color=accent, weight='bold', va='top')
    rv_text(fig, .65, .91, f'{rv_stamp(k, 0, duration)} · 선택 {len(windows)}개 구간의 관련 공정', fontsize=12, color=RV_SUB)
    headline, body, detail = primary_change_card(k, events, windows)
    rv_card(fig, .65, 7.25, '먼저 볼 변화', headline, body, detail, RV_FIRST)
    raw_groups = sorted({rv_process(s) for w in windows for s in w['sensors']})
    page_count = len(rv_pages(windows))
    rv_card(fig, 8.4, 7.2, '함께 볼 변화', f'{len(windows)}개 구간 · 확대 {page_count}장',
            ' · '.join(raw_groups), 'Z3 온도·출력은 Z4~Z6와 별도 배치', RV_TOGETHER)
    total = int((r.END - r.START).total_seconds()) + 1
    actual = raw[k].MEAS_DT.nunique()
    rv_card(fig, 16.1, 7.25, f'{r.START:%Y}년 관측 정보',
            f'{r.START:%m/%d %H:%M} → {r.END:%m/%d %H:%M}',
            f'관측 {duration:.1f}분 · 주로 1초 간격',
            f'시각 기록률 {actual/total:.1%} · 공백 {total-actual:,}초', RV_INK)
    rv_text(fig, .65, 3.25, '② 확인할 공정의 전체 흐름 · Z3와 Z4~Z6 분리', fontsize=16, color=RV_INK, weight='bold')
    rv_text(fig, 23.05, 3.25, 'CP: 1초 원본 · 온도·출력: 1분 중앙값', ha='right', fontsize=11.3, color=RV_SUB)
    axes = []
    for row, signals in enumerate(tracks):
        ax = rv_axis(fig, 1.65, first_top + row * pitch, 21.4, plot_height)
        if axes:
            ax.sharex(axes[0])
        axes.append(ax)
        for s in signals:
            if s == 'CP-Z4':
                x, g = rv_raw_window(k, 0, duration)
                ax.plot(x, g[s], color=rv_color(s), lw=.65, label=s)
            else:
                y = rv_profile(k, s)
                ax.plot(y.index + .5, y * sensor_scale(s), color=rv_color(s), lw=1.1, label=s)
        rv_style(ax, False)
        rv_panel(ax, signals)
        ax.set_xlim(0, duration)
        rv_attention(ax, k, windows, signals)
        main_point = rv_keypoint(k, primary) if primary else None
        if main_point and not rv_relevant_events(primary, signals):
            ax.axvline(main_point['minute'], color=rv_role_color(primary), ls='--', lw=1.2, alpha=.9)
        ax.tick_params(labelbottom=row == len(tracks) - 1)
        if row == len(tracks) - 1:
            ax.set_xlabel('LOT 관측 시작 후 경과시간[분]', color=RV_INK, labelpad=7)
    rv_text(fig, .65, map_title, '구간 지도 · 첫 그림과 같은 번호를 따라 확대', fontsize=16, color=RV_INK, weight='bold')
    nav = rv_axis(fig, 1.65, map_top, 21.4, map_height)
    nav.sharex(axes[0])
    nav.set_xlim(0, duration)
    nav.set_ylim(len(families) - .1, -.15)
    for row, family in enumerate(families):
        color = RV_COLORS[family] if family in ['peer', 'level', 'transition'] else RV_INK
        matching = [w for w in windows if family in rv_families(w)]
        nav.plot([0, duration], [row + .65] * 2, color='#ecf1f5', lw=3 if family in ['peer', 'level'] else 10, solid_capstyle='butt')
        nav.text(0, row + .05, RV_NAMES[family], color=color, fontsize=10.8, weight='bold', va='center',
                 transform=nav.get_yaxis_transform())
        ordered = sorted(matching, key=lambda w: (w['a'] + w['b']) / 2)
        positions, gap = [], duration * .03
        for w in ordered:
            x = (w['a'] + w['b']) / 2
            positions.append(max(x, duration * .02 if not positions else positions[-1] + gap))
        if positions and positions[-1] > duration * .98:
            positions[-1] = duration * .98
            for j in range(len(positions) - 2, -1, -1):
                positions[j] = min(positions[j], positions[j + 1] - gap)
        for w, x in zip(ordered, positions):
            rv_nav_interval(nav, w, family, row)
            entry_color = RV_COLORS[family] if family in ['peer', 'level', 'transition'] else rv_role_color(w)
            nav.text(x, row + .39, w['label'], ha='center', va='center', color=entry_color, fontsize=9.8, weight='bold')
    rv_style(nav)
    nav.set_yticks([])
    nav.grid(False)
    add_observation_caption(fig, 1.65, map_top + map_height + .82, 21.4, *window_observation(k, primary), section=primary['label'])
    name = rv_directory(k) + '/02_공정전체흐름.png'
    rv_save(fig, name, k, '확인 공정 전체 흐름·구간 지도')
    return name


def rv_detail(k, batch, page=1):
    r = lot_meta.loc[k]
    paired = len(batch) == 2
    specs = list(dict.fromkeys(spec for w in batch for group in rv_panel_pairs(w) for spec in group))
    start, pitch, plot_height = 2.45, 2.05, 1.42
    height = start + pitch * len(specs) + 2.1
    fig = plt.figure(figsize=(RV_W, height), facecolor='white')
    primary = rv_primary(batch)
    accent = rv_role_color(primary)
    rv_text(fig, .65, .35, f'{k} · {r.GRADE} 공정 반응 {page:02d}', fontsize=23, color=accent, weight='bold', va='top')
    text = '각 열은 한 시간대 · 같은 신호의 세로축 범위 동일' if paired else '관련 신호를 같은 시간축으로 위아래 비교'
    rv_text(fig, .65, .94, f'비교 기준: {k} 한 LOT의 같은 실제 시각 · 선 하나 = 센서 · 다른 LOT 비교는 ④에서 확인', fontsize=12, color=RV_SUB)
    for col, w in enumerate(batch):
        left = 1.65 + col * 11.25 if paired else 1.65
        rv_text(fig, left, 1.43, f"{w['label']} · {w['title']}", fontsize=14, color=rv_role_color(w), weight='bold')
        rv_text(fig, left, 1.83, rv_stamp(k, w['a'], w['b']), fontsize=12, color=RV_SUB)
    axes_by_window, all_axes = {}, []
    for row, (kind, tags) in enumerate(specs):
        row_axes = []
        for col, w in enumerate(batch):
            ax = rv_axis(fig, 1.65 + col * 11.25, start + row * pitch,
                         10.15 if paired else 21.4, plot_height)
            if id(w) in axes_by_window:
                ax.sharex(axes_by_window[id(w)])
            else:
                axes_by_window[id(w)] = ax
            signals = list(tags)
            if kind == 'peer':
                rv_peer_lines(ax, k, w, signals[0])
                rv_marks(ax, w)
                title = rv_peer_title(k, w, signals[0])
            else:
                focus = {s for e in w['events'] for s in e['sensors']}
                focus |= {s.replace('TC-', 'OP-') for s in list(focus)} | {s.replace('OP-', 'TC-') for s in list(focus)}
                rv_raw_lines(ax, k, w, signals, focus=focus if any(s in focus for s in signals) else None)
                title = None
            rv_panel(ax, signals, title)
            if len(signals) == 1 and kind == 'raw':
                # 단일 측정 태그는 소제목에 이미 있으므로 중복 범례를 덜어낸다.
                ax.get_legend().remove()
            values = rv_focus_limits(ax)
            if values:
                low, high = min(values), max(values)
                pad = max((high - low) * .16, max(abs(low), abs(high), 1.) * 1e-5)
                ax.set_ylim(low - pad, high + pad)
            ax.tick_params(labelbottom=row == len(specs) - 1)
            if row == len(specs) - 1:
                ax.set_xlabel('LOT 관측 시작 후 경과시간[분]', color=RV_INK, labelpad=7)
            row_axes.append(ax)
            all_axes.append((ax, w))
        if paired:
            low, high = min(ax.get_ylim()[0] for ax in row_axes), max(ax.get_ylim()[1] for ax in row_axes)
            for ax in row_axes:
                ax.set_ylim(low, high)
            assert np.allclose(row_axes[0].get_ylim(), row_axes[1].get_ylim())
        if kind == 'raw' and tags == ('CP-Z4',):
            for ax, w in zip(row_axes, batch):
                marked = [e for e in w['events'] if e['family'] == 'cp']
                if not marked:
                    continue
                e = max(marked, key=lambda e: abs(e['dev']))
                t = r.START + pd.Timedelta(minutes=e['peak'])
                value = raw[k].set_index('MEAS_DT').loc[t.round('s'), 'CP-Z4']
                ax.scatter([e['peak']], [value], s=24, color=rv_role_color(w), edgecolor='white', linewidth=.6, zorder=5)
                ax.text(.99, 1.07, f"선택점 {t:%H:%M:%S} · {value:.3f} · 중앙값 대비 {e['dev']:+.3f}",
                    transform=ax.transAxes, ha='right', va='bottom',
                    fontsize=10.5, color=rv_role_color(w))
    for row_ax, w in all_axes:
        point = rv_keypoint(k, w)
        if point:
            row_ax.axvline(point['minute'], color=rv_role_color(w), ls='--', lw=1.1, alpha=.95)
            located = rv_plot_point(row_ax, [line.get_label() for line in row_ax.lines
                                           if not line.get_label().startswith('_')], point)
            if located:
                row_ax.scatter([located[0]], [located[1]], s=30, color=rv_role_color(w),
                               edgecolor='white', linewidth=.7, zorder=6)
    for ax, w in all_axes:
        assert np.allclose(ax.get_xlim(), [w['a'], w['b']]), '관련 패널 시간축 불일치'
    footer = start + pitch * len(specs) + .05
    for col, w in enumerate(batch):
        add_observation_caption(fig, 1.65 + col*11.25, footer, 10.15 if paired else 21.4,
                                *window_observation(k, w, raw_values=True), section=w['label'])
    name = rv_directory(k) + f'/03_같은LOT공정반응_{page:02d}.png'
    rv_save(fig, name, k, '관련 신호 세로 확대', {'label': ' · '.join(w['label'] for w in batch)})
    for w in batch:
        rv_window_audit.append({'LOT': k, '번호': w['label'], '유형': w['title'], '시각': w['at'],
            '시작[분]': w['a'], '끝[분]': w['b'], '센서': ' · '.join(w['sensors']),
            '관찰 수': len(w['events']), '기존 확대 수': w['seed_count'], '파일': name})


import re
import unicodedata

figure_caption_audit = []


def observation_series(k, s, a, b, raw_values=False):
    if raw_values:
        g = raw[k]
        return g.loc[g.ELAPSED.ge(a) & g.ELAPSED.le(b), s].dropna()
    y = rv_profile(k, s)
    return y.loc[(y.index + .5 >= a) & (y.index + .5 <= b)].dropna()


def observation_span(k, s, a, b, raw_values=False):
    y = observation_series(k, s, a, b, raw_values)
    return float(y.max() - y.min()) if len(y) else np.nan


def primary_change_card(k, events, windows):
    primary = rv_primary(windows)
    a, b = primary['a'], primary['b']
    cp = [e for e in events if e['family'] == 'cp']
    if cp:
        biggest = max(cp, key=lambda e: abs(e['dev']))
        return (f"CP 변화 {len(cp)}회 · 최대 {biggest['dev']:+.3f}",
                rv_stamp(k, biggest['peak']) + ' · ' + primary['label'] + '에서 확대',
                '자체 중앙값과의 차이 · 고장 판정 기준 아님')
    if rv_families(primary) == ['level']:
        d = level_gap.loc[level_gap.LOT.eq(k) & level_gap.SENSOR.isin(primary['sensors'])].dropna(subset=['GAP'])
        row = d.loc[d.GAP.abs().idxmax()]
        return (f'{row.SENSOR} 강종 차이 {row.GAP:+.1f}℃',
                f"{primary['label']} · 안정 참고 중앙값 {row.LOT_MEDIAN:.1f}℃",
                f'같은 강종 {int(row.N_PEERS)} LOT의 중앙값과 대조')
    point = rv_keypoint(k, primary)
    signal = point['signal'] if point else next(s for s in primary['sensors'] if s in TC)
    if signal in OP:
        signal = signal.replace('OP-', 'TC-')
    span = observation_span(k, signal, a, b)
    related = {signal, signal.replace('TC-', 'OP-')}
    matching = [w for w in windows if any(related.intersection(e['sensors']) and e['family'] not in ['transition', 'level'] for e in w['events'])]
    count = max(1, len(matching))
    label = signal.removeprefix('TC-') if signal != 'CLN-IN' else '세정'
    when = rv_stamp(k, point['minute']) if point else rv_stamp(k, a)
    if signal == 'TC-Z3':
        others = max(observation_span(k, f'TC-Z{z}', a, b) for z in [4, 5, 6])
        detail = f'같은 구간 Z4~Z6 각 존 온도폭 {others:.1f}℃ 이하'
    elif signal.startswith('TC-Z') and signal.replace('TC-', 'OP-') in OP:
        op_span = observation_span(k, signal.replace('TC-', 'OP-'), a, b)
        detail = f'같은 구간 출력 변동폭 {op_span:.1f}%p · 온도·출력 동반 대조'
    else:
        detail = '선택 구간의 1분 온도폭 · 같은 강종의 중심 수준 대조'
    return (f'{label} {count}구간 · 선택 온도폭 {span:.1f}℃',
            when + ' · ' + primary['label'] + '에서 확대', detail)


def window_observation(k, w, raw_values=False):
    families = rv_families(w)
    a, b = w['a'], w['b']
    prefix = w['label'] + ' · '
    cp = [e for e in w['events'] if e['family'] == 'cp']
    if cp:
        e = max(cp, key=lambda e: abs(e['dev']))
        t = lot_meta.loc[k, 'START'] + pd.Timedelta(minutes=e['peak'])
        value = raw[k].set_index('MEAS_DT').loc[t.round('s'), 'CP-Z4']
        return (prefix + f"CP-Z4 {value:.3f}, LOT 중앙값 대비 {e['dev']:+.3f} ({t:%H:%M:%S}).",
                'CP 변화에 Z4 온도·출력도 동반되는지, CP만 변한 뒤 돌아오는지 비교합니다.')
    if families == ['level']:
        d = level_gap.loc[level_gap.LOT.eq(k) & level_gap.SENSOR.isin(w['sensors']) & level_gap.N_PEERS.ge(1)].dropna(subset=['GAP'])
        if len(d):
            row = d.loc[d.GAP.abs().idxmax()]
            check = ('Z7·Z8의 수준 차이가 대응 출력에도 나타나는지 같은 강종의 개별 LOT과 비교합니다.'
                     if row.SENSOR in ['TC-Z7', 'TC-Z8'] else
                     '세정 입구의 중심 수준 차이와 LOT 안의 반복 변동을 나눠 비교합니다.')
            return (prefix + f'{row.SENSOR} 안정 참고 중앙값 {row.LOT_MEDIAN:.1f}℃, 같은 강종 대조보다 {row.GAP:+.1f}℃.', check)
    if families == ['transition']:
        g = raw[k]
        first = g.loc[g.ELAPSED.ge(a) & g.ELAPSED.lt(min(b, a + 5)), HEAT].median()
        last = g.loc[g.ELAPSED.ge(max(a, b - 5)) & g.ELAPSED.le(b), HEAT].median()
        delta = last - first
        direction = '상승' if delta.median() >= 0 else '하강'
        return (prefix + f'창 처음·끝 5분 중앙값 기준, Z3~Z6 온도 변화 {delta.min():+.1f}~{delta.max():+.1f}℃.',
                f'여러 존의 온도·출력 {direction}이 함께 나타나는지 보고, Z3만의 반복 반응과 구분합니다.')
    if families == ['peer']:
        choices = []
        for s in w['sensors']:
            if (k, s) not in comparison:
                continue
            d = comparison[k, s]
            d = d.loc[(d.index + .5 >= a) & (d.index + .5 <= b)].dropna(subset=['DELTA'])
            if len(d):
                choices.append((float(d.DELTA.abs().max()), s, len(d)))
        if choices:
            gap, s, count = max(choices)
            check = ('Z3 반복의 봉우리·골 시점 차이와 중심 온도 차이를 나눠 비교합니다.' if s == 'TC-Z3' else
                     '같은 경과 구간에서 대상·대조 LOT의 중심 수준과 변화 폭을 비교합니다.')
            return (prefix + f'{s}의 같은 강종 대조 중앙값과 최대 차이 {gap:.1f}℃, 비교 가능한 {count}분.', check)
    zones = sorted({int(s.rsplit('Z', 1)[1]) for s in w['sensors'] if s.startswith(('TC-Z', 'OP-Z'))})
    if 3 in zones:
        span = observation_span(k, 'TC-Z3', a, b, raw_values)
        others = [observation_span(k, f'TC-Z{z}', a, b, raw_values) for z in [4, 5, 6]]
        largest = float(np.nanmax(others))
        source = '원본' if raw_values else '1분'
        check = ('Z3의 다른 반복 구간과 하강 폭·복귀 시간을 비교하고, 같은 강종의 반복 시점 차이도 확인합니다.'
                 if 'peer' in families else
                 'Z3의 다른 반복 구간과 하강 폭·복귀 시간, 출력 상·하한 부근의 지속시간을 비교합니다.')
        return (prefix + f'{source} 온도폭: Z3 {span:.1f}℃, 같은 구간 Z4~Z6 각 존의 최대 폭 {largest:.1f}℃.', check)
    if zones:
        z = zones[0]
        tc = observation_series(k, f'TC-Z{z}', a, b, raw_values)
        op = observation_series(k, f'OP-Z{z}', a, b, raw_values)
        if len(tc) and len(op):
            return (prefix + f'Z{z} 온도 {tc.min():.1f}~{tc.max():.1f}℃, 출력 {op.min():.1f}~{op.max():.1f}%.',
                    f'Z{z} 온도·출력의 변화 순서와 이웃 존의 동반 변화를 앞뒤 구간과 비교합니다.')
    y = observation_series(k, 'CLN-IN', a, b, raw_values)
    if len(y):
        return (prefix + f'CLN-IN {y.min():.1f}~{y.max():.1f}℃, 해당 구간의 온도폭 {y.max()-y.min():.1f}℃.',
                '같은 LOT의 앞뒤 반복과 같은 강종의 다른 LOT에서 중심 수준·변화 폭을 비교합니다.')
    raise AssertionError(f'관찰 설명을 만들 자료 없음: {k} {w["label"]}')


def caption_wrap(text, width, size=11.5):
    capacity = int(width * 72 / size * 1.8)
    result, line, length = [], '', 0
    for word in text.split():
        count = sum(2 if unicodedata.east_asian_width(c) in ['W', 'F'] else 1 for c in word)
        if line and length + count + 1 > capacity:
            result.append(line); line, length = word, count
        else:
            line += (' ' if line else '') + word
            length += count + 1
    if line:
        result.append(line)
    return '\n'.join(result)


def add_observation_caption(fig, left, top, width, observation, check, section='', color=RV_INK):
    observed = caption_wrap('관찰 포인트 · ' + observation, width)
    compare = caption_wrap('확인할 차이 · ' + check, width)
    observed_artist = rv_text(fig, left, top, observed, fontsize=11.5, color=color, va='top', linespacing=1.4)
    next_top = top + .31 * (observed.count('\n') + 1) + .09
    compare_artist = rv_text(fig, left, next_top, compare, fontsize=11.5, color=RV_SUB, va='top', linespacing=1.4)
    if not hasattr(fig, '_observation_captions'):
        fig._observation_captions = []
    fig._observation_captions.append({'구간': section, '관찰 포인트': observation, '확인할 차이': check})
    if not hasattr(fig, '_observation_caption_artists'):
        fig._observation_caption_artists = []
    fig._observation_caption_artists.append((observed_artist, compare_artist))


def separate_observation_captions(fig):
    """그래프 크기를 유지하고, 축 설명 아래에 충분한 간격을 둔 하단 설명을 배치한다."""
    pairs = fig._observation_caption_artists
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    old_height, dpi = fig.get_figheight(), fig.dpi
    positions = {ax: ax.get_position().frozen() for ax in fig.axes}
    physical_sizes = [(p.width * fig.get_figwidth(), p.height * old_height) for p in positions.values()]
    limits = [(ax.get_xlim(), ax.get_ylim()) for ax in fig.axes]
    visible = [ax for ax in fig.axes if ax.axison]
    graph_bottom = min(ax.get_tightbbox(renderer).y0 / dpi for ax in visible)
    observed_height = max(Text.get_window_extent(a, renderer).height / dpi for a, _ in pairs)
    compare_height = max(Text.get_window_extent(a, renderer).height / dpi for _, a in pairs)
    chart_gap, paragraph_gap, bottom_pad = .75, .20, .60
    block_height = observed_height + paragraph_gap + compare_height
    # 남아 있는 아래 여백을 먼저 쓰고, 필요할 때만 캔버스를 아래로 늘린다.
    caption_top = max(old_height - graph_bottom + chart_gap, old_height - bottom_pad - block_height)
    new_height = max(old_height, caption_top + block_height + bottom_pad)
    extra = new_height - old_height
    text_positions = {text: text.get_position() for text in fig.texts}
    legend_positions = {legend: legend.get_bbox_to_anchor().transformed(fig.transFigure.inverted())
                        for legend in fig.legends}
    fig.set_size_inches(fig.get_figwidth(), new_height)
    for ax, p in positions.items():
        ax.set_position([p.x0, (p.y0 * old_height + extra) / new_height,
                         p.width, p.height * old_height / new_height])
    for text, (x, y) in text_positions.items():
        text.set_position((x, (y * old_height + extra) / new_height))
    for legend, p in legend_positions.items():
        legend.set_bbox_to_anchor((p.x0, (p.y0 * old_height + extra) / new_height,
                                  p.width, p.height * old_height / new_height), transform=fig.transFigure)
    for observed, compare in pairs:
        observed.set_y(1 - caption_top / new_height)
        compare.set_y(1 - (caption_top + observed_height + paragraph_gap) / new_height)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    gap = (min(ax.get_tightbbox(renderer).y0 for ax in visible) -
           max(Text.get_window_extent(a, renderer).y1 for a, _ in pairs)) / dpi
    assert gap >= chart_gap - .01, f'그래프와 설명 간격 부족: {gap:.3f}in'
    assert limits == [(ax.get_xlim(), ax.get_ylim()) for ax in fig.axes]
    assert np.allclose(physical_sizes, [(ax.get_position().width * fig.get_figwidth(),
                                        ax.get_position().height * new_height) for ax in fig.axes])
    fig._observation_gap_pt = gap * 72
    area = sum(w * h for w, h in physical_sizes)
    return {'CAPTIONS': len(pairs) * 2, 'AREA_BEFORE': area, 'AREA_AFTER': area,
            'GAIN_PCT': 0., 'LIMITS_PRESERVED': True, 'CAPTION_GAP_PT': gap * 72}


def record_observation_captions(fig, name):
    notes = getattr(fig, '_observation_captions', [])
    assert notes, f'관찰 포인트 없는 그림: {name}'
    for row in notes:
        assert row['관찰 포인트'] and row['확인할 차이']
        figure_caption_audit.append({'파일': name, **row})


def grade_observation(grade, page):
    """강종 비교 그림 아래: 이 공정에서 다른 LOT과 차이 종류, 그래서 알 수 있는 것."""
    sensors = [s for pair in PAGE_SIGNALS[page] for s in pair]
    d = grade_differences(grade)
    d = d.loc[d['신호'].isin(sensors)]
    if not len(d):
        return (f'이 공정에서는 {grade} LOT 사이에 기준(온도 2.5℃ · 출력 5%p · CP 0.05) 이상 차이가 없습니다.',
                'LOT마다 출렁이는 시각만 다르고 수준은 같습니다. 이 공정 신호는 강종 기준선 하나로 비교할 수 있습니다.')
    observation = ' / '.join(f"{r['다른 LOT']} {r['신호']} {r['차이']}({r['차이 종류']})" for _, r in d.iterrows())
    return observation, d['알 수 있는 것'].iloc[0]


def phase_observation(grade):
    records = [row for row in phase_rows if row['GRADE'] == grade and pd.notna(row['ELAPSED_MAE_C'])]
    if records:
        row = max(records, key=lambda row: abs(row['ELAPSED_MAE_C']-row['PHASE_MAE_C']))
        return (f"{row['SENSOR']} {row['LOT_A']}·{row['LOT_B']} 평균 절대 차이: 피크 정렬 전 {row['ELAPSED_MAE_C']:.2f}℃ → 후 {row['PHASE_MAE_C']:.2f}℃.",
                '정렬 뒤에도 남는 중심·진폭 차이와, 반복 시점이 달라 생긴 차이를 나눠 봅니다.')
    d = cycle_table.loc[cycle_table.GRADE.eq(grade)]
    eligible = int(d.PHASE_ELIGIBLE.sum())
    return (f'{grade}의 주기 정렬 조건 충족은 {eligible}/{len(d)} LOT·센서 조합이며, 2 LOT 이상 대조할 센서가 없습니다.',
            '각 LOT의 전체 파형에서 피크 수·간격부터 확인하고, 정렬되지 않은 선의 높이 차이만으로 판단하지 않습니다.')


def repair_observation(context):
    before, after, p = context['before'], context['after'], context['pts']
    ta, tb, tc = p['a'][0], p['b'][0], p['c'][1]
    op = ' · '.join(f'Z{z} {p["op_before"][f"OP-Z{z}"]:.0f}→{p["op_after"][f"OP-Z{z}"]:.0f}%' for z in ZONES)
    observed = (f'전후 결과 · 안정 참고 Z4~Z6 온도 {p["level_before"]:.1f}→{p["level_after"]:.1f}℃ '
                f'({p["level_after"]-p["level_before"]:+.1f}℃). 출력 {op}. '
                f'강종 {lot_meta.loc[before,"GRADE"]}→{lot_meta.loc[after,"GRADE"]}의 관측 차이입니다.')
    compare = (f'(a) {ta:%H:%M} 온도 하강 시작 → (b) {tb:%H:%M} 출력 하강 시작: {(tb-ta).total_seconds()/60:.0f}분 차이. '
               f'(c) {tc:%H:%M} 승온 뒤 안정 참고 진입. '
               f'(d) {p["z3_time"]:%H:%M} Z3 단독 반응: Z4~Z6 대비 {p["z3_gap"]:+.1f}℃. '
               '전후 강종과 운전 조건이 달라 출력 감소를 정비 효과로 확정할 수 없습니다.')
    tasks = linked_work(after)
    if len(tasks):
        compare += ' 수리 후 관측 중 작업: ' + ' · '.join(f'{t.MNT_TYPE} {t.STRT_DT:%H:%M}–{t.END_DT:%H:%M}' for t in tasks.itertuples()) + '.'
    return observed, compare


def cp_history_observation(k):
    c = cp_candidates.set_index('LOT').loc[k]
    tasks = linked_work(k, pad_hours=12)
    distances = []
    for t in tasks.itertuples():
        if t.STRT_DT <= c.PEAK_TIME <= t.END_DT:
            distances.append((0, f'{t.MNT_TYPE} 실제 작업 중의 CP 극값'))
        elif c.PEAK_TIME < t.STRT_DT:
            gap = (t.STRT_DT-c.PEAK_TIME).total_seconds()/60
            distances.append((gap, f'{t.MNT_TYPE} 착수 {gap:.1f}분 전의 CP 극값'))
        else:
            gap = (c.PEAK_TIME-t.END_DT).total_seconds()/60
            distances.append((gap, f'{t.MNT_TYPE} 완료 {gap:.1f}분 후의 CP 극값'))
    relation = min(distances, key=lambda item:item[0])[1]
    return (f'{k} CP-Z4 극값 {c.PEAK_VALUE:.3f}, 자체 안정 참고 기준 대비 {c.DELTA:+.3f} ({c.PEAK_TIME:%m/%d %H:%M:%S}).',
            relation + '입니다. 실제 작업 구간과 관측 종료 위치를 대조해 전후 관계를 구분합니다.')


def stage_observation(k):
    d = pd.DataFrame(maintenance_stage_rows)
    d = d.loc[d.LOT.eq(k) & d.센서.eq('TC-Z3')].sort_values('단계')
    first, last = d.iloc[0], d.iloc[-1]
    return (f'Z3 온도 중앙값: {first["구간"]} {first["중앙값"]:.1f}℃ → {last["구간"]} {last["중앙값"]:.1f}℃ (차이 {last["중앙값"]-first["중앙값"]:+.1f}℃).',
            '작업 전 초기 승온과 TBM·CBM 중복을 구분하고, 대응 출력·다른 존의 변화가 함께 나타나는지 봅니다.')


def zoom_observation(context):
    tasks, windows = context['tasks'], context['windows']
    if context['fallback']:
        left, right, _ = windows[-1]
        last = lots.loc[lots.END.le(right)].sort_values('END').iloc[-1]
        at = tasks.DETC_DT.min()
        gap = (at-last.END).total_seconds()/60
        return (f'정비 경계에는 측정 기록이 없어 {int(last.LOT_NO)}의 직전 관측을 봅니다. 관측 종료→감지 간격은 {gap:.1f}분입니다.',
                '직전 CP·온도의 흐름과 실제 BM 시각을 구분합니다. 이 관측을 정비 후 회복 구간으로 해석하지 않습니다.')
    candidates = []
    for t in tasks.itertuples():
        for field, label, _ in EVENT_SPECS:
            at = getattr(t, field)
            before = temp.loc[temp.MEAS_DT.ge(at-pd.Timedelta(minutes=10)) & temp.MEAS_DT.lt(at), 'CP-Z4'].dropna()
            after = temp.loc[temp.MEAS_DT.ge(at) & temp.MEAS_DT.lt(at+pd.Timedelta(minutes=10)), 'CP-Z4'].dropna()
            if len(before) and len(after):
                candidates.append((min(len(before),len(after)), t.MNT_TYPE, label, at, before.median(), after.median()))
    if candidates:
        _, kind, label, at, before, after = max(candidates, key=lambda item:item[0])
        observation = f'{kind} {label} {at:%m/%d %H:%M} 전후 10분의 관측 CP 중앙값 {before:.3f} → {after:.3f} (차이 {after-before:+.3f}).'
    else:
        total = sum(measured_rows(a,b) for a,b,_ in windows)
        observation = f'{len(windows)}개 정비 경계 창에서 측정 {total:,}행을 확인했지만, 경계 양쪽의 CP 관측이 부족합니다.'
    post_work = sum(measured_rows(t.END_DT,t.END_DT+pd.Timedelta(minutes=10)) for t in tasks.itertuples())
    check = ('작업 완료 후 기록이 없어 감지 전후 변화만 확인합니다. 실제 작업 뒤의 회복 여부와 구분합니다.' if post_work==0 else
             '감지·착수·완료 중 어느 경계와 변화가 겹치는지, Z3~Z6 온도·출력도 함께 변했는지 비교합니다.')
    return observation, check


def add_non_lot_observation(fig, name):
    context = getattr(fig, '_caption_context', {})
    parts = name.split('/')
    if parts[0] == '03_강종':
        grade = parts[1]
        if name.endswith('Z9_냉각_주기위상.png'):
            observation, check = phase_observation(grade)
            for row in phase_layout_audit:
                if row['FILE'] == name:
                    row['CAPTION'] = observation + ' ' + check
            # 기존의 정렬 절차 설명은 앞쪽 표시 기준과 해당 표에 모은다.
            for text in list(fig.texts):
                if text is not fig._suptitle and text.get_position()[1] < .10:
                    text.remove()
        else:
            page = parts[-1].removesuffix('_전체비교.png')
            observation, check = grade_observation(grade, page)
    elif context.get('kind') == 'zoom':
        observation, check = zoom_observation(context)
    elif context.get('kind') == 'repair':
        observation, check = repair_observation(context)
    elif context.get('kind') == 'note':          # 그림을 만든 셀이 설명을 직접 넘긴다
        observation, check = context['observation'], context['check']
    else:
        ids = list(dict.fromkeys(int(value) for value in re.findall(r'(?<!\d)(24\d{4})(?!\d)', name)))
        if name.endswith('_CP_작업선후.png'):
            observation, check = cp_history_observation(ids[0])
        elif '정비전중후' in name:
            observation, check = stage_observation(ids[0])
        else:
            raise AssertionError(f'관찰 설명 분류가 없는 그림: {name}')
    # 기존 그래프·제목·범례의 물리적 위치를 유지하고 아래에 설명 공간을 보탠다.
    old_height, extra = fig.get_figheight(), 1.20
    positions = {ax: ax.get_position().frozen() for ax in fig.axes}
    text_positions = {text: text.get_position() for text in fig.texts}
    anchors = {legend:legend.get_bbox_to_anchor().transformed(fig.transFigure.inverted()) for legend in fig.legends}
    fig.set_size_inches(fig.get_figwidth(), old_height+extra)
    for ax, p in positions.items():
        ax.set_position([p.x0,(p.y0*old_height+extra)/(old_height+extra),p.width,p.height*old_height/(old_height+extra)])
    for text, (x,y) in text_positions.items():
        text.set_position((x,(y*old_height+extra)/(old_height+extra)))
    for legend, p in anchors.items():
        legend.set_bbox_to_anchor((p.x0,(p.y0*old_height+extra)/(old_height+extra),p.width,p.height*old_height/(old_height+extra)), transform=fig.transFigure)
    add_observation_caption(fig,.65,old_height+.18,fig.get_figwidth()-1.3,observation,check)

def rv_role_color(w):
    families = rv_families(w)
    if families == ['transition']:
        return RV_TRANSITION
    if w.get('_first'):
        return RV_FIRST
    if families == ['level']:
        return RV_LEVEL
    if families == ['peer']:
        return RV_PEER
    return RV_TOGETHER



def rv_role_key(fig):
    if getattr(fig, '_comparison_view', False):
        return
    for x, name, color in [(14.2, '먼저 볼 변화', RV_FIRST), (17.3, '함께 볼 변화', RV_TOGETHER),
                            (20.4, '초기·종료', RV_TRANSITION)]:
        y = 1 - .49 / fig.get_figheight()
        fig.add_artist(Line2D([x/RV_W, (x+.24)/RV_W], [y, y], transform=fig.transFigure, color=color, lw=2))
        rv_text(fig, x+.34, .40, name, fontsize=10.3, color=color, va='top')



def rv_comparison_rail(ax, family, events):
    color = RV_COLORS[family]
    for a, b in merge_ranges([(e['a'], e['b']) for e in events]):
        ax.plot([a, b], [.985, .985], transform=ax.get_xaxis_transform(),
                color=color, lw=1.5, ls='--' if family == 'peer' else ':', zorder=6)
        for end in [a, b]:
            ax.plot([end, end], [.955, .985], transform=ax.get_xaxis_transform(),
                    color=color, lw=1.3, zorder=6)



def rv_peer_title(k, w, s):
    title = s+' · 같은 강종 LOT 대조'
    if any(e['family'] == 'level' and s in e['sensors'] for e in w['events']):
        row = level_gap.loc[level_gap.LOT.eq(k) & level_gap.SENSOR.eq(s)].iloc[0]
        title += f' · 중앙값 차이 {row.GAP:+.1f}℃'
    return title



def rv_nav_interval(nav, w, family, row):
    own = [e for e in w['events'] if e['family'] == family]
    y = row+.65
    if family in ['peer', 'level']:
        color = RV_COLORS[family]
        for a, b in merge_ranges([(e['a'], e['b']) for e in own]):
            nav.plot([a, b], [y, y], color=color, lw=1.8, ls='--' if family == 'peer' else ':')
            for end in [a, b]:
                nav.plot([end, end], [y-.12, y+.12], color=color, lw=1.3)
    else:
        color = RV_TRANSITION if family == 'transition' else rv_role_color(w)
        nav.plot([w['a'], w['b']], [y, y], color=color, alpha=.16, lw=8, solid_capstyle='butt')
        for a, b in merge_ranges([(e['a'], e['b']) for e in own]):
            nav.plot([a, b], [y, y], color=color, alpha=.9, lw=6, solid_capstyle='butt')




def rv_raw_windows(windows):
    """Keep only events observed within this LOT in the process view."""
    result = []
    for original in windows:
        events = [e for e in original['events'] if e['family'] not in ['peer', 'level']]
        if not events:
            continue
        w = {key: value for key, value in original.items() if key != '_point'}
        w['events'] = events
        w['sensors'] = sorted({s for e in events for s in e['sensors']}, key=SIGNALS.index)
        w['family'] = rv_families(w)[0]
        w['title'] = ' · '.join(RV_NAMES[f] for f in rv_families(w))
        w['title'] = rv_window_title(w)
        result.append(w)
    return result

def rv_peer_windows(k, windows):
    grade = lot_meta.loc[k, 'GRADE']
    if lots.GRADE.eq(grade).sum() < 2:
        return []
    result = []
    for original in windows:
        events = [e for e in original['events'] if e['family'] in ['peer', 'level', 'clean']]
        sensors = sorted({s for e in events for s in e['sensors'] if s in TC}, key=SIGNALS.index)
        for start in range(0, len(sensors), 3):
            selected = sensors[start:start + 3]
            w = dict(original, sensors=selected, events=[e for e in events if set(e['sensors']) & set(selected)])
            w.pop('_point', None)
            result.append(w)
    return result

def rv_matched_peer(k, other, signal, window):
    """Match the same elapsed minute, valid records and stable-reference state."""
    target = rv_profile(k, signal).where(states[k].eq('안정 참고'))
    target = target.loc[(target.index + .5 >= window['a']) & (target.index + .5 <= window['b'])]
    peer = rv_profile(other, signal).where(states[other].eq('안정 참고')).reindex(target.index)
    valid = target.notna() & peer.notna()
    frame = pd.DataFrame({'target': target.where(valid), 'peer': peer.where(valid)})
    matched = frame.dropna()
    if matched.empty:
        return frame, None
    scale = sensor_scale(signal)
    a, b = float(matched.index.min()), float(matched.index.max())
    row = {'대상 LOT': k, '대조 LOT': other, '강종': lot_meta.loc[k, 'GRADE'], '신호': signal,
           '원본 구간': window['label'], '공통 유효 분': len(matched), '경과 시작[분]': a + .5,
           '경과 끝[분]': b + .5, '대상 중앙값': float(matched.target.median() * scale),
           '대조 중앙값': float(matched.peer.median() * scale),
           '중앙값 차이': float((matched.target.median() - matched.peer.median()) * scale),
           '같은 분 최대 절대 차이': float((matched.target - matched.peer).abs().max() * scale),
           '대상 실제 범위': rv_stamp(k, a, b + 1), '대조 실제 범위': rv_stamp(other, a, b + 1)}
    return frame, row

rv_peer_comparison_audit = []

def rv_peer_review(k, window, page):
    grade = lot_meta.loc[k, 'GRADE']
    others = lots.loc[lots.GRADE.eq(grade) & lots.LOT_NO.ne(k), 'LOT_NO'].astype(int).tolist()
    sensors = window['sensors']
    start, pitch, plot_height = 2.75, 3.35, 1.55
    fig = plt.figure(figsize=(RV_W, start + pitch * len(sensors) + 1.8), facecolor='white')
    fig._comparison_view = True
    rv_text(fig, .65, .35, f'④ 같은 강종 LOT 비교 · {grade}', fontsize=23, color=RV_INK, weight='bold', va='top')
    rv_text(fig, .65, .90, f'대상 {k} ({lot_meta.loc[k, "START"]:%Y/%m/%d})  ↔  '
            + ' · '.join(f'{o} ({lot_meta.loc[o, "START"]:%m/%d})' for o in others), fontsize=13, color=RV_INK)
    rv_text(fig, .65, 1.31, '비교 기준: 같은 측정 신호 · LOT 시작 후 같은 경과 분 · 양쪽 안정 참고 · 1분 기록량 90% 이상', fontsize=12, color=RV_SUB)
    purpose = '안정 수준 차이 확인' if any(e['family'] == 'level' for e in window['events']) else '선택 시간대의 파형 차이 확인'
    rv_text(fig, .65, 1.72, f'{window["label"]} 연결 · {purpose} · 선 하나 = 서로 다른 날짜의 LOT', fontsize=12.5, color=RV_INK)
    rv_text(fig, .65, 2.13, f'대상에서 고른 범위  {rv_stamp(k, window["a"], window["b"])}', fontsize=11.5, color=RV_SUB)
    styles = ['--', ':', '-.']
    records = []
    for row_index, signal in enumerate(sensors):
        ax = rv_axis(fig, 1.65, start + row_index * pitch, 21.4, plot_height)
        target_line = rv_profile(k, signal) * np.nan
        row_records = []
        missing = []
        for j, other in enumerate(others):
            frame, record = rv_matched_peer(k, other, signal, window)
            if record is None:
                missing.append(other)
                continue
            target_line = target_line.combine_first(frame.target)
            ax.plot(frame.index + .5, frame.peer * sensor_scale(signal), color=LOT_COLOR[other],
                    lw=1.2, ls=report_viz.lot_linestyle(other),
                    label=f'대조 {other} · {lot_meta.loc[other, "START"]:%m/%d}')
            row_records.append(record)
        target_line = target_line.loc[(target_line.index + .5 >= window['a']) & (target_line.index + .5 <= window['b'])]
        ax.plot(target_line.index + .5, target_line * sensor_scale(signal), color=LOT_COLOR[k], lw=2.0, ls=report_viz.lot_linestyle(k),
                label=f'대상 {k} · {lot_meta.loc[k, "START"]:%m/%d}', zorder=4)
        rv_style(ax)
        rv_panel(ax, [signal], f'{rv_process(signal)} · {signal} | 같은 신호의 LOT 간 비교')
        finite = target_line.dropna()
        if len(finite) > 1:
            ax.set_xlim(float(finite.index.min()) + .5, float(finite.index.max()) + .5)
        else:
            ax.set_xlim(window['a'], window['b'])
        unit_ = '℃' if signal in TC else '%p' if signal in OP else ' (단위 미확인)'
        if not row_records:
            ax.text(.5, .5, '양쪽의 공통 유효 분 없음 · 이 범위에서 LOT 비교 불가', transform=ax.transAxes,
                    ha='center', va='center', color=RV_SUB, fontsize=12)
        for j, record in enumerate(row_records):
            summary = (f'대조 {record["대조 LOT"]}: {k} {record["대상 중앙값"]:.1f}{unit_} ↔ '
                       f'{record["대조 LOT"]} {record["대조 중앙값"]:.1f}{unit_} · 중앙값 차이 {record["중앙값 차이"]:+.1f}{unit_} · '
                       f'공통 유효 {record["공통 유효 분"]}분 (경과 {record["경과 시작[분]"]:.1f}–{record["경과 끝[분]"]:.1f}분)')
            rv_text(fig, 1.65, start + row_index * pitch + plot_height + .72 + j * .25,
                    summary, fontsize=10.7, color=RV_INK)
        if missing:
            rv_text(fig, 1.65, start + row_index * pitch + plot_height + .72 + len(row_records) * .25,
                    '비교 제외: ' + ' · '.join(map(str, missing)) + ' · 양쪽 공통 유효 분 없음(관측 범위·안정 참고 조건 불일치)',
                    fontsize=10.7, color=RV_SUB)
        records.extend(row_records)
    if records:
        largest = max(records, key=lambda item: abs(item['중앙값 차이']))
        unit_ = '℃' if largest['신호'] in TC else '%p' if largest['신호'] in OP else ' (단위 미확인)'
        observation = (f'{grade}끼리 비교: 대상 {k}의 {largest["신호"]} 중앙값은 '
                       f'{largest["대조 LOT"]}보다 {largest["중앙값 차이"]:+.1f}{unit_}. '
                       '각 수치는 두 LOT의 공통 유효 분에서만 계산했습니다.')
        check = ('같은 강종이 2 LOT뿐이므로 두 LOT의 차이까지 확인할 수 있고, 어느 쪽이 평소와 다른지는 정할 수 없습니다.'
                 if len(others) == 1 else '대상과 대조 LOT마다 같은 방향의 차이가 남는지 확인합니다. 공통 유효 분은 대조 LOT마다 다릅니다.')
        if 'TC-Z3' in sensors:
            check += ' Z3는 반복 시점 차이와 중심 수준 차이를 구분해 봅니다.'
    else:
        observation, check = f'{grade}의 다른 LOT은 있지만 이 범위에는 양쪽 공통 유효 분이 없습니다.', '비교 가능한 기록이 있는 다른 시간대에서 확인합니다.'
    footer = start + pitch * len(sensors) + .02
    add_observation_caption(fig, 1.65, footer, 21.4, observation, check, section=window['label'])
    name = rv_directory(k) + f'/04_같은강종LOT비교_{page:02d}.png'
    rv_save(fig, name, k, '같은 강종·같은 신호 LOT 비교', window)
    for record in records:
        record['파일'] = name
        rv_peer_comparison_audit.append(record)
    rv_window_audit.append({'LOT': k, '번호': window['label'], '유형': '같은 강종 LOT 비교', '시각': window['at'],
                           '시작[분]': window['a'], '끝[분]': window['b'], '센서': ' · '.join(sensors),
                           '관찰 수': len(window['events']), '기존 확대 수': window['seed_count'], '파일': name})


def detail_legend(ax, title, sensors=None, heading=None):
    """Each graph owns a separate title row and a separate sensor legend row."""
    if heading:
        ax.set_title(f'{heading}\n{title}', loc='left', fontsize=14, pad=37, linespacing=1.65, color='#17344d')
    else:
        ax.set_title(title, loc='left', fontsize=14, pad=31, color='#17344d')
    handles, labels=ax.get_legend_handles_labels()
    unique=dict(zip(labels,handles))
    if unique:
        leg=ax.legend(unique.values(),unique.keys(),loc='lower left',bbox_to_anchor=(0,1.015),
                      borderaxespad=0,ncol=min(4,len(unique)),frameon=False,fontsize=11,
                      handlelength=2,columnspacing=1.5)
        leg._detail_legend=True


def detail_clock(ax, lo, hi, bottom=True):
    ax.set_xlim(lo,hi)
    style(ax,'실제 측정 시각[시:분]' if bottom else '')
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4,maxticks=7))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.yaxis.set_major_locator(MaxNLocator(4))
    ax.tick_params(labelsize=11,labelbottom=bottom)


def detail_tag(ax, at, text, level=1.06, color=None):
    # Reserve a physical row between the sensor legend and the panel title.
    height=ax.figure.get_figheight()*ax.get_position().height
    level=1+17/72/height
    tag=ax.text(at,level,text,transform=ax.get_xaxis_transform(),ha='center',va='bottom',
                fontsize=12,fontweight='bold',color=color or COL['FOCUS'],clip_on=False)
    tag._review_label=True
    return tag


def detail_raw(ax, k, sensors, lo, hi, title, bottom=False, heading=None):
    w=raw[k].set_index('MEAS_DT').loc[lo:hi].asfreq('s')
    for s in sensors:
        scale=1e10 if s=='CP-Z4M' else 1
        color='#6a3d9a' if s.startswith('CP') else sensor_color(s)
        ax.plot(w.index,w[s]*scale,label=s,color=color,lw=1)
    ax.set_ylabel('CP-Z4M (단위 미확인·10⁻¹⁰)' if sensors==['CP-Z4M'] else
                  'CP 기록값[단위 미확인]' if sensors==['CP-Z4'] else
                  '온도[℃]' if sensors[0].startswith('TC') else '제어출력[%]',fontsize=12,labelpad=8)
    detail_clock(ax,lo,hi,bottom)
    detail_legend(ax,title,heading=heading)
    return w


def event_figure(k, looks, windows, name, title, observation, check, z3=False, marks=()):
    r=lot_meta.loc[k];g=RAW_T[k];peaks=cp_peaks(k)
    rows=[('CP 기록 · CP-Z4',['CP-Z4']),('CP 보조 기록 · CP-Z4M',['CP-Z4M'])]
    if z3:
        rows += [('가열 초입 Z3 · 온도',['TC-Z3']),('가열 초입 Z3 · 출력',['OP-Z3'])]
    rows += [('가열·균열 Z4~Z6 · 온도',[f'TC-Z{z}' for z in [4,5,6]]),
             ('가열·균열 Z4~Z6 · 출력',[f'OP-Z{z}' for z in [4,5,6]])]
    fig=plt.figure(figsize=(28,7+3.1*len(rows)))
    gs=fig.add_gridspec(len(rows)+1,len(windows),height_ratios=[1.1]+[1]*len(rows),
                        left=.075,right=.975,top=.84,bottom=.07,hspace=1.05,wspace=.16)
    top=fig.add_subplot(gs[0,:])
    detail_raw(top,k,['CP-Z4'],r.START,r.END,'(1) LOT 전체 CP · 아래 확대의 위치',bottom=True)
    top.get_legend().remove()  # CP-Z4 is already named on this single-signal overview.
    top.set_title('(1) LOT 전체 CP · 아래 확대의 위치',loc='left',fontsize=14,pad=46)
    top.plot(peaks.index,peaks+g['CP-Z4'].median(),'o',ms=5,color=COL['FOCUS'])
    overview_groups=[]
    for j,(a,b,_) in enumerate(looks):
        top.axvspan(a,b,color=COL['FOCUS'],alpha=.16,lw=0,zorder=0)
        if overview_groups and a<=overview_groups[-1][1]:
            aa,bb,labels=overview_groups[-1];overview_groups[-1]=(aa,max(bb,b),labels+[chr(97+j)])
        else:overview_groups.append((a,b,[chr(97+j)]))
    for a,b,labels in overview_groups:
        detail_tag(top,a+(b-a)/2,'('+','.join(labels)+')',level=1.07)
    for col,(lo,hi) in enumerate(windows):
        for row,(label,ss) in enumerate(rows):
            ax=fig.add_subplot(gs[row+1,col])
            heading=f'({col+2}) {lo:%Y/%m/%d %H:%M} → {hi:%m/%d %H:%M}' if row==0 else None
            detail_raw(ax,k,ss,lo,hi,label,bottom=row==len(rows)-1,heading=heading)
            if ss==['CP-Z4']:
                inside=peaks.loc[lo:hi]
                ax.plot(inside.index,inside+g['CP-Z4'].median(),'o',ms=6,color=COL['FOCUS'],zorder=6)
            for j,(a,b,_) in enumerate(looks):
                if lo<=a<=hi or lo<=b<=hi or a<lo<hi<b:
                    ax.axvspan(max(a,lo),min(b,hi),color=COL['FOCUS'],alpha=.12,lw=0,zorder=0)
            for t,text in marks:
                if lo<=t<=hi:ax.axvline(t,color='#7b8794',lw=1.1,ls='--',zorder=0)
    details=' · '.join(f'({chr(97+j)}) {a:%m/%d %H:%M}–{b:%H:%M} {text}' for j,(a,b,text) in enumerate(looks))
    work=' · '.join(f'{t:%m/%d %H:%M} {text}' for t,text in marks)
    header(fig,title,f'{k} · {r.GRADE} · 1초 원본 · 빨간 음영: 확인 구간 · Z3와 Z4~Z6 별도 축 · 빈 시각은 채우지 않음')
    compact_header(fig,.93,.90,title_size=24,sub_size=13,legend_size=12)
    axis_tag(fig,TAG_REAL)
    fig._caption_context={'kind':'note','observation':observation+' '+details,
                          'check':check+(' 작업 경계: '+work+'.' if work else '')}
    finish(fig,name)


In [ ]:
def lot_history(k):
    r=lot_meta.loc[k];tasks=linked_work(k,pad_hours=72)
    rows=[]
    for t in tasks.itertuples():
        overlap=max(0,(min(r.END,t.END_DT)-max(r.START,t.STRT_DT)).total_seconds()/60)
        detection_in=r.START<=t.DETC_DT<=r.END
        rows.append({'LOT':k,'WO':t.WO_NO,'유형':t.MNT_TYPE,'관계':'작업 교집합' if overlap else '관측 중 감지' if detection_in else '관측 밖 인근 이력',
                     '감지':t.DETC_DT,'착수':t.STRT_DT,'완료':t.END_DT,'관측·작업 교집합[분]':overlap,
                     '관측 종료 → 착수[분]':(t.STRT_DT-r.END).total_seconds()/60,'작업 내용':remark_text(t),
                     '확인된 교체 부품':'; '.join(parts_for(t.WO_NO))})
    if rows:table(pd.DataFrame(rows).rename(columns={'유형':'유형'}))
    if rows:
        overlap_count = sum(row['관측·작업 교집합[분]'] > 0 for row in rows)
        detect_count = sum(r.START <= row['감지'] <= r.END for row in rows)
        note(f'**이력 관계:** 관측 ±72시간 CAL01 작업 {len(rows)}건 중, 실제 관측·작업 교집합 {overlap_count}건 · 관측 중 감지 {detect_count}건입니다.')
    else:
        note(f'**이력 관계:** {k} 관측 ±72시간에 연결되는 CAL01 작업 이력이 없어 자체 파형·다른 LOT 대조를 확인합니다.')

def cp_event_flow(k,name):
    r=lot_meta.loc[k];tasks=linked_work(k,pad_hours=12)
    if not len(tasks) or k not in set(cp_candidates.LOT):return
    c=cp_candidates.set_index('LOT').loc[k]
    lo=r.START-pd.Timedelta(minutes=10);hi=max(r.END,tasks.END_DT.max())+pd.Timedelta(minutes=30)
    fig,(ax,events)=plt.subplots(2,1,figsize=(28,13),sharex=True,gridspec_kw={'height_ratios':[2.2,1]})
    fig.subplots_adjust(left=.09,right=.975,top=.78,bottom=.10,hspace=.48)
    line(ax,k,'CP-Z4',clock=True,envelope=True)
    ax.axhline(c.BASE,color='#455a64',ls='--',lw=1)
    p0=r.START+pd.Timedelta(minutes=float(c.START_MIN));p1=r.START+pd.Timedelta(minutes=float(c.END_MIN))
    for a in [ax,events]:      # 볼 구간 (a): CP가 자체 기준에서 0.05 넘게 벗어나 있던 펄스 구간
        a.axvspan(p0,p1,color=COL['FOCUS'],alpha=.15,lw=0,zorder=0)
        a.axvline(c.PEAK_TIME,color=COL['FOCUS'],lw=1.5);a.axvline(r.END,color='#455a64',ls=':',lw=1.3)
    for row,t in enumerate(tasks.itertuples()):
        events.plot([t.STRT_DT,t.END_DT],[row,row],color=COL[t.MNT_TYPE],lw=10,solid_capstyle='butt')
        events.scatter(t.DETC_DT,row,marker='v',s=80,color=COL['DETECT'])
        if parts_for(t.WO_NO):events.scatter(t.END_DT,row,marker='D',s=65,color=COL['PART'])
    events.set_yticks(range(len(tasks)),[f'{t.MNT_TYPE} · {t.WO_NO}\n{remark_text(t)}' for t in tasks.itertuples()])
    events.set_ylim(len(tasks)-.5,-.5);ax.set_xlim(lo,hi)
    for a in [ax,events]:clock_axis(a);a.tick_params(labelsize=12)
    ax.set_title('(1) CP 전체 관측과 극값 위치',loc='left',fontsize=16,pad=12)
    events.set_title('(2) 같은 실제 시각의 정비 이력',loc='left',fontsize=16,pad=12)
    ax.text(c.PEAK_TIME,.97,f' (a) CP 펄스 {p0:%H:%M:%S}–{p1:%H:%M:%S} · 극값 {c.PEAK_VALUE:.3f}',transform=ax.get_xaxis_transform(),ha='left',va='top',
            fontsize=13,fontweight='bold',color=COL['FOCUS'],bbox=dict(boxstyle='round,pad=.2',fc='white',ec='none',alpha=.85))
    header(fig,f'{k} CP 변화·정비 시각 대조',f'{r.GRADE} · CP 극값 {c.PEAK_TIME:%m/%d %H:%M:%S}과 실제 정비 시각 대조',
           [Patch(facecolor=COL['FOCUS'],alpha=.3,label='(a) CP 펄스 구간 · 선은 극값'),Line2D([0],[0],color='#455a64',ls=':',label='제공 관측 끝'),
            *[Patch(facecolor=COL[x],label=x+' 작업') for x in tasks.MNT_TYPE.unique()],
            Line2D([0],[0],marker='v',color=COL['DETECT'],ls='None',label='감지 기록'),Line2D([0],[0],marker='D',color=COL['PART'],ls='None',label='교체 이력 · WO 완료 위치')],ncol=6)
    compact_header(fig,.935,.905,title_size=24,sub_size=13,legend_size=12);axis_tag(fig,TAG_REAL);finish(fig,name)
    for t in tasks.itertuples():
        if c.PEAK_TIME > t.END_DT:
            relation=f'작업 완료 {(c.PEAK_TIME-t.END_DT).total_seconds()/60:.2f}분 뒤 CP 극값'
        elif c.PEAK_TIME < t.STRT_DT:
            relation=f'CP 극값 {(t.STRT_DT-c.PEAK_TIME).total_seconds()/60:.2f}분 뒤 작업 착수'
        else:
            relation='실제 작업 중 CP 극값'
        note(f'**기록 시각 대조:** {t.WO_NO}: 감지 {t.DETC_DT:%m/%d %H:%M}, 작업 {t.STRT_DT:%m/%d %H:%M}–{t.END_DT:%m/%d %H:%M} [{remark_text(t)}]. '
             f'CP 극값 {c.PEAK_TIME:%m/%d %H:%M:%S} · {relation}. 제공 관측 끝 {r.END:%m/%d %H:%M:%S}.')
        if parts_for(t.WO_NO):note('**같은 WO 교체:** '+'; '.join(parts_for(t.WO_NO))+'. WO 완료 위치는 실제 교체 순간이 아니다.')
    if k==240117:
        start=r.START+pd.Timedelta(minutes=float(c.START_MIN));end=r.START+pd.Timedelta(minutes=float(c.END_MIN))
        before=raw[k].loc[raw[k].MEAS_DT.ge(start-pd.Timedelta(minutes=10))&raw[k].MEAS_DT.lt(start),'CP-Z4'].median()
        after=raw[k].loc[raw[k].MEAS_DT.ge(end)&raw[k].MEAS_DT.lt(end+pd.Timedelta(minutes=10)),'CP-Z4'].median()
        note(f'**관측 양상:** CP-Z4는 하강 직전 10분 중앙값 {before:.3f}, 극값 {c.PEAK_VALUE:.3f}, 직후 10분 중앙값 {after:.3f}다. '
             '짧은 하강 뒤 평소 수준으로 돌아왔다. 마모량 초과는 정비 기록의 내용이며, 이 CP 하강의 원인이나 CP 복귀의 정비 효과를 뜻하지 않는다. '
             '마모 부품·마모량과 CP 센서/계산/분위기 제어 기록을 함께 확인해야 한다.')
    note('**확인 범위:** 시간 선후관계와 작업 중 여부는 확인할 수 있다. CP 변화가 작업의 원인이나 결과인지는 이 순서만으로 판단할 수 없다. 관측이 끝난 뒤의 센서 값은 그리지 않는다. 정비 경계와의 거리가 우연보다 가까운지는 아래 [CP 변화 절](#cp-maintenance)에서 무작위 관측 시각과 비교한다.')


In [ ]:
PAGE_SIGNALS={"세정·건조":[['CLN-IN'],['TC-Z1','OP-Z1'],['TC-Z2','OP-Z2']],
"가열·균열·CP":[['TC-Z3','OP-Z3'],['TC-Z4','OP-Z4'],['CP-Z4','CP-Z4M']],
"가열·과시효":[['TC-Z5','OP-Z5'],['TC-Z6','OP-Z6'],['TC-Z7','OP-Z7'],['TC-Z8','OP-Z8']],
"Z9·냉각":[['TC-Z9'],['TC-OUT1','TC-OUT2']]}
# 같은 강종 안에서 '어느 LOT이, 어떤 신호에서, 어떻게' 다른지 세 종류로 나눈다.
#  전반적인 수준 차이: LOT 중앙값이 나머지 LOT보다 온도 2.5℃(03의 ② 기준)·출력 5%p 이상 다르고, 비교 가능한 분의 80% 이상에서 같은 방향
#  일시적인 변화: CP-Z4가 자체 기준에서 0.05 넘게 짧게 벗어남(예정된 정지 제외) · Z3가 같은 경과 분 대조보다 3℃ 넘게 벗어난 구간
#  반복 시점 차이: 반복 파형(Z9·냉각)인데 관측이 두 주기보다 짧아, 어느 부분을 봤는지에 따라 중앙값이 달라진 경우
GRADE_DIFF={}
def grade_differences(grade):
    if grade in GRADE_DIFF:return GRADE_DIFF[grade]
    ids=lots.loc[lots.GRADE.eq(grade),'LOT_NO'].astype(int).tolist();rows=[]
    period=cycle_table.groupby('SENSOR').PERIOD_MIN.median()
    gaps=level_gap.loc[level_gap.GRADE.eq(grade)&level_gap.SENSOR.isin(TC+OP)].dropna(subset=['GAP'])
    for s,d in gaps.groupby('SENSOR',sort=False):
        r=d.loc[d.GAP.abs().idxmax()];k=int(r.LOT);limit,unit=(2.5,'℃') if s in TC else (5,'%p')
        if abs(r.GAP)<limit:continue
        stable=int(states[k].eq('안정 참고').sum())
        row={'차이 종류':'','신호':s,'공정':PROCESS[s],'다른 LOT':k,'그 LOT':f"{r.LOT_MEDIAN:.1f}",'나머지 LOT':f"{r.PEER_MEDIAN:.1f}",'차이':f"{r.GAP:+.1f}{unit}",'다른 강종 같은 수준':''}
        if s in period and stable<2*period[s]:
            row.update({'차이 종류':'반복 시점 차이','근거':f"안정 관측 {stable}분 < 두 주기 {2*period[s]:.0f}분",
                        '알 수 있는 것':'반복 파형의 일부만 관측해 중앙값이 달라졌다. 수준 차이로 보지 않고 아래 주기·위상 그림으로 본다.'})
        else:
            delta=comparison[k,s].DELTA.dropna()
            share=(np.sign(delta).eq(np.sign(r.GAP))&delta.abs().ge(limit/2)).mean()
            row.update({'차이 종류':'전반적인 수준 차이' if share>=.8 else '일부 구간 차이','근거':f"비교 가능한 {len(delta)}분 중 {share:.0%}가 같은 방향"})
            if s in OP:
                tc=level_gap.loc[level_gap.LOT.eq(k)&level_gap.SENSOR.eq(s.replace('OP','TC')),'GAP'].iloc[0]
                row['알 수 있는 것']=('같은 구역 온도는 1℃ 안으로 같은데 출력만 다르다. 같은 온도에서 제어출력 수준이 다르다. 부하나 제어 조건은 추가 확인이 필요하며 실제 열량 차이는 알 수 없다.'
                                   if abs(tc)<1 else '같은 구역 온도도 함께 다르다. 설정 온도 차이에 따른 출력 차이로 본다.')
            elif share>=.8:
                # 같은 수준의 LOT이 다른 강종에도 있으면 강종 특성보다 LOT별 운전 단계 차이다.
                same=level_gap.loc[level_gap.SENSOR.eq(s)&level_gap.GRADE.ne(grade)&(level_gap.LOT_MEDIAN-r.LOT_MEDIAN).abs().le(.5)]
                row['다른 강종 같은 수준']=', '.join(f'{int(o.LOT)} {o.GRADE}' for o in same.itertuples())
                row['알 수 있는 것']='관측 내내 일정하게 다르다. 순간 이상이 아니라 LOT별 운전 설정·조건 차이로 보이며, 모델에는 강종 평균 대신 LOT 기준선을 함께 둔다.'
            else:
                row['알 수 있는 것']='일부 시간대에서만 다르다. 해당 LOT의 구간 확대 그림에서 시각을 확인한다.'
        rows.append(row)
    for c in cp_candidates.loc[cp_candidates.LOT.isin(ids)].itertuples():
        if in_expected(int(c.LOT),c.PEAK_MIN):continue
        rows.append({'차이 종류':'일시적인 변화','신호':'CP-Z4','공정':PROCESS['CP-Z4'],'다른 LOT':int(c.LOT),'그 LOT':f"{c.PEAK_VALUE:.3f}",
                     '나머지 LOT':f"{c.BASE:.3f}(자체 기준)",'차이':f"{c.DELTA:+.3f}",'근거':f"{c.PEAK_TIME:%m/%d %H:%M:%S} · {c.PULSE_WIDTH_MIN:.1f}분",
                     '알 수 있는 것':'짧게 튀었다 돌아왔다. LOT 전체 특성이 아니라 그 시각의 사건이므로, 같은 시각 Z4 온도·출력과 정비 이력을 LOT 확대 그림에서 본다.'})
    z3=episodes.loc[episodes.GRADE.eq(grade)&episodes.SENSOR.eq('TC-Z3')]
    if len(z3):
        top=z3.loc[z3.MAX_ABS_DELTA_C.idxmax()];n=z3.LOT.nunique()
        rows.append({'차이 종류':'일시적인 변화','신호':'TC-Z3','공정':PROCESS['TC-Z3'],'다른 LOT':f"공통 {n} LOT" if n>1 else int(top.LOT),
                     '그 LOT':'-','나머지 LOT':'-','차이':f"최대 {top.MAX_ABS_DELTA_C:.1f}℃({int(top.LOT)})",'근거':f"{len(ids)} LOT 중 {n} LOT에 3℃ 넘는 구간 {len(z3)}개",
                     '알 수 있는 것':('여러 LOT에서 시각만 다르게 나타난다. 특정 LOT의 차이가 아니라 Z3 구역의 공통 출렁임이다.' if n>1 else
                                  '이 LOT에서만 나타난다. 해당 시각의 Z3 출력과 정비 이력을 확인한다.')})
    GRADE_DIFF[grade]=pd.DataFrame(rows,columns=['차이 종류','신호','공정','다른 LOT','그 LOT','나머지 LOT','차이','근거',
                                               '다른 강종 같은 수준','알 수 있는 것']).fillna('')
    return GRADE_DIFF[grade]

def grade_difference_note(grade):
    """강종 비교 맨 앞에 '어느 LOT이 무엇이 다른지'와 그래서 알 수 있는 것을 적는다."""
    d=grade_differences(grade);ids=lots.loc[lots.GRADE.eq(grade),'LOT_NO'].astype(int).tolist()
    lines=[f"**{grade}에서 다른 LOT** (온도 2.5℃·출력 5%p 이상 또는 CP 0.05 이상)"]
    for k,g in d.groupby('다른 LOT',sort=False):
        lines.append(f"- **{k}**: "+' · '.join(f"{r['신호']} {r['차이']}({r['차이 종류']})" for _,r in g.iterrows()))
    lines.append('\n**그래서 알 수 있는 것**')
    for (kind,meaning),g in d.groupby(['차이 종류','알 수 있는 것'],sort=False):
        lines.append(f"- **{kind} · {', '.join(dict.fromkeys(g.신호))}:** {meaning}")
    same=d.loc[d['다른 강종 같은 수준'].ne('')]
    if len(same):
        lines.append('- **강종 차이인가?** '+' / '.join(f"{r['신호']} {r['그 LOT']}℃ 수준은 다른 강종에도 있다({r['다른 강종 같은 수준']})" for _,r in same.iterrows())
                     +'. 강종 고유 특성보다 LOT마다 정해진 운전 단계 차이로 읽는다.')
    # 기준 ±20% 확인(온도·출력 수준): 기준을 올리면 빠지는 차이, 내리면 더해지는 차이
    gaps=level_gap.loc[level_gap.GRADE.eq(grade)&level_gap.SENSOR.isin(TC+OP)].dropna(subset=['GAP'])
    top=gaps.loc[gaps.groupby('SENSOR').GAP.apply(lambda x:x.abs().idxmax())]
    lim=top.SENSOR.map(lambda s:2.5 if s in TC else 5)
    fmt=lambda t:', '.join(f"{int(r.LOT)} {r.SENSOR} {r.GAP:+.1f}" for r in t.itertuples()) or '없음'
    lines.append(f"- **기준 ±20% 확인:** 20% 올리면 빠지는 차이 {fmt(top.loc[top.GAP.abs().ge(lim)&top.GAP.abs().lt(1.2*lim)])} / "
                 f"20% 내리면 더해지는 차이 {fmt(top.loc[top.GAP.abs().ge(.8*lim)&top.GAP.abs().lt(lim)])}. 나머지 차이는 기준을 바꿔도 남는다.")
    if len(ids)==2:lines.append('- 2 LOT뿐이라 어느 쪽이 평소와 다른지는 정할 수 없고, 두 LOT 사이의 차이로 읽는다.')
    note('\n'.join(lines));table(d)

def grade_compare(grade,page,name):
    ids=lots.loc[lots.GRADE.eq(grade),'LOT_NO'].astype(int).tolist();rows=PAGE_SIGNALS[page]
    diff=grade_differences(grade)
    fig,axes=plt.subplots(len(rows),2,figsize=(28,4+4.3*len(rows)),squeeze=False)
    fig.subplots_adjust(left=.085,right=.975,top=.84,bottom=.055,wspace=.22,hspace=.75)
    for row,ss in enumerate(rows):
        for col in range(2):
            ax=axes[row,col]
            if col>=len(ss):ax.set_axis_off();continue
            s=ss[col];hit=diff.loc[diff.신호.eq(s)]
            for k in ids:
                p=profile[k];scale=sensor_scale(s);bold=k in set(hit['다른 LOT'])      # 다른 LOT은 굵게
                ax.plot(p.index,p[(s,'median')]*scale,color=LOT_COLOR[k],lw=2.8 if bold else 1.5,zorder=3 if bold else 2, ls=report_viz.lot_linestyle(k))
            title=f"{PROCESS[s]} [{s}]"+''.join(f" · {r['다른 LOT']} {r['차이']} {r['차이 종류']}" for _,r in hit.iterrows())
            ax.set_title(title,loc='left',fontsize=15,pad=12,color=SELECT_COLOR if len(hit) else '#17344d')
            ax.set_ylabel(shown_unit(s));style(ax);ax.set_xlim(0,lots.loc[lots.GRADE.eq(grade),'MINUTES'].max());ax.tick_params(labelsize=12)
    header(fig,f"같은 강종 LOT 전체 비교 · {grade} · {page}",f"선 하나 = 서로 다른 날짜의 LOT · 패널 하나 = 같은 측정 신호 · {grade} {len(ids)} LOT · 차이 수치는 각 LOT 안정 참고 중앙값 기준",lot_handles(ids)+[QUALITY_HANDLE],ncol=5)
    compact_header(fig,.945,.915,title_size=24,sub_size=13,legend_size=12);axis_tag(fig,TAG_ELAPSED);finish(fig,name)

def grade_peer_note(grade):
    """별도의 차이·대조 범위 그림을 만들지 않고 전체 비교 아래 한 줄만 안내한다."""
    ids = lots.loc[lots.GRADE.eq(grade), 'LOT_NO'].astype(int).tolist()
    lengths = sorted((float(lot_meta.loc[k, 'MINUTES']) for k in ids), reverse=True)
    if len(ids) == 2:
        text = f'같은 강종 2 LOT → 다른 1 LOT과의 참고 비교. 공통 관측창은 경과 {lengths[1]:.1f}분까지다.'
    else:
        text = (f'같은 강종 {len(ids)} LOT → 대상 제외 최대 {len(ids)-1} LOT. '
                f'다른 2 LOT 이상을 확보할 수 있는 관측창은 최대 경과 {lengths[2]:.1f}분까지다.')
    note('**대조 조건:** ' + text + ' 센서별 안정 참고·기록량 조건에 따라 일부 시간은 제외되며, 해당 확대 패널 옆에서 실제 대조 수를 확인한다.')


In [ ]:
ZONES = [3, 4, 5, 6]
ZONE_COLOR = report_viz.ZONE_COLORS.copy()   # 구역마다 다른 색 · LOT은 제목 색으로 구분
GRADE_COLOR = report_viz.GRADE_COLORS.copy()

def remark_text(task):
    return str(task.REMARK).strip() if pd.notna(task.REMARK) else "작업 내용 미기재"

def part_text(wo):
    d = cal_parts.loc[cal_parts.WO_NO.eq(wo)]
    return ", ".join(f"{p.PART_NM} ×{int(p.QTY)}({p.CHG_RSN})" for p in d.itertuples())

def work_label(t):
    """작업 표시: REMARK · 작업 길이 · FAIL_MODE 코드 · 교체 부품."""
    text = f"{remark_text(t)} · {(t.END_DT - t.STRT_DT).total_seconds() / 60:.0f}분"
    if pd.notna(t.FAIL_MODE):
        text += f" · FAIL_MODE {t.FAIL_MODE}"
    if part_text(t.WO_NO):
        text += f" · 교체: {part_text(t.WO_NO)}"
    return text

def grade_periods(lo, hi):
    """lo~hi 사이 강종 구간: 각 강종변경부터 다음 변경까지."""
    ev = grade_evt.loc[grade_evt.EVT_DT.lt(hi)]
    ev = pd.concat([ev.loc[ev.EVT_DT.le(lo)].tail(1), ev.loc[ev.EVT_DT.gt(lo)]])
    starts = [max(t, lo) for t in ev.EVT_DT]
    return list(zip(starts, starts[1:] + [hi], ev.AFT_VAL))

def raw_window(lot, a, b):
    """1초 격자에 원본을 놓아 기록 공백에서 선을 끊는다. 값을 채우지 않는다."""
    r = lot_meta.loc[lot]
    times = pd.date_range(r.START + pd.Timedelta(seconds=int(a*60)), min(r.END, r.START + pd.Timedelta(seconds=int(b*60))), freq="s")
    return raw[lot].set_index("MEAS_DT").reindex(times)

def zone_panel(ax, lot, a, b, kind, title, date_heading=False):
    w = raw_window(lot, a, b)
    for z in ZONES:
        ax.plot(w.index, w[f"{kind}-Z{z}"], color=ZONE_COLOR[z], lw=1.4, label=f"{kind}-Z{z}")
    # Two-line column header above the plot, with a separate sensor-legend row.
    if date_heading:
        dates = (f"{w.index[0]:%Y/%m/%d} · {w.index[0]:%H:%M}–{w.index[-1]:%H:%M}"
                 if w.index[0].date() == w.index[-1].date()
                 else f"{w.index[0]:%m/%d %H:%M}–{w.index[-1]:%m/%d %H:%M}")
        ax.set_title(f"{title}\n{dates}", loc="left", fontsize=16, pad=48,
                     color=RV_INK, linespacing=1.6)
    else:
        ax.set_title(title, loc="left", fontsize=15, pad=42, color=RV_INK)
    leg = ax.legend(loc="lower left", bbox_to_anchor=(0, 1.02), borderaxespad=0,
                    ncol=4, frameon=False, fontsize=12, handlelength=1.8, columnspacing=1.2)
    leg._detail_legend=True
    for line_ in leg.get_lines():
        line_.set_linewidth(3)
    ax.set_xlim(w.index[0], w.index[-1])
    ax.set_ylabel("온도[℃]" if kind == "TC" else "제어출력[%]", labelpad=8)
    style(ax, "실제 측정 시각[시:분]")
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=5, maxticks=9))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.yaxis.set_major_locator(MaxNLocator(5))
    ax.tick_params(axis="both", labelsize=12.5)
    return w

def case_points(before, after, after_window):
    """수리 전후를 비교할 볼 구간. (a) 온도만 하강 (b) 온도·출력 함께 하강 (c) 승온 (d) Z3만 Z4~Z6에서 3℃ 넘게 벗어남."""
    rb, ra = lot_meta.loc[before], lot_meta.loc[after]
    warm = float(state_summary.set_index("LOT").loc[after, "WARM_END_MIN"])
    mid, ops = ["TC-Z4", "TC-Z5", "TC-Z6"], [f"OP-Z{z}" for z in ZONES]
    pb, pa = profile[before], profile[after]
    sb = states[before].eq("안정 참고")
    sa = states[after].eq("안정 참고") & (pa.index <= after_window[1])
    level = lambda p, m: float(p.loc[m, [(c, "median") for c in mid]].mean(axis=1).median())
    level_b = level(pb, sb)
    # 온도 하강 시작: Z4~Z6 평균이 수리 전 안정 수준 −1℃ 위에 있던 마지막 분의 다음 분 (그 뒤로는 계속 아래)
    heat = pb[[(c, "median") for c in mid]].mean(axis=1)
    drop = int(heat.index[heat.ge(level_b - 1)].max()) + 1
    # 출력 하강 시작: 그 뒤 Z4~Z6 출력 중앙값이 안정 수준보다 10%p 이상 낮은 상태가 3분 이어진 첫 분
    op_mid = pb[[(c, "median") for c in ops[1:]]].median(axis=1)
    low = ((op_mid - op_mid[sb].median()).le(-10) & (pb.index >= drop)).rolling(3).sum().eq(3)
    op_drop = int(low.idxmax()) - 2
    w = raw_window(after, *after_window)
    gap = (w["TC-Z3"] - w[mid].mean(axis=1)).dropna()       # 기록 없는 초는 빼고 본다
    z3 = gap.abs().idxmax()
    off = gap.abs().ge(3); block = off.ne(off.shift()).cumsum()       # 가장 큰 이탈을 포함해 3℃ 넘게 벗어나 있던 연속 구간
    d0, d1 = gap.index[block.eq(block[z3])][[0, -1]]
    ta, tb, tc = (rb.START + pd.Timedelta(minutes=drop), rb.START + pd.Timedelta(minutes=op_drop),
                  ra.START + pd.Timedelta(minutes=warm))
    m = lambda x0, x1: f"{(x1 - x0).total_seconds() / 60:.0f}분"
    return {"a": (ta, tb, f"{ta:%H:%M}–{tb:%H:%M} · {m(ta, tb)}\n온도만 하강 · 출력 유지"),
            "b": (tb, rb.END, f"{tb:%H:%M}–{rb.END:%H:%M} · {m(tb, rb.END)}\n온도·출력 함께 하강(정지)"),
            "c": (ra.START, tc, f"{ra.START:%H:%M}–{tc:%H:%M} · {m(ra.START, tc)}\n승온 종료 · 안정 참고 진입"),
            "d": (d0, d1, f"{d0:%H:%M}–{d1:%H:%M} · Z3 단독 반응\nZ4~Z6 대비 최대 {gap[z3]:+.1f}℃"), "z3_time": z3, "z3_gap": float(gap[z3]),
            "a_min": drop, "level_before": level_b, "level_after": level(pa, sa), "z3_value": float(w.loc[z3, "TC-Z3"]),
            "op_before": pb.loc[sb, [(c, "median") for c in ops]].median().droplevel(1),
            "op_after": pa.loc[sa, [(c, "median") for c in ops]].median().droplevel(1)}

def case_figure(plan, before, after, after_window, title, extra=None, more_handles=()):
    rb, ra = lot_meta.loc[before], lot_meta.loc[after]
    pts = case_points(before, after, after_window)
    b_win = (max(0, pts["a_min"] - 30), rb.MINUTES)     # 수리 전: 온도 하강 30분 전부터 관측 끝까지
    lo, hi = rb.START - pd.Timedelta(hours=6), ra.END + pd.Timedelta(hours=6)
    span = hi - lo
    ev = grade_evt.loc[grade_evt.EVT_DT.between(lo, hi)]
    tasks = cal_work.loc[((cal_work.STRT_DT < hi) & (cal_work.END_DT > lo)) | cal_work.DETC_DT.between(lo, hi)].sort_values("STRT_DT")

    fig = plt.figure(figsize=(30, 30))
    gs = fig.add_gridspec(3, 2, height_ratios=[1.9, 1, 1], left=.1, right=.975, top=.87, bottom=.04, hspace=.46, wspace=.14)
    axa = fig.add_subplot(gs[0, :])
    rows = ["강종 구간", "온도 관측"] + [f"{t.MNT_TYPE}\n{t.WO_NO}" for t in tasks.itertuples()]
    axa.axvspan(plan.STRT_DT, plan.END_EXCL, color=COL["PLAN"], alpha=.45, lw=0)
    # 강종 구간: 변경 시각부터 다음 변경까지 강종 색 띠, 띠 위 상자는 변경 시각
    for a, b_, g in grade_periods(lo, hi):
        axa.plot([a, b_], [0, 0], color=GRADE_COLOR.get(g, "#888888"), lw=30, solid_capstyle="butt", alpha=.9)
        if b_ - a > span * .045:
            axa.text(a + (b_ - a) / 2, 0, g, ha="center", va="center", fontsize=15, color="white", fontweight="bold")
    # 강종변경 상자: 날짜·시각을 윗줄, 바뀐 강종을 아랫줄에 적는다.
    for e in ev.itertuples():
        axa.plot([e.EVT_DT, e.EVT_DT], [-.3, -.6], color="#17344d", lw=1)
        axa.text(e.EVT_DT, -.6, f"{e.EVT_DT:%m/%d %H:%M}\n{e.BEF_VAL}→{e.AFT_VAL}", ha="center", va="bottom", fontsize=13,
                 color="#17344d", linespacing=1.3, bbox=dict(boxstyle="round,pad=.3", fc="white", ec="#17344d", lw=.7))
    for lot in [before, after]:
        r = lot_meta.loc[lot]
        axa.plot([r.START, r.END], [1, 1], color=LOT_COLOR[lot], lw=11, solid_capstyle="butt", ls=report_viz.lot_linestyle(lot))
    for i, t in enumerate(tasks.itertuples(), start=2):
        axa.plot([t.STRT_DT, t.END_DT], [i, i], color=COL[t.MNT_TYPE], lw=9, solid_capstyle="butt")
        axa.plot(t.DETC_DT, i, "v", color=COL["DETECT"], ms=11)
        if part_text(t.WO_NO):
            axa.plot(t.END_DT, i, "D", color=COL["PART"], ms=11)
        right = t.END_DT < lo + span * .55
        x = t.END_DT + span * .008 if right else t.DETC_DT - span * .008
        axa.text(x, i, work_label(t), ha="left" if right else "right", va="center", fontsize=13,
                 color=COL["PART"] if part_text(t.WO_NO) else COL[t.MNT_TYPE],
                 fontweight="bold" if part_text(t.WO_NO) else "normal",
                 bbox=dict(boxstyle="round,pad=.25", fc="white", ec=COL["PART"] if part_text(t.WO_NO) else COL[t.MNT_TYPE], lw=.7))
    axa.set_yticks(range(len(rows)), rows)
    for tick, color in zip(axa.get_yticklabels(), ["#17344d", COL["LOT"]] + [COL[t] for t in tasks.MNT_TYPE]):
        tick.set_color(color)
    axa.set_ylim(len(rows) - .4, -2.1)
    axa.set_xlim(lo, hi)
    style(axa, "2024년 실제 기록 시각[월/일]")
    axa.xaxis.set_major_locator(mdates.DayLocator())
    axa.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
    axa.tick_params(axis="both", labelsize=13.5)
    axa.set_title(f"(1) 전체 사건 흐름 · 계획수리 {plan.STRT_DT:%m/%d}–{plan.END_DT:%m/%d}({plan.SCOPE}) · CAL01 실제 작업 {len(tasks)} WO",
                  loc="left", fontsize=16, pad=14)
    # 빨간 상자는 아래 확대 패널 번호 (2)·(3)의 위치다.
    for key, lot, (a, b_) in [("2", before, b_win), ("3", after, after_window)]:
        r = lot_meta.loc[lot]
        x0, x1 = r.START + pd.Timedelta(minutes=a), r.START + pd.Timedelta(minutes=b_)
        pad = span * .004
        axa.add_patch(Rectangle((mdates.date2num(x0 - pad), .55), mdates.date2num(x1 + pad) - mdates.date2num(x0 - pad), .9,
                                fill=False, ec=COL["FOCUS"], lw=2.2, zorder=6))
        axa.text(x0 + (x1 - x0) / 2, 1.62, f"({key})", ha="center", va="top", fontsize=15, fontweight="bold", color=COL["FOCUS"])
    # 왼쪽 열 = 수리 전, 오른쪽 열 = 수리 후. 위는 온도, 아래는 출력이며 같은 신호는 세로축을 맞춘다.
    bt, ct = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])
    bo, co = fig.add_subplot(gs[2, 0]), fig.add_subplot(gs[2, 1])
    ct.sharey(bt); co.sharey(bo)
    label = "전체 관측" if after_window[1] >= ra.MINUTES else "초기"
    bw = zone_panel(bt, before, *b_win, "TC", f"(2) 수리 전 · {before} · {rb.GRADE}", date_heading=True)
    zone_panel(bo, before, *b_win, "OP", "가열·균열 제어 출력 · OP-Z3~Z6")
    aw = zone_panel(ct, after, *after_window, "TC", f"(3) 수리 후 · {after} · {ra.GRADE}", date_heading=True)
    zone_panel(co, after, *after_window, "OP", "가열·균열 제어 출력 · OP-Z3~Z6")
    for top, bottom in [(bt, bo), (ct, co)]:
        top.set_xlabel("")
        top.tick_params(axis="x", labelbottom=False)
        bottom.sharex(top)
    # Maintenance intervals stay in panel (1); do not duplicate rails over the raw plots.
    # Short letter markers sit at the observed values. No range-wide leader lines.
    point_specs = [("a", bt, bo, bw, pts["a"][0]), ("b", bt, bo, bw, pts["b"][0]),
                   ("c", ct, co, aw, pts["c"][1]), ("d", ct, co, aw, pts["z3_time"])]
    for key, top, bottom, values, at in point_specs:
        nearest = values.index.get_indexer([at], method="nearest")[0]
        row = values.iloc[nearest]
        for ax, kind in [(top, "TC"), (bottom, "OP")]:
            value = float(row[f"{kind}-Z3"]) if key == "d" else float(row[f"{kind}-Z4"])
            if not np.isfinite(value):
                continue
            ax.plot(at, value, "o", ms=7, mfc="white", mec=COL["FOCUS"], mew=2, zorder=8)
            # Labels stay beside their points, with a compact white background.
            up = value < ax.get_ylim()[0] + .78 * np.diff(ax.get_ylim())[0]
            dx = -9 if at > values.index[0] + (values.index[-1]-values.index[0])*.88 else 9
            ax.annotate(f"({key})", (at, value), xytext=(dx, 8 if up else -8),
                        textcoords="offset points", ha="right" if dx < 0 else "left",
                        va="bottom" if up else "top", fontsize=13, fontweight="bold",
                        color=COL["FOCUS"], zorder=9,
                        bbox=dict(boxstyle="round,pad=.18", fc="white", ec="none", alpha=.95))
    # Highlight only the local Z3 response; neighboring zones remain fully visible.
    for ax in [ct, co]:
        ax.axvspan(pts["d"][0], pts["d"][1], color=COL["FOCUS"], alpha=.15, lw=0, zorder=0)
    if extra is not None:
        extra(ct, co)
    handles = (lot_handles([before, after]) +
               [Patch(facecolor=COL["PLAN"], alpha=.6, label="CR1 계획수리 날짜"),
                *[Line2D([0], [0], color=COL[k], lw=6, label=f"{k} 실제 작업") for k in tasks.MNT_TYPE.drop_duplicates()],
                Line2D([0], [0], color=COL["DETECT"], marker="v", ls="None", ms=10, label="DETC_DT 감지 · 방법 미제공"),
                Line2D([0], [0], color=COL["PART"], marker="D", ls="None", ms=10, label="부품교체 · WO 완료 기준"),
                Patch(facecolor="white", edgecolor=COL["FOCUS"], lw=2, label="(1)의 확대 위치 → (2) 수리 전 · (3) 수리 후"),
                Line2D([0], [0], marker="o", mfc="white", mec=COL["FOCUS"], mew=2, ls="None", label="(a)~(d) 확인 지점 · 설명은 아래"), *more_handles])
    header(fig, title, f"왼쪽 (2) 수리 전 {before}({rb.GRADE}) 종료부 · 오른쪽 (3) 수리 후 {after}({ra.GRADE}) · (a)~(d)는 변화가 갈리는 지점 · 추가 확대는 뒤에서 확인",
           handles, ncol=6)
    compact_header(fig, .956, .936)
    axis_tag(fig, TAG_REAL)
    fig._caption_context = {"kind": "repair", "before": before, "after": after, "pts": pts}
    return fig, tasks, ev, b_win

def detect_data(tasks):
    """각 감지 시각 ±10분의 온도 기록 행 수."""
    return {t.WO_NO: int(temp.MEAS_DT.between(t.DETC_DT - pd.Timedelta(minutes=10), t.DETC_DT + pd.Timedelta(minutes=10)).sum())
            for t in tasks.itertuples()}

def work_table(tasks):
    near = detect_data(tasks)
    return pd.DataFrame([{"WO": t.WO_NO, "종류": t.MNT_TYPE, "DETC_DT": t.DETC_DT, "감지 ±10분 온도[행]": near[t.WO_NO],
                          "작업 시작": t.STRT_DT, "작업 완료": t.END_DT, "작업 기록 길이[분]": (t.END_DT - t.STRT_DT).total_seconds() / 60,
                          "DOWN_MIN 기록[분]": t.DOWN_MIN, "FAIL_MODE 코드": t.FAIL_MODE, "REMARK 원문": t.REMARK,
                          "교체 부품·수량·사유": part_text(t.WO_NO) or "-"} for t in tasks.itertuples()])

In [ ]:
# CAL01 정비 경계와 실제 온도 기록의 교집합을 먼저 확인한다.
EVENT_PAD_MIN = 25
NEAREST_OBSERVATION_MIN = 30
_sensor_times = temp.MEAS_DT.sort_values().reset_index(drop=True)
EVENT_SPECS = [("DETC_DT", "감지", ":"), ("STRT_DT", "착수", "--"), ("END_DT", "완료", "-.")]
ZOOM_GROUPS = [
    ("가열 초입 Z3 · 온도", ["TC-Z3"]), ("가열 초입 Z3 · 출력", ["OP-Z3"]),
    ("가열·균열 Z4~Z6 · 온도", [f"TC-Z{z}" for z in [4,5,6]]),
    ("가열·균열 Z4~Z6 · 출력", [f"OP-Z{z}" for z in [4,5,6]]),
    ("CP-Z4", ["CP-Z4"]), ("CP-Z4M", ["CP-Z4M"]),
]


def measured_rows(start, end):
    return int(_sensor_times.searchsorted(end, side="right") - _sensor_times.searchsorted(start, side="left"))


def zoom_parts(wo):
    records = cal_parts.loc[cal_parts.WO_NO.eq(wo)]
    return [f"{r.PART_NM} [{r.PART_CD}] ×{r.QTY:g} ({r.CHG_RSN})" for r in records.itertuples()]


maintenance_zoom_audit = []
for task in cal_work.itertuples():
    row = {"WO": task.WO_NO, "유형": task.MNT_TYPE, "작업 내용": remark_text(task)}
    for field, label, _ in EVENT_SPECS:
        at = getattr(task, field)
        row[label + " 시각"] = at
        row[label + " 시각 측정 기록"] = bool(measured_rows(at, at))
        row[label + " ±25분 기록[행]"] = measured_rows(at-pd.Timedelta(minutes=EVENT_PAD_MIN), at+pd.Timedelta(minutes=EVENT_PAD_MIN))
    row["작업 중 관측 겹침[분]"] = sum(
        max(0, (min(task.END_DT, lot.END)-max(task.STRT_DT, lot.START)).total_seconds()/60)
        for lot in lots.itertuples())
    row["기록된 교체 부품"] = "; ".join(zoom_parts(task.WO_NO))
    maintenance_zoom_audit.append(row)
maintenance_zoom_audit = pd.DataFrame(maintenance_zoom_audit)
observed_event_wos = maintenance_zoom_audit.loc[
    maintenance_zoom_audit[[name+" ±25분 기록[행]" for _,name,_ in EVENT_SPECS]].gt(0).any(axis=1), "WO"]


def merge_event_windows(events, pad_min):
    """맞닿거나 겹치는 검토창은 한 번만 그린다. 떨어진 시각은 별도 창을 유지한다."""
    spans = sorted((r["time"]-pd.Timedelta(minutes=pad_min), r["time"]+pd.Timedelta(minutes=pad_min)) for r in events)
    merged = []
    for left,right in spans:
        if merged and left <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(right,merged[-1][1]))
        else:
            merged.append((left,right))
    return merged


def measurement_gap_spans(grid, present):
    """음영은 양쪽 관측점까지 연결하고, 공백 수는 실제 빠진 예상 시각으로 센다."""
    missing = np.flatnonzero(~present)
    if not len(missing):
        return []
    groups = np.split(missing, np.flatnonzero(np.diff(missing)>1)+1)
    return [(grid[max(0,int(a[0])-1)], grid[min(len(grid)-1,int(a[-1])+1)], len(a)) for a in groups]


maintenance_zoom_windows = []
maintenance_zoom_checks = []


def maintenance_zoom(wo_numbers, name, title, pad_min=EVENT_PAD_MIN,
                     nearest_minutes=NEAREST_OBSERVATION_MIN, groups=ZOOM_GROUPS):
    """정비 경계의 1초 원본. 경계에 기록이 없으면 날짜상 직전 제공 관측을 별도로 표시한다."""
    tasks = cal_work.loc[cal_work.WO_NO.isin(wo_numbers)].sort_values("STRT_DT")
    assert set(tasks.WO_NO)==set(wo_numbers), "CAL01에 없는 WO"
    events = [{"WO":t.WO_NO,"type":t.MNT_TYPE,"label":label,"style":style_,"time":getattr(t,field)}
              for t in tasks.itertuples() for field,label,style_ in EVENT_SPECS]
    requested = merge_event_windows(events,pad_min)
    windows = [(a,b,"정비 경계") for a,b in requested if measured_rows(a,b)>0]
    fallback = not windows
    if fallback:
        first_event = min(e["time"] for e in events)
        prior = lots.loc[lots.END.lt(first_event)].sort_values("END").tail(1)
        assert len(prior), "직전 제공 LOT도 없어 확대할 센서 자료가 없음"
        nearest = prior.iloc[0]
        gap = (first_event-nearest.END).total_seconds()/3600
        assert gap <= 72, "직전 제공 관측도 72시간 밖: 작업 시각만 이력 표에서 확인"
        windows = [(max(nearest.START,nearest.END-pd.Timedelta(minutes=nearest_minutes)),nearest.END,"직전 제공 관측")]

    # 각 창은 실제 날짜·시각을 보존한다. 다른 LOT의 선을 서로 이어 붙이지 않는다.
    window_lots = []
    for left,right,kind in windows:
        observed = lots.loc[lots.START.le(right)&lots.END.ge(left)].sort_values("START")
        assert len(observed), "관측이 없는 창은 원본 확대에 넣지 않음"
        window_lots.append(observed)
    shown_lots = pd.concat(window_lots).drop_duplicates("LOT_NO").sort_values("START")
    ncols = len(windows)
    width = max(24,3+9*ncols)
    fig = plt.figure(figsize=(width,6+3.5*len(groups)))
    grid_spec = fig.add_gridspec(len(groups)+1,ncols,
        height_ratios=[1.8]+[1]*len(groups),left=.14,right=.975,top=.85,bottom=.05,
        wspace=.25,hspace=1.05)
    timeline = fig.add_subplot(grid_spec[0,:])
    timeline_labels = []
    lane = 0
    for lot in shown_lots.itertuples():
        timeline.plot([lot.START,lot.END],[lane,lane],color=GRADE_COLOR[lot.GRADE],lw=9,solid_capstyle="butt")
        timeline_labels.append(f"온도 {lot.LOT_NO} · {lot.GRADE}")
        lane += 1
    task_lanes = {}
    for task in tasks.itertuples():
        task_lanes[task.WO_NO] = lane
        timeline.plot([task.STRT_DT,task.END_DT],[lane,lane],color=COL[task.MNT_TYPE],lw=8,solid_capstyle="butt")
        timeline.scatter(task.DETC_DT,lane,marker="v",s=75,color=COL["DETECT"],zorder=5)
        timeline.scatter([task.STRT_DT,task.END_DT],[lane,lane],marker="|",s=170,color=COL[task.MNT_TYPE],zorder=5)
        timeline_labels.append(f"{task.MNT_TYPE} · {task.WO_NO}\n{remark_text(task)}")
        lane += 1
        for part_record in zoom_parts(task.WO_NO):
            timeline.scatter(task.END_DT,lane,marker="D",s=65,color=COL["PART"],zorder=5)
            timeline_labels.append("교체 기록\n" + part_record)
            lane += 1
    # 본창 이전 감지는 시각을 옮기지 않고 아래 이력 표에 보존한다.
    # 직전 관측 확대에서는 해당 30분과 BM의 사건 순서에 집중한다.
    lo = min(min(a for a,_,_ in windows),tasks.STRT_DT.min())
    if not fallback:
        lo = min(lo,shown_lots.START.min())
    hi = max(max(e["time"] for e in events),max(b for _,b,_ in windows),shown_lots.END.max())
    margin = max((hi-lo)*.025,pd.Timedelta(minutes=5))
    timeline.set_xlim(lo-margin,hi+margin)
    timeline.set_ylim(lane-.45,-.85)
    timeline.set_yticks(range(lane),timeline_labels)
    timeline.tick_params(axis="both",labelsize=12.5)
    style(timeline,"실제 기록 날짜·시각 · 음영 (2)~: 아래 원본 확대 범위")
    timeline.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4,maxticks=7))
    timeline.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d %H:%M"))
    timeline.set_title("(1) 관측·감지·실제 작업·부품교체 이력",loc="left",fontsize=16,pad=15)
    # 그래프 안에서 볼 지점: 확대 창에 들어온 정비 경계를 시간 순서로 (a)(b)…라 부른다.
    shown = sorted([e for e in events if any(a<=e["time"]<=b for a,b,_ in windows)], key=lambda e:e["time"])
    mark = {id(e):chr(97+i) for i,e in enumerate(shown)}

    for column,((left,right,kind),observed) in enumerate(zip(windows,window_lots)):
        panel = f"({column+2})"
        timeline.axvspan(left,right,color=COL["FOCUS"],alpha=.09,lw=0)
        timeline.text(left+(right-left)/2,-.55,panel,ha="center",va="center",
                      fontsize=15,fontweight="bold",color=COL["FOCUS"])
        times = pd.date_range(left,right,freq="s")
        subset = temp.loc[temp.MEAS_DT.between(left,right)].set_index("MEAS_DT")
        assert subset.index.is_unique
        measured = subset.reindex(times)
        gaps = measurement_gap_spans(times,times.isin(subset.index))
        primary_lot = int(observed.iloc[0].LOT_NO)
        in_window = [e for e in events if left<=e["time"]<=right]
        axes = []
        for row,(group_name,sensors) in enumerate(groups):
            ax = fig.add_subplot(grid_spec[row+1,column],sharex=axes[0] if axes else None)
            axes.append(ax)
            for lot in observed.itertuples():
                valid_lot = measured.LOT_NO.eq(lot.LOT_NO)
                for sensor in sensors:
                    color = ZONE_COLOR[int(sensor.rsplit("Z",1)[1])] if sensor.startswith(("TC-Z","OP-Z")) and int(sensor.rsplit("Z",1)[1]) in ZONE_COLOR else GRADE_COLOR[lot.GRADE]
                    y = measured[sensor].where(valid_lot) * (1e10 if sensor=="CP-Z4M" else 1)
                    ax.plot(times,y,color=color,lw=.95 if len(sensors)>1 else 1.3,label=sensor)
            for start,end,missing_seconds in gaps:
                ax.axvspan(start,end,color="#d8dfe5",alpha=.8,lw=0,zorder=0)
                if row==0 and missing_seconds>=300:
                    ax.text(start+(end-start)/2,.12,"측정 기록 없음",transform=ax.get_xaxis_transform(),
                            ha="center",va="center",fontsize=11,color="#526d80")
            for j,event in enumerate(sorted(in_window,key=lambda e:e["time"])):
                color = COL["DETECT"] if event["label"]=="감지" else COL[event["type"]]
                ax.axvline(event["time"],color=color,ls=event["style"],lw=1.6,zorder=4)
                if row==0:
                    detail_tag(ax,event["time"],f'({mark[id(event)]})',level=1.28+.13*(j%2),color=color)
            xlabel = f"{left:%m/%d} 실제 측정 시각[시:분]" if row==len(groups)-1 else ""
            style(ax,xlabel)
            ax.set_xlim(left,right)
            ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=5))
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
            ax.tick_params(axis="both",labelsize=12,labelbottom=row==len(groups)-1)
            ax.yaxis.set_major_locator(MaxNLocator(4))
            ax.set_ylabel("온도[℃]" if sensors[0] in TC else "제어출력[%]" if sensors[0] in OP else unit(sensors[0]),fontsize=13,labelpad=8)
            if sensors==["CP-Z4M"]:ax.set_ylabel("CP-Z4M (단위 미확인·10⁻¹⁰)")
            heading=f"{panel} {primary_lot} · {kind} · {left:%m/%d %H:%M}–{right:%H:%M}" if row==0 else None
            detail_legend(ax,group_name,heading=heading)
        count = measured_rows(left,right)
        maintenance_zoom_windows.append({
            "그림":name,"창":panel,"종류":kind,"LOT":", ".join(str(int(k)) for k in observed.LOT_NO),
            "확대 시작":left,"확대 종료":right,"측정 기록[행]":count,
            "창 안의 정비 경계":" / ".join(f"({mark[id(e)]}) {e['type']} {e['WO'].rsplit('-',1)[-1]} {e['label']} {e['time']:%m/%d %H:%M}" for e in in_window)})
        assert all(ax.get_xlim()==axes[0].get_xlim() for ax in axes)

    handles = []
    handles += [Line2D([0],[0],color=COL["DETECT"],ls=":",lw=1.7,label="DETC_DT 감지 기록"),
                Line2D([0],[0],color="#455a64",ls="--",lw=1.7,label="작업 착수"),
                Line2D([0],[0],color="#455a64",ls="-.",lw=1.7,label="작업 완료"),
                Patch(facecolor="#d8dfe5",label="측정 기록 공백"),
                Patch(facecolor=COL["FOCUS"],alpha=.25,label="원본 확대 시간창")]
    if any(zoom_parts(wo) for wo in tasks.WO_NO):
        handles.append(Line2D([0],[0],color=COL["PART"],marker="D",ls="None",label="교체 이력 · WO 완료 위치"))
    subtitle = (f"정비 경계 ±{pad_min}분: 온도 기록 없음 · 아래는 날짜상 직전 제공 LOT의 마지막 {nearest_minutes}분"
                if fallback else f"정비 경계 ±{pad_min}분 중 기록이 있는 시간창 · 겹치는 창은 합침 · 선: 1초 원본 · 실제 날짜·시각 유지")
    header(fig,title,subtitle,handles,ncol=5)
    compact_header(fig,.948,.916,title_size=24,sub_size=13,legend_size=12.5)
    axis_tag(fig,TAG_REAL)
    fig._caption_context = {'kind':'zoom','tasks':tasks,'windows':windows,'fallback':fallback}
    finish(fig,name)
    maintenance_zoom_checks.append({"FILE":name,"MODE":"직전 관측" if fallback else "정비 경계",
                                    "N_WINDOWS":len(windows),"WO":list(tasks.WO_NO),
                                    "ORIGINAL_CLOCK":True,"NO_INTERPOLATION":True})
    rows = maintenance_zoom_audit.loc[maintenance_zoom_audit.WO.isin(tasks.WO_NO)]
    table(rows)
    table(pd.DataFrame(maintenance_zoom_windows).loc[lambda d:d.그림.eq(name)])
    for task in tasks.itertuples():
        note(f"**작업 내용:** {task.WO_NO} · {task.MNT_TYPE} · {remark_text(task)}. "
             f"감지 {task.DETC_DT:%m/%d %H:%M} → 착수 {task.STRT_DT:%m/%d %H:%M} → 완료 {task.END_DT:%m/%d %H:%M}.")
        if zoom_parts(task.WO_NO):
            note("**같은 WO의 교체 이력:** " + "; ".join(zoom_parts(task.WO_NO)) + ". 다이아몬드 위치는 WO 완료 시각이며 실제 교체 실행 시각은 제공되지 않았다.")
    if fallback:
        task = tasks.iloc[0]
        last = shown_lots.iloc[-1]
        note(f"**이 확대의 범위:** {int(last.LOT_NO)}의 마지막 관측은 {last.END:%m/%d %H:%M:%S}이며 "
             f"감지까지 {(task.DETC_DT-last.END).total_seconds()/60:.2f}분, 착수까지 {(task.STRT_DT-last.END).total_seconds()/60:.2f}분 떨어져 있다. "
             "직전 제공 관측의 형태만 확인할 수 있고, 실제 BM 시점의 온도나 정비 뒤 회복을 이 그림으로 확인할 수 없다.")
    return windows

In [ ]:
def work_boundary_effects(k):
    tasks=linked_work(k); r=lot_meta.loc[k]; result=[]
    if 'RAW_T' not in globals():
        source=raw[k].set_index('MEAS_DT')
    else:
        source=RAW_T[k]
    sensors=['CP-Z4','CP-Z4M',*[f'{kind}-Z{z}' for kind in ['TC','OP'] for z in [3,4,5,6]]]
    for t in tasks.itertuples():
        for field,label in [('DETC_DT','감지'),('STRT_DT','착수'),('END_DT','완료')]:
            at=getattr(t,field)
            other=tasks.loc[tasks.WO_NO.ne(t.WO_NO)&tasks.STRT_DT.lt(at+pd.Timedelta(minutes=10))&tasks.END_DT.gt(at-pd.Timedelta(minutes=10))]
            for s in sensors:
                intervals=[(at-pd.Timedelta(minutes=10),at),(at,at+pd.Timedelta(minutes=10)),
                           (at+pd.Timedelta(minutes=10),at+pd.Timedelta(minutes=30))]
                values=[source.loc[(source.index>=a)&(source.index<b),s].dropna() for a,b in intervals]
                n=[len(v) for v in values]
                med=[v.median() if len(v)>=.8*(b-a).total_seconds() else np.nan for v,(a,b) in zip(values,intervals)]
                delayed=values[2].resample('30s').median().where(values[2].resample('30s').count().ge(24))-med[0]
                peak=delayed.abs().idxmax() if delayed.notna().any() else pd.NaT
                state=states[k].loc[max(0,int((at-r.START).total_seconds()/60)-10):max(0,int((at-r.START).total_seconds()/60)+10)]
                result.append({'LOT':k,'WO':t.WO_NO,'작업':t.MNT_TYPE,'경계':label,'시각':at,'신호':s,
                    '전10분':med[0],'후0~10분':med[1],'후10~30분':med[2],
                    '직후 차이':med[1]-med[0],'지연 차이':med[2]-med[0],
                    '후10~30분 최대 변화':float(delayed.loc[peak]) if pd.notna(peak) else np.nan,
                    '최대 변화 시각':peak,
                    '전/직후/지연 기록 수':' / '.join(map(str,n)),
                    '겹친 다른 작업':' · '.join(other.MNT_TYPE) or '없음',
                    '운전 상태':' · '.join(state.unique()) if r.START<=at<=r.END else '관측 밖'})
    return pd.DataFrame(result)

def maintenance_detail_windows(k, windows, name, title, observation, check, boundary_points=()):
    rows=[('CP 기록 · CP-Z4',['CP-Z4']),('가열 초입 Z3 · 온도',['TC-Z3']),
          ('가열 초입 Z3 · 출력',['OP-Z3']),('가열·균열 Z4~Z6 · 온도',[f'TC-Z{z}' for z in [4,5,6]]),
          ('가열·균열 Z4~Z6 · 출력',[f'OP-Z{z}' for z in [4,5,6]])]
    fig=plt.figure(figsize=(28,23))
    gs=fig.add_gridspec(len(rows),len(windows),left=.075,right=.975,top=.83,bottom=.10,hspace=1.15,wspace=.18)
    by_row={i:[] for i in range(len(rows))};focus_notes=[]
    for col,(lo,hi,label) in enumerate(windows):
        values=raw[k].set_index('MEAS_DT').loc[lo:hi]
        gap=values['TC-Z3']-values[['TC-Z4','TC-Z5','TC-Z6']].mean(axis=1)
        z3_at=gap.abs().idxmax() if gap.notna().any() else None
        cp_delta=values['CP-Z4']-raw[k]['CP-Z4'].median()
        cp_at=cp_delta.abs().idxmax() if cp_delta.notna().any() else None
        show_z3=z3_at is not None and abs(gap.loc[z3_at])>=3 and '승온' not in label
        show_cp=cp_at is not None and abs(cp_delta.loc[cp_at])>=.05
        if show_z3:focus_notes.append(f'{label}: {z3_at:%H:%M:%S} Z3–Z4~Z6 {gap.loc[z3_at]:+.1f}℃')
        if show_cp:focus_notes.append(f'{label}: {cp_at:%H:%M:%S} CP {cp_delta.loc[cp_at]:+.3f}')
        for row,(title_,ss) in enumerate(rows):
            ax=fig.add_subplot(gs[row,col])
            by_row[row].append(ax)
            detail_raw(ax,k,ss,lo,hi,title_,bottom=row==len(rows)-1,
                       heading=f'{label}\n{lo:%Y/%m/%d %H:%M} → {hi:%H:%M}' if row==0 else None)
            at=z3_at if show_z3 and row in [1,2] else cp_at if show_cp and row==0 else None
            if at is not None:
                ax.plot(at,values.loc[at,ss[0]],'o',ms=8,mfc='white',mec=COL['FOCUS'],mew=2,zorder=7)
            for letter,at,label_ in boundary_points:
                if lo<=at<=hi:
                    ax.axvline(at,color='#7b8794',lw=1,ls='--',zorder=0)
                    # Boundary names have their own row, away from data and the legend.
                    if row==0:detail_tag(ax,at,f'({letter})',level=1.30,color='#526d80')
    if len(windows)>1:
        for axs in by_row.values():
            limits=[ax.get_ylim() for ax in axs];lo_=min(a for a,b in limits);hi_=max(b for a,b in limits)
            for ax in axs:ax.set_ylim(lo_,hi_)
    header(fig,title,f'{k} · {lot_meta.loc[k,"GRADE"]} · 1초 원본 · Z3와 Z4~Z6 별도 표시 · 세로선: 실제 경계',[])
    compact_header(fig,.955,.918,title_size=24,sub_size=13)
    axis_tag(fig,TAG_REAL)
    keys=' · '.join(f'({a}) {t:%H:%M} {label}' for a,t,label in boundary_points)
    fig._caption_context={'kind':'note','observation':observation+(' 빨간 원: '+' · '.join(focus_notes)+'.' if focus_notes else ''),
                          'check':check+(' 경계: '+keys if keys else '')+(' 같은 행의 세로축은 좌우 동일합니다.' if len(windows)>1 else '')}
    finish(fig,name)


def maintenance_extra_views(before,after,root_name):
    """Audit all catalog candidates and show representative previously hidden episodes."""
    to_time=lambda k,m:lot_meta.loc[k,'START']+pd.Timedelta(minutes=m)
    if before==240058:
        episodes=[(65,94,'(a) 1回目'.replace('1回目','첫 번째 Z3 반응')),(110,137,'(b) 두 번째 Z3 반응')]
        obs='수리 전 240058에서도 Z3 온도·출력의 큰 반응이 두 번 보입니다. 수리 후의 Z3 반응만 보고 새로 생긴 변화라고 판단하지 않습니다.'
    else:
        episodes=[(68,94,'(a) 수리 전 Z3 반복 반응'),(205,231,'(b) 수리 전 종료 하강 앞 반응')]
        obs='수리 전 240114에도 Z3 반응이 있습니다. 수리 후 240117은 이 반복 반응과 비교해 Z3 하강·회복 폭, 출력 끝값 지속, CP 동반 여부를 확인합니다.'
    maintenance_detail_windows(before,[(to_time(before,a),to_time(before,b),label) for a,b,label in episodes],
        root_name+'/03_수리전_Z3반복반응_추가확대.png',f'{before} · 수리 전에도 있었던 Z3 반응',obs,
        '큰 Z3 반응은 가열 초입의 자체 반응과 함께 읽습니다. 같은 시간의 Z4~Z6와 CP가 동반하는지를 아래위로 비교합니다.')
    # The post-repair Z3 episode should have enough space, rather than remain a mark on a multi-hour overview.
    if after==240061:
        lo,hi=pd.Timestamp('2024-06-15 03:02'),pd.Timestamp('2024-06-15 03:32')
    else:
        lo,hi=pd.Timestamp('2024-11-13 12:45'),pd.Timestamp('2024-11-13 13:17')
    maintenance_detail_windows(after,[(lo,hi,'수리 후 Z3 단독 반응과 CP')],
        root_name+'/04_수리후_Z3_CP_추가확대.png',f'{after} · Z3와 CP를 같은 시각으로 확대',
        'Z3 출력이 낮은데 온도가 올라가는 시간과 CP의 변화를 분리해 확인합니다. Z4~Z6는 같은 축 묶음으로 별도 표시합니다.',
        '240061은 작업 없는 구간의 Z3 반응이고, 240117은 CBM 완료 뒤 관측입니다. 반응의 유무와 정비 원인은 구분합니다.')
    # Drying response is not buried among heating-zone plots.
    lo,hi=((pd.Timestamp('2024-06-15 04:27'),pd.Timestamp('2024-06-15 04:44')) if after==240061 else
           (pd.Timestamp('2024-11-13 14:06'),pd.Timestamp('2024-11-13 14:26')))
    fig,axes=plt.subplots(2,1,figsize=(27,12))
    fig.subplots_adjust(left=.08,right=.975,top=.76,bottom=.12,hspace=.85)
    for row,(kind,title_) in enumerate([('TC','건조 Z1~Z2 · 온도'),('OP','건조 Z1~Z2 · 출력')]):
        detail_raw(axes[row],after,[f'{kind}-Z1',f'{kind}-Z2'],lo,hi,title_,bottom=row==1)
    header(fig,f'{after} · 후반 건조 출력의 별도 확인',f'{lo:%Y/%m/%d %H:%M} → {hi:%H:%M} · 가열·균열과 다른 공정은 별도 그림으로 확인')
    compact_header(fig,.905,.875,title_size=24,sub_size=13)
    axis_tag(fig,TAG_REAL)
    fig._caption_context={'kind':'note','observation':'전체 흐름에서는 잘 보이지 않던 관측 후반의 건조 출력 변화를 확대했습니다. 위 온도와 아래 출력을 같은 실제 시각으로 비교합니다.',
                          'check':'짧은 출력 변화가 온도에도 남는지 확인합니다. 이 반응을 가열·균열 정비의 개선 또는 악화로 연결하지 않습니다.'}
    finish(fig,root_name+'/05_수리후_건조출력_추가확대.png')
    audit=[]
    for k in [before,after]:
        for e in rv_catalog(k):
            if e['family'] in ['peer','level']:continue
            audit.append({'LOT':k,'분류':e['family'],'시작':to_time(k,e['a']),'끝':to_time(k,e['b']),
                          '확인 신호':' · '.join(e['sensors']),
                          '확인 위치':'전체 흐름 + 전후 확대' if e['family']=='transition' else
                                      'Z3/CP 추가 확대 및 05의 같은 LOT 공정 반응' if e['family'] in ['thermal','cp'] else
                                      '건조 확대 / 05의 같은 LOT 공정 반응'})
    note('**추가 확인 목록:** (a)~(d)는 전후 흐름을 읽는 대표 지점입니다. 아래는 전체 선택 목록이며, 짧은 단일 출력 변화도 남겨두었습니다. 크거나 반복되는 Z3, CP, 건조 변화를 추가 그림으로 확대합니다.')
    table(pd.DataFrame(audit))
    return pd.DataFrame(audit)


def maintenance_stages(k,name):
    r=lot_meta.loc[k];tasks=linked_work(k)
    if not len(tasks):return
    cuts=sorted({r.START,r.END+pd.Timedelta(seconds=1),*[max(r.START,t.STRT_DT) for t in tasks.itertuples()],
                 *[min(r.END+pd.Timedelta(seconds=1),t.END_DT) for t in tasks.itertuples()]})
    stages=[]
    for a,b in zip(cuts[:-1],cuts[1:]):
        active=tasks.loc[tasks.STRT_DT.lt(b)&tasks.END_DT.gt(a)]
        label='·'.join(active.MNT_TYPE) if len(active) else '작업 전' if b<=tasks.STRT_DT.min() else '전체 작업 완료 후'
        stages.append({'A':a,'B':b,'NAME':label})
    rows=[('CP 기록',['CP-Z4']),('가열 초입 Z3 · 온도',['TC-Z3']),('가열 초입 Z3 · 출력',['OP-Z3']),
          ('가열·균열 Z4~Z6 · 온도',[f'TC-Z{z}' for z in [4,5,6]]),
          ('가열·균열 Z4~Z6 · 출력',[f'OP-Z{z}' for z in [4,5,6]]),('CP 보조 기록',['CP-Z4M'])]
    fig=plt.figure(figsize=(28,28));gs=fig.add_gridspec(7,1,height_ratios=[.95]+[1]*6,
        left=.09,right=.975,top=.85,bottom=.08,hspace=.92)
    timeline=fig.add_subplot(gs[0,0]);timeline.plot([r.START,r.END],[0,0],color=LOT_COLOR[k],lw=7, ls=report_viz.lot_linestyle(k))
    timeline.set_yticks(range(len(tasks)+1),['온도 관측']+[f'{t.MNT_TYPE} · {t.WO_NO}' for t in tasks.itertuples()])
    timeline.set_ylim(len(tasks)+.5,-.7)
    boundary_times=sorted((getattr(t,field),f'{t.MNT_TYPE} '+('착수' if field=='STRT_DT' else '완료'))
                          for t in tasks.itertuples() for field in ['STRT_DT','END_DT'])
    points=[(chr(97+i),at,label) for i,(at,label) in enumerate(boundary_times)]
    for i,t in enumerate(tasks.itertuples(),1):
        timeline.plot([t.STRT_DT,t.END_DT],[i,i],color=COL[t.MNT_TYPE],lw=9,solid_capstyle='butt')
        for field in ['STRT_DT','END_DT']:
            at=getattr(t,field);letter=next(letter for letter,time_,label_ in points if time_==at)
            timeline.annotate(f'({letter}) {at:%H:%M}',(at,i),xytext=(0,-18),textcoords='offset points',ha='center',va='top',fontsize=11,color='#526d80')
    detail_clock(timeline,r.START,r.END,True);timeline.set_title('(1) 실제 작업 시간 · 경계 전후는 다음 그림에서 확대',loc='left',fontsize=16,pad=14)
    for row,(title_,ss) in enumerate(rows):
        ax=fig.add_subplot(gs[row+1,0]);detail_raw(ax,k,ss,r.START,r.END,f'({row+2}) {title_}',bottom=row==len(rows)-1)
        for letter,at,label in points:
            ax.axvline(at,color='#a2b0bb',lw=.8,ls='--',zorder=0)
    header(fig,f'{k} 정비 전·중·후 · 전체 흐름',
           '같은 LOT·실제 측정 시각 · 실제 작업은 맨 위 시간표에서 확인 · Z3와 Z4~Z6 별도 축 · 범례는 각 그래프 바로 위')
    compact_header(fig,.955,.923,title_size=24,sub_size=13);axis_tag(fig,TAG_REAL)
    fig._caption_context={'kind':'note','observation':'작업 전은 승온 중이고 TBM과 CBM은 일부 겹칩니다. 전 구간 중앙값 차이보다 다음의 경계별 동일 길이 비교를 먼저 확인합니다.',
        'check':'실제 착수·완료 앞뒤를 별도 확대합니다. 08:54 CBM 착수 때는 TBM이 진행 중이며, 12:44 완료 직후와 약 20분 뒤 CP 하강은 다른 변화입니다.'}
    finish(fig,name)
    # Keep full 22-signal statistics, and add a state-controlled view so warmup is not compared with a stable stage.
    stable_rows=[]
    for i,st in enumerate(stages):
        g=raw[k].loc[raw[k].MEAS_DT.ge(st['A'])&raw[k].MEAS_DT.lt(st['B'])]
        for s in SIGNALS:
            v=g[s].dropna();elapsed=(g.MEAS_DT-st['A']).dt.total_seconds()/60;ok=g[s].notna()
            slope=np.polyfit(elapsed[ok],g.loc[ok,s],1)[0] if ok.sum()>=60 else np.nan
            maintenance_stage_rows.append({'LOT':k,'단계':i,'구간':st['NAME'],'시작':st['A'],'끝':st['B'],
                '길이[분]':(st['B']-st['A']).total_seconds()/60,'센서':s,'유효 기록[개]':len(v),'중앙값':v.median(),
                'Q25':v.quantile(.25),'Q75':v.quantile(.75),'IQR':v.quantile(.75)-v.quantile(.25),
                '최소':v.min(),'최대':v.max(),'선형 변화율/분':slope})
            p=profile[k];minutes=p.index
            at=r.START+pd.to_timedelta(minutes,unit='min')
            mask=(at>=st['A'])&(at<st['B'])&states[k].reindex(minutes).eq('안정 참고').to_numpy()
            q=p.loc[mask,(s,'median')].where(quality[k].loc[minutes[mask],s].ge(.9)).dropna()
            stable_rows.append({'단계':i,'구간':st['NAME'],'신호':s,'안정 참고[분]':len(q),'중앙값':q.median(),
                                'IQR':q.quantile(.75)-q.quantile(.25),'비교 상태':'5분 미만 · 비교 보류' if len(q)<5 else '안정 참고만'})
    stat=pd.DataFrame(maintenance_stage_rows).loc[lambda d:d.LOT.eq(k)]
    stable_stat=pd.DataFrame(stable_rows)
    note('**승온을 제외한 단계 비교:** 안정 참고·유효 기록 90% 이상인 분만 비교합니다. 작업 전 단계에 안정 관측이 없으면 개선률을 계산하지 않습니다.')
    table(stable_stat.loc[stable_stat.신호.isin(['TC-Z3','TC-Z4','TC-Z5','TC-Z6','OP-Z3','OP-Z4','OP-Z5','OP-Z6','CP-Z4'])])
    boundaries=work_boundary_effects(k)
    folder=str(Path(name).parent)
    windows=[([(pd.Timestamp('2024-11-13 07:08'),pd.Timestamp('2024-11-13 08:45'),'TBM 착수와 승온')],
              '03_240117_TBM착수_승온_확대.png','TBM 착수 전부터 이미 승온 중입니다. 작업 전 관측이 짧은 감지 07:13은 전10분 비교가 부족합니다.'),
             ([(pd.Timestamp('2024-11-13 08:40'),pd.Timestamp('2024-11-13 09:55'),'CBM 착수 · TBM 완료')],
              '04_240117_CBM착수_TBM완료_확대.png','CBM 착수 08:54 전10분 대비 직후10분: Z3 온도 -2.39℃, 출력 +25.98%p. TBM이 겹쳐 한 작업의 효과로 분리할 수 없습니다.'),
             ([(pd.Timestamp('2024-11-13 12:34'),pd.Timestamp('2024-11-13 13:17'),'CBM 완료와 지연 변화')],
              '05_240117_CBM완료_Z3_CP_확대.png','CBM 완료 12:44 직후10분 CP 중앙값 차이는 -0.0013입니다. 약 11분 뒤 Z3 출력 0%, 약 21분 뒤 CP 최저 0.367로 늦은 변화가 나타납니다.')]
    for windows_,filename,obs in windows:
        maintenance_detail_windows(k,windows_,folder+'/'+filename,f'{k} · '+windows_[0][2],obs,
            '직후 0~10분과 후10~30분을 구분하고, 중앙값뿐 아니라 지연 극값 시각도 아래 표에서 확인합니다. 관측 차이이며 정비에 의한 개선·악화 판정은 아닙니다.',points)
    note('**작업 경계별 비교:** 전10분 → 직후10분 → 지연20분(후10~30분). 각 창 유효 기록이 80% 미만이면 미산출합니다. 지연 최대 변화는 30초 중앙값 기준이며, 좁은 펄스가 긴 구간 중앙값에 가려지지 않게 별도로 제공합니다.')
    table(boundaries)
    for stage_id,frame in stat.groupby('단계',sort=True):
        label=frame.iloc[0];note(f"**보조 통계 · {label['구간']} · 전체 22개 신호:** {label['시작']:%m/%d %H:%M}–{label['끝']:%H:%M}. 승온·작업 중복을 포함한 전체 관측값입니다.")
        table(frame.drop(columns=['LOT','단계']))
    note('**다음 확인:** 08:54 착수 시점에는 Z3 출력 상승과 온도 하강이 보이고, 12:44 완료 직후에는 CP의 큰 변화가 보이지 않습니다. 이후 Z3·CP 변화는 지연되어 나타납니다. CBM에는 교체 부품이 없으므로 이 그림으로 부품 교체 효과를 평가하지 않습니다.')


In [ ]:
# 과시효·냉각: TC-Z9·냉각은 원래 오르내리는 반복 파형이라, 같은 경과 분 차이와 피크를 맞춘 뒤 차이를 함께 본다 (기존 03과 같은 계산)
cycle_rows,phase,peak_dict,cycle_values,smooth_values=[],{},{},{},{}
CYCLIC=["TC-Z9","TC-OUT1","TC-OUT2"]
for r in lots.itertuples():
    k=int(r.LOT_NO)
    for s in CYCLIC:
        y=profile[k][(s,"median")].where(quality[k][s].ge(.9)&states[k].eq("안정 참고"))
        # 원본을 채우지 않는다. 관측된 이웃 분 3개 이상의 사후 이동 중앙값으로만 피크를 찾는다.
        smooth=y.rolling(5,center=True,min_periods=3).median()
        cycle_values[k,s],smooth_values[k,s]=y,smooth
        peaks,_=find_peaks(smooth.to_numpy(),prominence=5,distance=80 if s=="TC-Z9" else 30)
        peak_dict[k,s]=peaks
        candidate_period=float(np.median(np.diff(peaks))) if len(peaks)>=3 else np.nan
        interval_iqr=float(np.quantile(np.diff(peaks),.75)-np.quantile(np.diff(peaks),.25)) if len(peaks)>=3 else np.nan
        consistent=bool(pd.notna(candidate_period) and interval_iqr/candidate_period<=.2)
        period=candidate_period if consistent else np.nan
        cycle_rows.append({"LOT":k,"GRADE":r.GRADE,"SENSOR":s,"N_PEAKS":len(peaks),"PEAK_INTERVAL_MEDIAN_MIN":candidate_period,"PERIOD_MIN":period,"PHASE_ELIGIBLE":consistent,
                          "PEAK_INTERVAL_IQR_MIN":interval_iqr,
                          "AMPLITUDE_Q95_Q05_C":y.quantile(.95)-y.quantile(.05),"CENTER_C":y.median(),
                          "ASSESSMENT":"피크 간격 일관·주기 비교 가능" if consistent else "피크 간격 불일치·보류" if len(peaks)>=3 else "검출 피크 3개 미만·주기 비교 보류"})
        if pd.notna(period):
            bins=np.floor((y.index.to_numpy()-peaks[0])%period).astype(int)
            phase[k,s]=pd.DataFrame({"PHASE_MIN":bins,"VALUE":y.to_numpy()}).groupby("PHASE_MIN").VALUE.median()
cycle_table=pd.DataFrame(cycle_rows)

phase_rows = []
phase_layout_audit = []

def phase_figure(grade, name):
    ids=lots.loc[lots.GRADE.eq(grade),"LOT_NO"].astype(int).tolist()
    valid_by_sensor={s:[k for k in ids if (k,s) in phase] for s in CYCLIC}
    comparable=[s for s in CYCLIC if len(valid_by_sensor[s]) >= 2]
    note(f"**그림 목적 — {grade} 주기 비교:** 위에서 세 센서의 전체 파형과 검출 피크를 확인한다. 아래는 두 LOT 이상이 피크 수·간격 조건을 충족하는 센서만, 피크 후 경과시간을 맞춰 비교한다.")
    fig=plt.figure(figsize=(24,12 if comparable else 7))
    gs=fig.add_gridspec(2 if comparable else 1,3,left=.09,right=.97,top=.77,
                        bottom=.12,hspace=.62,wspace=.28)
    for j,s in enumerate(CYCLIC):
        ax=fig.add_subplot(gs[0,j])
        for k in ids:
            line(ax,k,s)
            peaks=peak_dict[k,s]
            ax.scatter(peaks,smooth_values[k,s].loc[peaks],color=LOT_COLOR[k],s=22)
        ax.set_xlim(0,float(lots.loc[lots.GRADE.eq(grade),"MINUTES"].max()))
    if comparable:
        lower=gs[1,:].subgridspec(1,len(comparable),wspace=.28)
        for j,s in enumerate(comparable):
            ax=fig.add_subplot(lower[0,j])
            valid=valid_by_sensor[s]
            for k in valid:
                p=phase[k,s]
                ax.plot(p.index,p,color=LOT_COLOR[k],lw=1.8, ls=report_viz.lot_linestyle(k))
            ax.set_title(f"{PROCESS[s]} [{s}] · 피크를 맞춘 {len(valid)} LOT 비교",loc="left",pad=12)
            ax.set_ylabel("온도[℃]",labelpad=10)
            style(ax,"주기 안의 피크 후 경과시간[분]")
            for ix,k in enumerate(valid):
                for other in valid[ix+1:]:
                    pair=pd.concat([phase[k,s].rename("A"),phase[other,s].rename("B")],axis=1).dropna()
                    before_pair=pd.concat([cycle_values[k,s].rename("A"),
                                           cycle_values[other,s].rename("B")],axis=1).dropna()
                    before_mae=(before_pair.A-before_pair.B).abs().mean()
                    phase_rows.append({"ELAPSED_MAE_C":before_mae,"GRADE":grade,"SENSOR":s,"LOT_A":k,"LOT_B":other,"N_PHASE_MIN":len(pair),
                                       "PHASE_MAE_C":(pair.A-pair.B).abs().mean(),"PHASE_MEDIAN_DELTA_C":(pair.A-pair.B).median()})
    names=lambda s:s.replace('TC-','')
    excluded=cycle_table.loc[cycle_table.GRADE.eq(grade)&~cycle_table.PHASE_ELIGIBLE]
    details=[]
    for k,records in excluded.groupby('LOT',sort=False):
        reasons=[]
        for record in records.itertuples():
            reasons.append(f'{names(record.SENSOR)} 피크 {record.N_PEAKS}개' if record.N_PEAKS < 3
                           else f'{names(record.SENSOR)} 피크 간격 불일치')
        details.append(f"{k}({', '.join(reasons)})")
    comparison_text=('주기 비교: '+ '·'.join(names(s) for s in comparable)
                     + ('만 가능' if len(comparable) < len(CYCLIC) else ' 가능')) if comparable else '주기 비교 가능한 센서 없음'
    omitted_text=' / '.join(details)
    caption=(comparison_text + ' · 정렬 제외: ' + omitted_text + ' → 위쪽 전체 파형으로 확인'
             if details else comparison_text)
    fig.text(.09,.055,caption,ha='left',va='bottom',fontsize=12.5,color='#526d80')
    header(fig,f"{grade} 주기·위상 대조",
           f"{grade} Z9·냉각 반복의 주기·중심·진폭 차이 대조",
           lot_handles(ids)+[Line2D([0],[0],color="#34495e",marker="o",ls="None",label="각 LOT 색 점: 평활곡선 검출 피크"),QUALITY_HANDLE],ncol=4)
    axis_tag(fig,'TIME AXIS · 위: 관측 시작 후 / 아래: 주기 안 피크 후 [분]')
    assert len(fig.axes)==len(CYCLIC)+len(comparable)
    assert all(len(valid_by_sensor[s])>=2 for s in comparable)
    phase_layout_audit.append({'GRADE':grade,'FILE':name,'TOP_SENSORS':list(CYCLIC),
                               'LOWER_SENSORS':comparable,'VALID_LOTS':valid_by_sensor,
                               'N_AXES':len(fig.axes),'CAPTION':caption})
    finish(fig,name)
    pp=pd.DataFrame([z for z in phase_rows if z["GRADE"]==grade])
    if len(pp):
        for z in pp.itertuples():
            note(f"**위상 대조 결과:** [{z.SENSOR}] {z.LOT_A}·{z.LOT_B}의 동일 경과 분 평균 절대 차이 {z.ELAPSED_MAE_C:.3f}℃, 피크 정렬 뒤 {z.PHASE_MAE_C:.3f}℃다. 정렬 영향과 실제 중심·진폭 차이를 함께 검토한다.")
    for z in cycle_table.loc[cycle_table.GRADE.eq(grade)].itertuples():
        note(f"**파형 결과:** {z.LOT} [{z.SENSOR}] 중심 {z.CENTER_C:.3f}℃, Q95–Q05 폭 {z.AMPLITUDE_Q95_Q05_C:.3f}℃, 피크 {z.N_PEAKS}개, 주기 {f'{z.PERIOD_MIN:.1f}분' if pd.notna(z.PERIOD_MIN) else '판정 보류'}이다.")
    note(f"**비교 범위:** {comparison_text}. {omitted_text + '은(는) 정렬에서 제외하고 전체 파형으로 확인한다.' if details else ''} "
         "이번 정렬 기준은 검출 피크 3개 이상·피크 간격 IQR/중앙값 20% 이하다. 제외는 정상·이상 판정이 아니다.")
    table(cycle_table.loc[cycle_table.GRADE.eq(grade)])
    if len(pp):table(pp)


<a id="grade-sgcc"></a>

## SGCC — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240001"></a>

### 240001 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240001)
lot_history(240001)

같은 강종의 다른 LOT이 없어 강종 간 선을 대조 LOT처럼 겹치지 않는다. 자체 변화·인접 구역·정비 기록을 읽는다.

<a id="grade-spfh590"></a>

## SPFH590 — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240024"></a>

### 240024 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240024)
lot_history(240024)

<a id="lot-240108"></a>

### 240108 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240108)
lot_history(240108)

### SPFH590 같은 강종 LOT 전체 비교 · 선 하나 = LOT

경과시간으로 비교한다. 다른 LOT은 다른 실제 날짜의 자료이며 같은 소재 위치나 같은 운전 설정을 보장하지 않는다. 빨간 제목 = 다른 LOT이 있는 신호, 굵은 선 = 그 LOT이다. 어느 LOT이 무엇이 다른지는 [03의 6-1](05_all_data_timeseries_jinny_03.ipynb#grade-diff) 표에 있다.

In [ ]:
for page in PAGE_SIGNALS:
 grade_compare("SPFH590",page,f"03_강종/SPFH590/{page}_전체비교.png")
grade_peer_note("SPFH590")
phase_figure("SPFH590","03_강종/SPFH590/Z9_냉각_주기위상.png")

<a id="grade-spce"></a>

## SPCE — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240037"></a>

### 240037 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240037)
lot_history(240037)

<a id="lot-240121"></a>

### 240121 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240121)
lot_history(240121)

<a id="lot-240133"></a>

### 240133 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240133)
lot_history(240133)

### SPCE 같은 강종 LOT 전체 비교 · 선 하나 = LOT

경과시간으로 비교한다. 다른 LOT은 다른 실제 날짜의 자료이며 같은 소재 위치나 같은 운전 설정을 보장하지 않는다. 빨간 제목 = 다른 LOT이 있는 신호, 굵은 선 = 그 LOT이다. 어느 LOT이 무엇이 다른지는 [03의 6-1](05_all_data_timeseries_jinny_03.ipynb#grade-diff) 표에 있다.

In [ ]:
for page in PAGE_SIGNALS:
 grade_compare("SPCE",page,f"03_강종/SPCE/{page}_전체비교.png")
grade_peer_note("SPCE")
phase_figure("SPCE","03_강종/SPCE/Z9_냉각_주기위상.png")

<a id="grade-spcen"></a>

## SPCEN — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240038"></a>

### 240038 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240038)
lot_history(240038)

<a id="lot-240114"></a>

### 240114 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240114)
lot_history(240114)

### SPCEN 같은 강종 LOT 전체 비교 · 선 하나 = LOT

경과시간으로 비교한다. 다른 LOT은 다른 실제 날짜의 자료이며 같은 소재 위치나 같은 운전 설정을 보장하지 않는다. 빨간 제목 = 다른 LOT이 있는 신호, 굵은 선 = 그 LOT이다. 어느 LOT이 무엇이 다른지는 [03의 6-1](05_all_data_timeseries_jinny_03.ipynb#grade-diff) 표에 있다.

In [ ]:
for page in PAGE_SIGNALS:
 grade_compare("SPCEN",page,f"03_강종/SPCEN/{page}_전체비교.png")
grade_peer_note("SPCEN")
phase_figure("SPCEN","03_강종/SPCEN/Z9_냉각_주기위상.png")

<a id="grade-sphc"></a>

## SPHC — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240044"></a>

### 240044 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240044)
lot_history(240044)

<a id="lot-240061"></a>

### 240061 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240061)
lot_history(240061)

<a id="lot-240069"></a>

### 240069 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240069)
lot_history(240069)

<a id="mnt-0708"></a>

#### 07/08 BM 이력과 직전 제공 관측 — CP 펄스와는 시간만 가까움

감지 시각에 온도 자료가 없으면 직전 관측을 날짜를 보존해 따로 본다. 절연 저하는 정비 이력의 사유이며 온도 CSV의 직접 측정 항목이 아니다.

이 BM은 240069의 CP 펄스(+0.459, 02:31) 6시간 뒤 감지됐지만, 정비 경계까지 이 정도 거리는 무작위 관측 시각의 18%에서도 나오는 수준이다. 교체 부품도 O링(예방)이라 CP 계측과 바로 이어지지 않는다. CP와 이어질 수 있는 정비는 17일 뒤 **07/25 가스 분석계 셀 손상 교체**다([03의 1절](05_all_data_timeseries_jinny_03.ipynb#cp-maintenance)).

In [ ]:
maintenance_zoom(["WO-2024-01094"],"04_정비전후/02_0708_BM/01_240069_BM_직전관측.png","7월 BM 이력과 직전 제공 관측")
cp_event_flow(240069,"04_정비전후/02_0708_BM/02_240069_CP_작업선후.png")

<a id="lot-240091"></a>

### 240091 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240091)
lot_history(240091)

### SPHC 같은 강종 LOT 전체 비교 · 선 하나 = LOT

경과시간으로 비교한다. 다른 LOT은 다른 실제 날짜의 자료이며 같은 소재 위치나 같은 운전 설정을 보장하지 않는다. 빨간 제목 = 다른 LOT이 있는 신호, 굵은 선 = 그 LOT이다. 어느 LOT이 무엇이 다른지는 [03의 6-1](05_all_data_timeseries_jinny_03.ipynb#grade-diff) 표에 있다.

In [ ]:
for page in PAGE_SIGNALS:
 grade_compare("SPHC",page,f"03_강종/SPHC/{page}_전체비교.png")
grade_peer_note("SPHC")
phase_figure("SPHC","03_강종/SPHC/Z9_냉각_주기위상.png")

<a id="grade-dp590"></a>

## DP590 — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240058"></a>

### 240058 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240058)
lot_history(240058)

같은 강종의 다른 LOT이 없어 강종 간 선을 대조 LOT처럼 겹치지 않는다. 자체 변화·인접 구역·정비 기록을 읽는다.

<a id="grade-spcd"></a>

## SPCD — LOT 전체에서 특이 구간과 이력까지

<a id="lot-240117"></a>

### 240117 · A 같은 LOT 공정 반응 → B 같은 강종 LOT 비교

A에서는 이 LOT의 전체 신호 → 공정 흐름 → 원본 확대를 보고, B에서는 다른 날짜의 같은 강종 LOT과 같은 신호를 별도로 비교합니다.

In [ ]:
render_lot_review(240117)
lot_history(240117)

같은 강종의 다른 LOT이 없어 강종 간 선을 대조 LOT처럼 겹치지 않는다. 자체 변화·인접 구역·정비 기록을 읽는다.

## 전수 확인과 본문 연결

14 LOT의 전체 22개 신호와 모든 관찰–묶음 구간–비교 그림 연결을 아래에서 확인합니다. 본문은 [01 분석 개요 및 판정 기준](05_all_data_timeseries_jinny_01.ipynb) → [02 탐지 구간 분류 및 분석 대상 선별](05_all_data_timeseries_jinny_02.ipynb) → [03 주요 이상 의심 구간 상세 분석 및 비교](05_all_data_timeseries_jinny_03.ipynb) → [04 종합 판정 및 후속 검증 계획](05_all_data_timeseries_jinny_04.ipynb) 순서로 읽습니다.

In [ ]:
assert set(rv_coverage)==set(lots.LOT_NO.astype(int))
assert all(sensors==set(SIGNALS) for sensors in rv_coverage.values())
review_observations=pd.DataFrame(rv_observation_audit)
review_windows=pd.DataFrame(rv_window_audit)
review_images=pd.DataFrame(rv_image_manifest)
assert review_images['파일'].is_unique
assert all((OUT_DIR/name).is_file() for name in review_images['파일'])
assert all(len(str(OUT_DIR/name))<240 for name in review_images['파일'])
note(f"**확인 결과:** {len(rv_coverage)} LOT × 22개 신호, 관찰 {len(review_observations)}건 → 뷰별 구간 연결 {len(review_windows)}건 · 비교 그림 {review_windows["파일"].nunique()}장. 모든 그림은 figures에 저장했습니다.")
assert {row['파일'] for row in figure_caption_audit} == {row['FILE'] for row in figure_checks}
from IPython.display import HTML
for title, frame in [("구간·그림 연결", review_windows), ("전체 관찰", review_observations),
                     ("그림 목록", review_images), ("그림별 관찰 포인트", pd.DataFrame(figure_caption_audit)), ("정비 확대 검증", pd.DataFrame(maintenance_zoom_audit)),
                     ("글자·범례 배치 검사", pd.DataFrame(figure_checks))]:
    display(HTML("<details><summary>" + title + "</summary>" + frame.to_html(index=False, escape=True) + "</details>"))


table(pd.DataFrame(rv_peer_comparison_audit))
